# Final 379-region TVB semantic-versus-episodic musical-memory proxy experiment

This is the single canonical notebook for the final experiment. It runs the
same code path with one or many worker processes; set `RISE_N_WORKERS=1` for
sequential execution.

The confirmatory comparison is now intentionally narrow:

> How does increasing AD-like amyloid-linked inhibitory perturbation
> differentially affect stimulus-evoked transmission into an expanded
> musical-semantic-associated proxy and an expanded
> musical-episodic-associated proxy?

The expanded definitions combine two complementary evidence sources rather
than replacing either:

1. task-associated HCP-MMP parcels mapped from the direct musical
   semantic-versus-episodic contrasts in Platel et al. (2003); and
2. bilateral medial-frontal versus posterior-medial anatomical cores
   motivated by the revised pathway diagram, posterior-cingulate/precuneus
   anatomy, and Alzheimer musical-memory findings.

The notebook also reports anatomical-core-only, original-Platel-only, and
left/right sensitivity analyses. These are proxy parcel sets, not literal
isolated pathways and not measurements of memory performance.

### Final locked workload

- 20 paired numerical initializations in the main experiment
- the same 20 initializations at 0.25 ms for the required integration-step interaction gate
- 762 planned TVB calls in the unchanged primary experiment in final mode
- a secondary prolonged-stimulation follow-up with 480 additional TVB calls in final mode:
  240 at 0.5 ms and the same 240 at 0.25 ms
- the same 20 initializations in the local-dynamics-held-baseline
  counterfactual
- 50 descriptive spatial shuffles
- 500 topology/pathology/hemisphere-matched parcel-set controls
- pulse, 2 Hz, and 5 Hz probes delivered to bilateral A1
- A1-normalized transfer gain, stimulus-induced functional connectivity,
  and relative pulse-response latency
- a paired condition-by-network interaction, paired Hedges' \(g_z\), and a
  95% interval across numerical initializations
- lossless parcel-level stimulated, matched-control, and evoked PSP
  traces for both A1 parcels and every parcel in the two expanded
  proxies for every main severity, seed, and probe

The previous music-versus-speech run remains exploratory and is not
overwritten by this notebook's output directory.

The prolonged-stimulation analysis is a secondary stability and entrainment follow-up. It does not replace, redefine, or rescue the original 4.5-14.5 second primary endpoint.


## 1. Scientific scope and interpretation boundary

This notebook is a mechanistic neural-mass simulation. It tests whether two
predeclared parcel-set proxies show different model-internal response
trajectories as one amyloid-linked inhibitory parameter field is varied.

It does **not** simulate familiar melodies, learning, recognition,
recollection, semantic knowledge, patients, or preserved clinical
function. The 2 Hz and 5 Hz inputs are temporal probes, not literal speech
or music. Numerical initializations quantify numerical sensitivity, not
biological or patient variability.

The semantic and episodic expanded sets are the primary comparison.
Component-only and laterality analyses are prespecified sensitivity checks,
not alternative definitions selected after seeing the new results.


## 2. Environment setup

Run this cell once in a fresh Colab or local Python environment. Package
installation is skipped when the required imports are already available.


In [ ]:
import importlib.metadata
import importlib.util
import subprocess
import sys

pinned = {
    "tvb-library": "2.10.0",
    "tvb-data": "3.0.0",
}
required_if_missing = {
    "pandas": "pandas>=2.0",
    "matplotlib": "matplotlib>=3.7",
    "scipy": "scipy>=1.10",
    "joblib": "joblib>=1.4,<2.0",
}

install_specs = []
for distribution_name, expected_version in pinned.items():
    try:
        installed_version = importlib.metadata.version(
            distribution_name
        )
    except importlib.metadata.PackageNotFoundError:
        installed_version = None
    if installed_version != expected_version:
        install_specs.append(
            f"{distribution_name}=={expected_version}"
        )

for import_name, requirement in required_if_missing.items():
    if importlib.util.find_spec(import_name) is None:
        install_specs.append(requirement)

if install_specs:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", *install_specs]
    )

version_failures = {}
for distribution_name, expected_version in pinned.items():
    actual_version = importlib.metadata.version(distribution_name)
    if actual_version != expected_version:
        version_failures[distribution_name] = {
            "expected": expected_version,
            "actual": actual_version,
        }
if version_failures:
    raise RuntimeError(
        f"Pinned dependency verification failed: {version_failures}"
    )
print("Pinned TVB dependencies verified.")


In [ ]:
import gc
import hashlib
import io
import json
import logging
import math
import os
import platform
import shutil
import time
import urllib.request
import warnings
from datetime import datetime, timezone
from pathlib import Path

# Process-level parallelism is the outer layer. Force every process to use
# one native numerical thread so n workers do not each create n BLAS threads.
NATIVE_THREAD_ENVIRONMENT_VARIABLES = (
    "OMP_NUM_THREADS",
    "OPENBLAS_NUM_THREADS",
    "MKL_NUM_THREADS",
    "VECLIB_MAXIMUM_THREADS",
    "BLIS_NUM_THREADS",
    "NUMEXPR_NUM_THREADS",
)
for variable_name in NATIVE_THREAD_ENVIRONMENT_VARIABLES:
    os.environ[variable_name] = "1"

os.environ.setdefault("TVB_USER_HOME", "/tmp/rise_tvb_user")
os.environ.setdefault("MPLCONFIGDIR", "/tmp/rise_matplotlib")

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy
from scipy import signal, stats
from IPython.display import display
from joblib import Parallel, delayed, parallel_config
import tvb
from tvb.simulator.lab import (
    connectivity,
    coupling,
    equations,
    integrators,
    models,
    monitors,
    patterns,
    simulator,
)

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 140)
plt.style.use("seaborn-v0_8-whitegrid")

IS_COLAB = "google.colab" in sys.modules
WORK_DIR = (
    Path("/content/rise_tvb379")
    if IS_COLAB
    else Path.cwd() / "rise_tvb379_work"
)
DATA_DIR = WORK_DIR / "source_data"
WORK_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)

print(
    {
        "python": platform.python_version(),
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "scipy": scipy.__version__,
        "joblib": joblib.__version__,
        "tvb_library": importlib.metadata.version("tvb-library"),
        "colab": IS_COLAB,
        "work_dir": str(WORK_DIR),
    }
)


def _positive_environment_integer(variable_name):
    raw_value = os.environ.get(variable_name)
    if raw_value is None or not raw_value.strip():
        return None
    try:
        parsed = int(raw_value)
    except ValueError as error:
        raise ValueError(
            f"{variable_name} must be a positive integer, got {raw_value!r}."
        ) from error
    if parsed < 1:
        raise ValueError(
            f"{variable_name} must be a positive integer, got {parsed}."
        )
    return parsed


def detect_cpu_allocation():
    '''Return conservative CPU limits visible to this process.'''
    sources = {"joblib_cpu_count": max(1, int(joblib.cpu_count()))}

    affinity_function = getattr(os, "sched_getaffinity", None)
    if affinity_function is not None:
        try:
            affinity_count = len(affinity_function(0))
        except (OSError, TypeError, ValueError):
            affinity_count = 0
        if affinity_count > 0:
            sources["process_affinity"] = affinity_count

    # Respect common batch-scheduler allocations instead of using the
    # complete physical node when only part of it was requested.
    for variable_name in (
        "SLURM_CPUS_PER_TASK",
        "NSLOTS",
        "PBS_NP",
        "LSB_DJOB_NUMPROC",
    ):
        value = _positive_environment_integer(variable_name)
        if value is not None:
            sources[variable_name] = value

    return sources


class SimpleProgress:
    def __init__(self, total, description):
        self.total = int(total)
        self.description = str(description)
        self.count = 0
        self.report_every = max(1, self.total // 10)
        self.next_report = self.report_every
        print(f"{self.description}: 0/{self.total}")

    def update(self, amount=1):
        self.count += int(amount)
        if self.count >= self.next_report or self.count >= self.total:
            print(
                f"{self.description}: "
                f"{min(self.count, self.total)}/{self.total}"
            )
            while self.next_report <= self.count:
                self.next_report += self.report_every

    def close(self):
        if self.count != self.total:
            print(f"{self.description}: stopped at {self.count}/{self.total}")


def progress_iter(values, description):
    values = list(values)
    progress = SimpleProgress(len(values), description)
    for value in values:
        yield value
        progress.update()
    progress.close()


def initialize_parallel_worker(work_dir):
    '''Give each spawned worker private TVB and Matplotlib runtime paths.'''
    worker_root = (
        Path(work_dir)
        / ".parallel_runtime"
        / f"worker_{os.getpid()}"
    )
    tvb_home = worker_root / "tvb"
    matplotlib_home = worker_root / "matplotlib"
    tvb_home.mkdir(parents=True, exist_ok=True)
    matplotlib_home.mkdir(parents=True, exist_ok=True)
    os.environ["TVB_USER_HOME"] = str(tvb_home)
    os.environ["MPLCONFIGDIR"] = str(matplotlib_home)
    for variable_name in NATIVE_THREAD_ENVIRONMENT_VARIABLES:
        os.environ[variable_name] = "1"
    if os.environ.get("RISE_DISABLE_LOKY_PSUTIL", "").strip() == "1":
        # Workaround for restricted PID namespaces where psutil cannot see
        # the worker's own /proc entry. Normal Colab/VM runs leave this off.
        from joblib.externals.loky import process_executor
        process_executor._USE_PSUTIL = False


def run_parallel_jobs(
    worker_function,
    job_payloads,
    shared_args,
    description,
):
    '''Run independent jobs in loky worker processes and stream completions.'''
    jobs = list(job_payloads)
    if not jobs:
        return []

    progress = SimpleProgress(len(jobs), description)
    outcomes = []
    try:
        if PARALLEL_WORKERS == 1:
            for job in jobs:
                outcomes.append(worker_function(job, *shared_args))
                progress.update()
            return outcomes

        memmap_dir = WORK_DIR / ".joblib_memmap"
        memmap_dir.mkdir(parents=True, exist_ok=True)
        with parallel_config(
            backend="loky",
            n_jobs=PARALLEL_WORKERS,
            inner_max_num_threads=1,
            initializer=initialize_parallel_worker,
            initargs=(str(WORK_DIR),),
            idle_worker_timeout=900,
            temp_folder=str(memmap_dir),
            max_nbytes="512K",
            mmap_mode="r",
        ):
            outcome_stream = Parallel(
                return_as="generator_unordered",
                batch_size=1,
                pre_dispatch="2*n_jobs",
            )(
                delayed(worker_function)(job, *shared_args)
                for job in jobs
            )
            for outcome in outcome_stream:
                outcomes.append(outcome)
                progress.update()
        return outcomes
    finally:
        progress.close()


## 3. Locked run configuration

`smoke` and `pilot` modes exercise the complete scientific code path with
reduced workloads. `final` is the preregistered workload. The output path is
made unique when a prior result directory already exists, so exploratory or
completed runs are never silently mixed.


In [ ]:
RUN_MODE = os.environ.get("RISE_RUN_MODE", "final").strip().lower()
if RUN_MODE not in {"smoke", "pilot", "final"}:
    raise ValueError("RISE_RUN_MODE must be 'smoke', 'pilot', or 'final'.")

FINAL_NUMERICAL_SEEDS = [
    11, 23, 37, 53, 71, 89, 107, 127, 149, 173,
    197, 223, 251, 281, 313, 347, 383, 421, 461, 503,
]

MODE_CONFIG = {
    "smoke": {
        "seeds": [11],
        "severities": [0.0, 1.0],
        "calibration_couplings": [60.0],
        "matched_null_sets": 20,
        "spatial_shuffles": 2,
        "sensitivity_seeds": [11],
        "dt_check_seeds": [11],
        "sensitivity_scenarios": [
            {"scenario": "G30", "global_coupling": 30.0,
             "input_peak": 0.02},
        ],
    },
    "pilot": {
        "seeds": [11, 23],
        "severities": [0.0, 0.5, 1.0],
        "calibration_couplings": [30.0, 60.0, 100.0],
        "matched_null_sets": 100,
        "spatial_shuffles": 5,
        "sensitivity_seeds": [11, 23],
        "dt_check_seeds": [11],
        "sensitivity_scenarios": [
            {"scenario": "G30", "global_coupling": 30.0,
             "input_peak": 0.02},
            {"scenario": "G100", "global_coupling": 100.0,
             "input_peak": 0.02},
        ],
    },
    "final": {
        "seeds": FINAL_NUMERICAL_SEEDS,
        "severities": [0.0, 0.5, 1.0],
        "calibration_couplings": [
            10.0, 30.0, 60.0, 100.0, 200.0, 300.0
        ],
        "matched_null_sets": 500,
        "spatial_shuffles": 50,
        "sensitivity_seeds": [11, 23, 37, 53, 71],
        "dt_check_seeds": FINAL_NUMERICAL_SEEDS,
        "sensitivity_scenarios": [
            {"scenario": "G30", "global_coupling": 30.0,
             "input_peak": 0.02},
            {"scenario": "G100", "global_coupling": 100.0,
             "input_peak": 0.02},
            {"scenario": "input_0.01", "global_coupling": 60.0,
             "input_peak": 0.01},
            {"scenario": "input_0.04", "global_coupling": 60.0,
             "input_peak": 0.04},
        ],
    },
}
CFG = MODE_CONFIG[RUN_MODE]

N_REGIONS = 379
MAIN_GLOBAL_COUPLING = 60.0
MAIN_INPUT_PEAK_PER_MS = 0.02
MAIN_DT_MS = 0.5
REFERENCE_DT_MS = 0.25
MONITOR_PERIOD_MS = 2.0

STIMULUS_ONSET_MS = 2500.0
PULSE_WIDTH_MS = 100.0
PULSE_ANALYSIS_END_MS = 6000.0
PERIODIC_SETTLING_END_MS = 4500.0
PERIODIC_ANALYSIS_START_MS = 4500.0
PERIODIC_ANALYSIS_END_MS = 14500.0

ORIGINAL_WINDOW_MS = (4500.0, 14500.0)
LATE_WINDOW_MS = (14500.0, 24500.0)
FOLLOWUP_WINDOWS_MS = {
    "original": ORIGINAL_WINDOW_MS,
    "late": LATE_WINDOW_MS,
}
FOLLOWUP_SIMULATION_END_MS = LATE_WINDOW_MS[1]
FOLLOWUP_PERIODIC_PROBES = ("2Hz", "5Hz")
FOLLOWUP_REFERENCE_TYPES = ("zero_input", "dc_matched")
FOLLOWUP_DT_VALUES_MS = (MAIN_DT_MS, REFERENCE_DT_MS)

PROBE_SIMULATION_MS = {
    "pulse": PULSE_ANALYSIS_END_MS,
    "2Hz": PERIODIC_ANALYSIS_END_MS,
    "5Hz": PERIODIC_ANALYSIS_END_MS,
}
CONTROL_SIMULATION_MS = max(PROBE_SIMULATION_MS.values())

PROBES = ("pulse", "2Hz", "5Hz")
PERIODIC_PROBES = ("2Hz", "5Hz")
SEVERITY_LABELS = {
    0.0: "Baseline",
    0.5: "Intermediate AD-like perturbation",
    1.0: "High AD-like perturbation",
}

PERIODIC_SEGMENT_MS = 2000.0
PERIODIC_SEGMENT_COUNT = 5
MULTITAPER_TIME_BANDWIDTH = 3.0
MULTITAPER_TAPERS = 5
SNR_SIGNAL_HALF_WIDTH_HZ = 0.3
SNR_FLANK_INNER_HZ = 0.5
SNR_FLANK_OUTER_HZ = 1.0
A1_SNR_GATE_DB = 6.0
A1_PHASE_CONSISTENCY_GATE = 0.80
TARGET_RESPONSE_RATIO_FLOOR = 1e-5
MIN_TARGET_RESPONSE_COVERAGE = 0.80
MAX_FC_SPLIT_ABS_Z = 0.10
MAX_PULSE_TAIL_FRACTION = 0.10
MAX_NODE_PULSE_TAIL_FRACTION = 0.15
PULSE_TAIL_WINDOW_MS = 200.0
MIN_TARGET_FREQUENCY_SNR_DB = 0.0
MIN_TARGET_PHASE_CONSISTENCY = 0.20
NONSTATIONARY_HALF_LOG2_FLAG = 1.0
FOLLOWUP_TRANSFER_EQUIVALENCE_MARGIN_LOG2 = math.log2(1.20)
FOLLOWUP_FC_EQUIVALENCE_MARGIN_Z = MAX_FC_SPLIT_ABS_Z
FOLLOWUP_DOMINANT_RANGE_HZ = (0.5, 40.0)
FOLLOWUP_EQUIVALENCE_ALPHA = 0.05
PREFLIGHT_SEED_COUNT = 2

DT_TRANSFER_RELATIVE_TOLERANCE = 0.05
DT_TRANSFER_INTERACTION_LOG2_TOLERANCE = math.log2(
    1.0 + DT_TRANSFER_RELATIVE_TOLERANCE
)
DT_FC_ABSOLUTE_TOLERANCE = 0.05
DT_LATENCY_TOLERANCE_MS = 5.0
DT_A1_SNR_TOLERANCE_DB = 1.0

TRACE_EXPORT_SCOPES = ("main_full_field",)
TRACE_FORMAT_VERSION = "parcel_psp_v1"
TRACE_ARCHIVE_SUBDIRECTORY = "main_parcel_traces"
TRACE_CHECKPOINT_TAG = "raw-trace-v1"

FOLLOWUP_SCOPE = "late_window_followup"
FOLLOWUP_REFERENCE_SCOPE = "late_window_followup_dt_reference_0.25ms"
FOLLOWUP_TRACE_FORMAT_VERSION = "prolonged_periodic_psp_v1"
FOLLOWUP_TRACE_ARCHIVE_SUBDIRECTORY = "late_followup_parcel_traces"
FOLLOWUP_ANALYSIS_VERSION = "late-window-dc-matched-v1"

DOWNLOAD_RESULTS_AT_END = False

CPU_ALLOCATION_SOURCES = detect_cpu_allocation()
AVAILABLE_CPU_COUNT = min(CPU_ALLOCATION_SOURCES.values())
REQUESTED_PARALLEL_WORKERS = _positive_environment_integer(
    "RISE_N_WORKERS"
)
PARALLEL_WORKERS = (
    AVAILABLE_CPU_COUNT
    if REQUESTED_PARALLEL_WORKERS is None
    else min(REQUESTED_PARALLEL_WORKERS, AVAILABLE_CPU_COUNT)
)
if (
    REQUESTED_PARALLEL_WORKERS is not None
    and REQUESTED_PARALLEL_WORKERS > AVAILABLE_CPU_COUNT
):
    print(
        "Requested RISE_N_WORKERS exceeds the detected allocation; "
        f"using {AVAILABLE_CPU_COUNT}."
    )
PARALLEL_BACKEND = "joblib-loky-processes"
NATIVE_THREADS_PER_WORKER = 1

requested_run_id = os.environ.get("RISE_RUN_ID", "").strip()
base_run_id = requested_run_id or f"semantic_episodic_v3_{RUN_MODE}"
candidate_results_dir = WORK_DIR / f"results_{base_run_id}"
if candidate_results_dir.exists() and any(candidate_results_dir.iterdir()):
    timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    candidate_results_dir = WORK_DIR / (
        f"results_{base_run_id}_{timestamp}"
    )
RESULTS_DIR = candidate_results_dir
FIGURE_DIR = RESULTS_DIR / "figures"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
RAW_TRACE_DIR = RESULTS_DIR / TRACE_ARCHIVE_SUBDIRECTORY
RAW_TRACE_DIR.mkdir(parents=True, exist_ok=True)
FOLLOWUP_TRACE_DIR = RESULTS_DIR / FOLLOWUP_TRACE_ARCHIVE_SUBDIRECTORY
FOLLOWUP_TRACE_DIR.mkdir(parents=True, exist_ok=True)

print("RUN_MODE:", RUN_MODE)
print("Numerical seeds:", CFG["seeds"])
print("Spatial shuffles:", CFG["spatial_shuffles"])
print("CPU allocation sources:", CPU_ALLOCATION_SOURCES)
print("Parallel worker processes:", PARALLEL_WORKERS)
print("Results directory:", RESULTS_DIR)


## 4. Download pinned source files and verify hashes

The source commits and SHA-256 hashes are fixed. Set `RISE_DATA_CACHE` to a
directory containing these exact files to run without downloading.


In [ ]:
EDUCASE_COMMIT = "659d4fcbf58d74867fa9d10a874deac854532ee1"
PIPELINE_COMMIT = "8be09e33e1131ed2f0764506940e6172de275285"

SOURCE_SPECS = {
    "structural_connectivity": {
        "filename": "avg_healthy_normSC_mod.txt",
        "url": (
            "https://raw.githubusercontent.com/BrainModes/"
            "TVB_EducaseAD_molecular_pathways_TVB/"
            f"{EDUCASE_COMMIT}/avg_healthy_normSC_mod.txt"
        ),
        "sha256": "141fc993c84bde0b2f0ee0280ce1ccc47e1731ddcbf37845a4eef38dad9fa562",
    },
    "ad_left_cortex": {
        "filename": "AD_LH.txt",
        "url": (
            "https://raw.githubusercontent.com/BrainModes/"
            "TVB_EducaseAD_molecular_pathways_TVB/"
            f"{EDUCASE_COMMIT}/_AD/AD_LH.txt"
        ),
        "sha256": "566e770e93f50d3378a0cf7d2dc8b1fa5af9475ca432463acf7bc2b5507907af",
    },
    "ad_right_cortex": {
        "filename": "AD_RH.txt",
        "url": (
            "https://raw.githubusercontent.com/BrainModes/"
            "TVB_EducaseAD_molecular_pathways_TVB/"
            f"{EDUCASE_COMMIT}/_AD/AD_RH.txt"
        ),
        "sha256": "4f42c31c08e6d191d415953f763889f816d9c2443d115612f0c076cfd5b2f129",
    },
    "ad_subcortical": {
        "filename": "AD_subcortical.txt",
        "url": (
            "https://raw.githubusercontent.com/BrainModes/"
            "TVB_EducaseAD_molecular_pathways_TVB/"
            f"{EDUCASE_COMMIT}/_AD/AD_subcortical.txt"
        ),
        "sha256": "f28926c7955db2c2762b5ba032f0bbde770a0da3d3705dd037a746382506d824",
    },
    "region_labels": {
        "filename": "region_labels.txt",
        "url": (
            "https://raw.githubusercontent.com/BrainModes/"
            f"ADNI-TVB-pipeline/{PIPELINE_COMMIT}/misc_files/region_labels.txt"
        ),
        "sha256": "f9688592130a034210b482a0556fdc14383eaaf578bef39d2ddba072537e3484",
    },
}

def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def fetch_verified(name, spec):
    destination = DATA_DIR / spec["filename"]
    local_cache = os.environ.get("RISE_DATA_CACHE")
    cache_candidate = (
        Path(local_cache) / spec["filename"] if local_cache else None
    )

    if destination.exists() and sha256_file(destination) == spec["sha256"]:
        source_used = "existing verified file"
    elif (
        cache_candidate is not None
        and cache_candidate.exists()
        and sha256_file(cache_candidate) == spec["sha256"]
    ):
        shutil.copyfile(cache_candidate, destination)
        source_used = "verified local validation cache"
    else:
        print(f"Downloading {name}...")
        with urllib.request.urlopen(spec["url"], timeout=120) as response:
            destination.write_bytes(response.read())
        source_used = spec["url"]

    actual = sha256_file(destination)
    if actual != spec["sha256"]:
        raise RuntimeError(
            f"SHA-256 mismatch for {name}. Expected {spec['sha256']}, got {actual}."
        )
    return {
        "source": name,
        "path": str(destination),
        "sha256": actual,
        "retrieved_from": source_used,
    }

source_manifest_df = pd.DataFrame(
    [fetch_verified(name, spec) for name, spec in SOURCE_SPECS.items()]
)
display(source_manifest_df)

## 5. Load and validate the 379-region alignment

Hard checks lock the structural matrix, amyloid surrogate, region order,
hemisphere blocks, and key atlas anchors before any ROI is defined.


In [ ]:
WEIGHTS = np.loadtxt(DATA_DIR / SOURCE_SPECS["structural_connectivity"]["filename"])
LABELS = np.array(
    [
        line.strip()
        for line in (
            DATA_DIR / SOURCE_SPECS["region_labels"]["filename"]
        ).read_text().splitlines()
        if line.strip()
    ]
)
AD_AMYLOID = np.concatenate(
    [
        np.loadtxt(DATA_DIR / SOURCE_SPECS["ad_left_cortex"]["filename"]),
        np.loadtxt(DATA_DIR / SOURCE_SPECS["ad_right_cortex"]["filename"]),
        np.loadtxt(DATA_DIR / SOURCE_SPECS["ad_subcortical"]["filename"]),
    ]
)

hard_checks = {
    "SC is 379 x 379": WEIGHTS.shape == (N_REGIONS, N_REGIONS),
    "379 unique labels": len(LABELS) == N_REGIONS
    and len(set(LABELS.tolist())) == N_REGIONS,
    "379 amyloid values": AD_AMYLOID.shape == (N_REGIONS,),
    "SC values finite": np.isfinite(WEIGHTS).all(),
    "SC values nonnegative": (WEIGHTS >= 0).all(),
    "amyloid values finite": np.isfinite(AD_AMYLOID).all(),
    "left cortex occupies 0:180": all(
        label.startswith("L_") for label in LABELS[:180]
    ),
    "right cortex occupies 180:360": all(
        label.startswith("R_") for label in LABELS[180:360]
    ),
    "brainstem is final parcel": LABELS[-1] == "Brainstem",
}
failed_checks = [name for name, passed in hard_checks.items() if not passed]
if failed_checks:
    raise RuntimeError("Input validation failed: " + "; ".join(failed_checks))

LABEL_TO_INDEX = {label: index for index, label in enumerate(LABELS)}
expected_anchors = {
    "L_A1": 23,
    "R_A1": 203,
    "L_6ma": 43,
    "R_6ma": 223,
    "L_24dd": 39,
    "R_24dd": 219,
    "L_STSdp": 128,
    "R_STSdp": 308,
    "L_44": 73,
    "R_44": 253,
}
anchor_failures = {
    label: (LABEL_TO_INDEX.get(label), expected_index)
    for label, expected_index in expected_anchors.items()
    if LABEL_TO_INDEX.get(label) != expected_index
}
if anchor_failures:
    raise RuntimeError(f"Parcel-order anchor check failed: {anchor_failures}")

relative_asymmetry = float(
    np.linalg.norm(WEIGHTS - WEIGHTS.T) / np.linalg.norm(WEIGHTS)
)
data_quality_df = pd.DataFrame(
    {
        "check": list(hard_checks.keys())
        + [
            "SC minimum",
            "SC maximum",
            "SC relative asymmetry",
            "amyloid minimum",
            "amyloid maximum",
            "amyloid mean",
        ],
        "result": list(hard_checks.values())
        + [
            float(WEIGHTS.min()),
            float(WEIGHTS.max()),
            relative_asymmetry,
            float(AD_AMYLOID.min()),
            float(AD_AMYLOID.max()),
            float(AD_AMYLOID.mean()),
        ],
    }
)
display(data_quality_df)
print(
    "The nonzero asymmetry is a property of the published matrix. "
    "The simulation preserves it."
)

## 6. Predeclared semantic- and episodic-associated proxy definitions

### Primary expanded proxies

| Proxy | Anatomical core | Original musical-task peaks | Unique nodes |
|---|---|---|---:|
| Semantic-associated | bilateral `9m`, `10r`, `10v`, `8BM`, `25` | `R_9m`, `L_25`, `R_TE1a`, `L_TF`, `L_STSda` | 13 |
| Episodic-associated | bilateral `31pd`, `31pv`, `d23ab`, `v23ab`, `23d`, `31a`, `PCV`, `7m` | `R_IP1`, `R_PCV`, `R_11l`, `R_8Av` | 19 |

The union is deliberate. The Platel parcels preserve direct
musical-task-contrast specificity. The revised medial-frontal and
posterior-medial cores provide broader anatomical coverage supported by
HCP-MMP anatomy and posterior-cingulate/precuneus memory literature. The
component sets remain separately analyzed.

The primary aggregation gives left and right hemispheres equal weight.
This prevents the unequal count and laterality of the published task peaks
from silently weighting one hemisphere more heavily. The Platel-only
sensitivity preserves its published asymmetry and is interpreted
accordingly.

`25` is subgenual anterior cingulate, while `8BM`, `9m`, `10r`, and `10v`
span medial frontal territories. Therefore the semantic core is named
medial-prefrontal/anterior-cingulate, not uniformly "superomedial PFC."
The episodic core spans posterior cingulate and medial parietal/precuneus
territories.


In [ ]:
def bilateral(*parcel_names):
    return tuple(
        label
        for parcel in parcel_names
        for label in (f"L_{parcel}", f"R_{parcel}")
    )


def ordered_union(*groups):
    return tuple(
        dict.fromkeys(
            label
            for group in groups
            for label in group
        )
    )


def label_hemisphere(label):
    if label.startswith("L_") or label.startswith("Left-"):
        return "L"
    if label.startswith("R_") or label.startswith("Right-"):
        return "R"
    return "M"


A1_LABELS = bilateral("A1")

SEMANTIC_ANATOMICAL_CORE_LABELS = bilateral(
    "9m", "10r", "10v", "8BM", "25"
)
EPISODIC_ANATOMICAL_CORE_LABELS = bilateral(
    "31pd", "31pv", "d23ab", "v23ab",
    "23d", "31a", "PCV", "7m",
)

SEMANTIC_PLATEL_LABELS = (
    "R_9m",
    "L_25",
    "R_TE1a",
    "L_TF",
    "L_STSda",
)
EPISODIC_PLATEL_LABELS = (
    "R_IP1",
    "R_PCV",
    "R_11l",
    "R_8Av",
)

SEMANTIC_EXPANDED_LABELS = ordered_union(
    SEMANTIC_ANATOMICAL_CORE_LABELS,
    SEMANTIC_PLATEL_LABELS,
)
EPISODIC_EXPANDED_LABELS = ordered_union(
    EPISODIC_ANATOMICAL_CORE_LABELS,
    EPISODIC_PLATEL_LABELS,
)

SEMANTIC_EXPANDED_LEFT_LABELS = tuple(
    label for label in SEMANTIC_EXPANDED_LABELS
    if label_hemisphere(label) == "L"
)
SEMANTIC_EXPANDED_RIGHT_LABELS = tuple(
    label for label in SEMANTIC_EXPANDED_LABELS
    if label_hemisphere(label) == "R"
)
EPISODIC_EXPANDED_LEFT_LABELS = tuple(
    label for label in EPISODIC_EXPANDED_LABELS
    if label_hemisphere(label) == "L"
)
EPISODIC_EXPANDED_RIGHT_LABELS = tuple(
    label for label in EPISODIC_EXPANDED_LABELS
    if label_hemisphere(label) == "R"
)

ROI_GROUPS = {
    "a1_input": {
        "labels": A1_LABELS,
        "analysis_role": "stimulus",
        "evidence_component": "auditory_input",
        "interpretation": "Bilateral primary auditory cortex.",
    },
    "semantic_expanded": {
        "labels": SEMANTIC_EXPANDED_LABELS,
        "analysis_role": "primary",
        "evidence_component": "union",
        "interpretation": (
            "Primary expanded musical-semantic-associated proxy: "
            "bilateral medial-prefrontal/anterior-cingulate core plus "
            "Platel musical-task peaks."
        ),
    },
    "episodic_expanded": {
        "labels": EPISODIC_EXPANDED_LABELS,
        "analysis_role": "primary",
        "evidence_component": "union",
        "interpretation": (
            "Primary expanded musical-episodic-associated proxy: "
            "bilateral posterior-cingulate/precuneus core plus Platel "
            "musical-task peaks."
        ),
    },
    "semantic_anatomical_core": {
        "labels": SEMANTIC_ANATOMICAL_CORE_LABELS,
        "analysis_role": "definition_sensitivity",
        "evidence_component": "revised_anatomical_core",
        "interpretation": (
            "Bilateral medial-prefrontal/anterior-cingulate core."
        ),
    },
    "episodic_anatomical_core": {
        "labels": EPISODIC_ANATOMICAL_CORE_LABELS,
        "analysis_role": "definition_sensitivity",
        "evidence_component": "revised_anatomical_core",
        "interpretation": (
            "Bilateral posterior-cingulate/medial-parietal core."
        ),
    },
    "semantic_platel_peaks": {
        "labels": SEMANTIC_PLATEL_LABELS,
        "analysis_role": "definition_sensitivity",
        "evidence_component": "platel_2003_direct_contrast",
        "interpretation": (
            "HCP-MMP parcels mapped from Platel semantic-greater-than-"
            "episodic musical-memory cortical peaks."
        ),
    },
    "episodic_platel_peaks": {
        "labels": EPISODIC_PLATEL_LABELS,
        "analysis_role": "definition_sensitivity",
        "evidence_component": "platel_2003_direct_contrast",
        "interpretation": (
            "HCP-MMP parcels mapped from Platel episodic-greater-than-"
            "semantic musical-memory cortical peaks."
        ),
    },
    "semantic_expanded_left": {
        "labels": SEMANTIC_EXPANDED_LEFT_LABELS,
        "analysis_role": "laterality_sensitivity",
        "evidence_component": "union_left",
        "interpretation": "Left-hemisphere portion of semantic_expanded.",
    },
    "episodic_expanded_left": {
        "labels": EPISODIC_EXPANDED_LEFT_LABELS,
        "analysis_role": "laterality_sensitivity",
        "evidence_component": "union_left",
        "interpretation": "Left-hemisphere portion of episodic_expanded.",
    },
    "semantic_expanded_right": {
        "labels": SEMANTIC_EXPANDED_RIGHT_LABELS,
        "analysis_role": "laterality_sensitivity",
        "evidence_component": "union_right",
        "interpretation": "Right-hemisphere portion of semantic_expanded.",
    },
    "episodic_expanded_right": {
        "labels": EPISODIC_EXPANDED_RIGHT_LABELS,
        "analysis_role": "laterality_sensitivity",
        "evidence_component": "union_right",
        "interpretation": "Right-hemisphere portion of episodic_expanded.",
    },
    "shared_auditory_relay": {
        "labels": bilateral(
            "52", "MBelt", "LBelt", "RI", "PBelt", "A4", "A5"
        ),
        "analysis_role": "diagnostic_only",
        "evidence_component": "auditory_relay",
        "interpretation": (
            "Shared belt, parabelt, retroinsular, and auditory-association "
            "context; not part of either primary target average."
        ),
    },
    "anterior_temporal_context": {
        "labels": bilateral(
            "TGd", "TGv", "TE1a", "TE1m", "TE1p",
            "TE2a", "TE2p", "TF", "STSda",
        ),
        "analysis_role": "diagnostic_only",
        "evidence_component": "semantic_context",
        "interpretation": (
            "Anterior/lateral temporal conceptual context, saved "
            "separately to avoid arbitrarily broadening the primary proxy."
        ),
    },
    "hippocampal_context": {
        "labels": ("Left-Hippocampus", "Right-Hippocampus"),
        "analysis_role": "diagnostic_only",
        "evidence_component": "memory_context",
        "interpretation": (
            "Bilateral hippocampal context. The Jansen-Rit simulation "
            "does not implement encoding or recollection."
        ),
    },
    "angular_parietal_context": {
        "labels": bilateral("PGi", "PGs", "PGp"),
        "analysis_role": "diagnostic_only",
        "evidence_component": "memory_context",
        "interpretation": (
            "Angular/inferior-parietal context, reported separately."
        ),
    },
}

ALL_ROI_LABELS = ordered_union(
    *(specification["labels"] for specification in ROI_GROUPS.values())
)
missing_roi_labels = sorted(set(ALL_ROI_LABELS) - set(LABELS))
if missing_roi_labels:
    raise RuntimeError(
        "ROI labels absent from the loaded 379-region order: "
        f"{missing_roi_labels}"
    )

for group_name, specification in ROI_GROUPS.items():
    labels_in_group = specification["labels"]
    if not labels_in_group:
        raise RuntimeError(f"ROI group {group_name!r} is empty.")
    if len(labels_in_group) != len(set(labels_in_group)):
        raise RuntimeError(
            f"Duplicate label inside ROI group {group_name!r}."
        )

if len(SEMANTIC_EXPANDED_LABELS) != 13:
    raise RuntimeError("Semantic expanded proxy must contain 13 nodes.")
if len(EPISODIC_EXPANDED_LABELS) != 19:
    raise RuntimeError("Episodic expanded proxy must contain 19 nodes.")
expanded_overlap = sorted(
    set(SEMANTIC_EXPANDED_LABELS)
    & set(EPISODIC_EXPANDED_LABELS)
)
if expanded_overlap:
    raise RuntimeError(
        f"Primary expanded proxies overlap: {expanded_overlap}"
    )

A1_INDICES = np.array(
    [LABEL_TO_INDEX[label] for label in A1_LABELS],
    dtype=int,
)
A1_INDEX_BY_HEMISPHERE = {
    label_hemisphere(label): LABEL_TO_INDEX[label]
    for label in A1_LABELS
}

NETWORK_LABELS = {
    group_name: specification["labels"]
    for group_name, specification in ROI_GROUPS.items()
    if group_name != "a1_input"
}
NETWORK_INDICES = {
    network_name: np.array(
        [LABEL_TO_INDEX[label] for label in network_labels],
        dtype=int,
    )
    for network_name, network_labels in NETWORK_LABELS.items()
}

PRIMARY_SEMANTIC_NETWORK = "semantic_expanded"
PRIMARY_EPISODIC_NETWORK = "episodic_expanded"
PRIMARY_INFERENTIAL_NETWORKS = (
    PRIMARY_SEMANTIC_NETWORK,
    PRIMARY_EPISODIC_NETWORK,
)
NETWORK_PAIRS = {
    "expanded_bilateral": (
        "semantic_expanded",
        "episodic_expanded",
    ),
    "anatomical_core_only": (
        "semantic_anatomical_core",
        "episodic_anatomical_core",
    ),
    "platel_peak_only": (
        "semantic_platel_peaks",
        "episodic_platel_peaks",
    ),
    "expanded_left_only": (
        "semantic_expanded_left",
        "episodic_expanded_left",
    ),
    "expanded_right_only": (
        "semantic_expanded_right",
        "episodic_expanded_right",
    ),
}

ALL_DECLARED_INDICES = np.array(
    sorted(LABEL_TO_INDEX[label] for label in ALL_ROI_LABELS),
    dtype=int,
)
MEMORY_COUNTERFACTUAL_FIXED_INDICES = np.array(
    sorted(
        set(A1_INDICES.tolist())
        | set(NETWORK_INDICES["semantic_expanded"].tolist())
        | set(NETWORK_INDICES["episodic_expanded"].tolist())
    ),
    dtype=int,
)

TRACE_REGION_LABELS = ordered_union(
    A1_LABELS,
    SEMANTIC_EXPANDED_LABELS,
    EPISODIC_EXPANDED_LABELS,
)
TRACE_REGION_INDICES = np.array(
    [LABEL_TO_INDEX[label] for label in TRACE_REGION_LABELS],
    dtype=int,
)
if len(TRACE_REGION_LABELS) != 34:
    raise RuntimeError(
        "Raw trace export must contain 2 A1, 13 semantic, and "
        "19 episodic parcels."
    )
if len(set(TRACE_REGION_LABELS)) != len(TRACE_REGION_LABELS):
    raise RuntimeError("Raw trace export labels must be unique.")

MUSIC_MEMORY_PEAK_MAPPINGS = [
    {
        "source_contrast": "semantic > episodic",
        "reported_region": "bilateral medial frontal cortex (BA 11/10)",
        "spm99_x": 0, "spm99_y": 60, "spm99_z": 10,
        "hcp_label": "R_9m", "mapping_distance_mm": 1.0,
        "mapping_note": "nearest volumetric HCP-MMP parcel",
    },
    {
        "source_contrast": "semantic > episodic",
        "reported_region": "bilateral medial frontal cortex (BA 11/10)",
        "spm99_x": -4, "spm99_y": 18, "spm99_z": -18,
        "hcp_label": "L_25", "mapping_distance_mm": 0.0,
        "mapping_note": "coordinate falls inside parcel",
    },
    {
        "source_contrast": "semantic > episodic",
        "reported_region": "right middle temporal gyrus (BA 21)",
        "spm99_x": 56, "spm99_y": 4, "spm99_z": -24,
        "hcp_label": "R_TE1a", "mapping_distance_mm": 0.0,
        "mapping_note": "coordinate falls inside parcel",
    },
    {
        "source_contrast": "semantic > episodic",
        "reported_region": "left inferior/middle temporal gyri (BA 20/21)",
        "spm99_x": -48, "spm99_y": -26, "spm99_z": -22,
        "hcp_label": "L_TF", "mapping_distance_mm": 0.0,
        "mapping_note": "coordinate falls inside parcel",
    },
    {
        "source_contrast": "semantic > episodic",
        "reported_region": "left inferior/middle temporal gyri (BA 20/21)",
        "spm99_x": -54, "spm99_y": -2, "spm99_z": -18,
        "hcp_label": "L_STSda", "mapping_distance_mm": 0.0,
        "mapping_note": "coordinate falls inside parcel",
    },
    {
        "source_contrast": "episodic > semantic",
        "reported_region": "right precuneus/parietal cortex (BA 7/19)",
        "spm99_x": 36, "spm99_y": -66, "spm99_z": 38,
        "hcp_label": "R_IP1", "mapping_distance_mm": 2.0,
        "mapping_note": "nearest volumetric HCP-MMP parcel",
    },
    {
        "source_contrast": "episodic > semantic",
        "reported_region": "precuneus (BA 7)",
        "spm99_x": 4, "spm99_y": -56, "spm99_z": 42,
        "hcp_label": "R_PCV", "mapping_distance_mm": 0.0,
        "mapping_note": "coordinate falls inside parcel",
    },
    {
        "source_contrast": "episodic > semantic",
        "reported_region": "right superior frontal gyrus (BA 11)",
        "spm99_x": 34, "spm99_y": 52, "spm99_z": -14,
        "hcp_label": "R_11l", "mapping_distance_mm": 1.0,
        "mapping_note": (
            "nearest parcel consistent with reported BA11 anatomy; "
            "volumetric boundary"
        ),
    },
    {
        "source_contrast": "episodic > semantic",
        "reported_region": "right middle frontal gyrus (BA 8/9)",
        "spm99_x": 38, "spm99_y": 12, "spm99_z": 44,
        "hcp_label": "R_8Av", "mapping_distance_mm": 1.0,
        "mapping_note": "nearest volumetric HCP-MMP parcel",
    },
]
music_memory_peak_mapping_df = pd.DataFrame(
    MUSIC_MEMORY_PEAK_MAPPINGS
)

mapped_semantic = set(
    music_memory_peak_mapping_df.loc[
        music_memory_peak_mapping_df["source_contrast"]
        == "semantic > episodic",
        "hcp_label",
    ]
)
mapped_episodic = set(
    music_memory_peak_mapping_df.loc[
        music_memory_peak_mapping_df["source_contrast"]
        == "episodic > semantic",
        "hcp_label",
    ]
)
if mapped_semantic != set(SEMANTIC_PLATEL_LABELS):
    raise RuntimeError("Semantic Platel mapping and ROI list disagree.")
if mapped_episodic != set(EPISODIC_PLATEL_LABELS):
    raise RuntimeError("Episodic Platel mapping and ROI list disagree.")

roi_rows = []
for group_name, specification in ROI_GROUPS.items():
    for label in specification["labels"]:
        roi_rows.append(
            {
                "network": group_name,
                "analysis_role": specification["analysis_role"],
                "evidence_component": (
                    specification["evidence_component"]
                ),
                "label": label,
                "hemisphere": label_hemisphere(label),
                "zero_based_index": LABEL_TO_INDEX[label],
                "interpretation": specification["interpretation"],
            }
        )
roi_definition_df = pd.DataFrame(roi_rows)

network_evidence_df = pd.DataFrame(
    [
        {
            "component": "Platel task peaks",
            "supports": "musical semantic/episodic task specificity",
            "source": (
                "Platel et al. 2003, NeuroImage 20:244-256, "
                "doi:10.1016/S1053-8119(03)00287-8"
            ),
            "limitation": (
                "Small PET study; coordinate-to-HCP parcel mapping is "
                "approximate and represents peaks, not full clusters."
            ),
        },
        {
            "component": "Medial-frontal semantic core",
            "supports": (
                "Broader medial-prefrontal/anterior-cingulate anatomical "
                "coverage in the revised operational diagram."
            ),
            "source": (
                "Glasser et al. 2016 HCP-MMP atlas; interpreted together "
                "with Platel 2003 rather than as a standalone music network."
            ),
            "limitation": (
                "An anatomical proxy, not a complete or selectively "
                "localized musical-semantic network."
            ),
        },
        {
            "component": "Posterior-medial episodic core",
            "supports": (
                "Posterior cingulate/precuneus anatomy and its relationship "
                "to memory; AD musical-memory abnormalities in posterior "
                "cingulate/precuneus."
            ),
            "source": (
                "Rolls et al. 2023, Human Brain Mapping 44:629-655, "
                "doi:10.1002/hbm.26089; "
                "Slattery et al. 2019, Cortex 115:357-370."
            ),
            "limitation": (
                "A posterior-medial proxy, not the complete episodic-memory "
                "system and not a directed pathway."
            ),
        },
    ]
)

print("Primary semantic expanded nodes:", len(SEMANTIC_EXPANDED_LABELS))
print("Primary episodic expanded nodes:", len(EPISODIC_EXPANDED_LABELS))
print(
    "Semantic left/right:",
    len(SEMANTIC_EXPANDED_LEFT_LABELS),
    len(SEMANTIC_EXPANDED_RIGHT_LABELS),
)
print(
    "Episodic left/right:",
    len(EPISODIC_EXPANDED_LEFT_LABELS),
    len(EPISODIC_EXPANDED_RIGHT_LABELS),
)
display(roi_definition_df)
display(music_memory_peak_mapping_df)
display(network_evidence_df)


## 7. Construct the AD-like inhibitory perturbation

This reproduces the public Stefanovski educational model's transformation
of the surrogate amyloid field into the Jansen-Rit inhibitory rate
parameter \(b\). Severity 0, 0.5, and 1 are deterministic interpolation
levels. They are named baseline, intermediate, and high **AD-like
perturbation**, not HC, MCI, or AD participants.


In [ ]:
def transform_amyloid_to_b(
    amyloid,
    max_val=0.05,
    min_val=0.02,
    amyloid_max=2.65,
    amyloid_offset=1.4,
):
    amyloid = np.asarray(amyloid, dtype=float)
    x0 = (amyloid_max - amyloid_offset) / 2.0 + amyloid_offset
    k = (
        np.log(max_val / ((min_val + 0.001) - min_val) - 1.0)
        / (amyloid_max - x0)
    )
    return max_val / (1.0 + np.exp(k * (amyloid - x0))) + min_val

BASELINE_B = np.full(N_REGIONS, 0.07, dtype=float)
HIGH_B = transform_amyloid_to_b(AD_AMYLOID)
B_BY_SEVERITY = {
    severity: BASELINE_B + severity * (HIGH_B - BASELINE_B)
    for severity in (0.0, 0.5, 1.0)
}

if not np.isfinite(HIGH_B).all():
    raise RuntimeError("The transformed inhibitory vector contains nonfinite values.")
if HIGH_B.min() < 0.0199 or HIGH_B.max() > 0.0701:
    raise RuntimeError("The transformed inhibitory vector is outside the expected range.")

pathology_summary_df = pd.DataFrame(
    [
        {
            "severity": severity,
            "condition": SEVERITY_LABELS[severity],
            "b_min": float(values.min()),
            "b_mean": float(values.mean()),
            "b_max": float(values.max()),
            "mean_tau_i_ms": float(np.mean(1.0 / values)),
        }
        for severity, values in B_BY_SEVERITY.items()
    ]
)
display(pathology_summary_df)

roi_pathology_df = roi_definition_df.copy()
roi_pathology_df["surrogate_amyloid"] = [
    AD_AMYLOID[LABEL_TO_INDEX[label]]
    for label in roi_pathology_df["label"]
]
roi_pathology_df["baseline_b"] = 0.07
roi_pathology_df["high_b"] = [
    HIGH_B[LABEL_TO_INDEX[label]] for label in roi_pathology_df["label"]
]
roi_pathology_df["b_reduction"] = (
    roi_pathology_df["baseline_b"] - roi_pathology_df["high_b"]
)
display(roi_pathology_df)
print(
    "Important: the target parcels have different local surrogate amyloid values. "
    "The local-dynamics-held-baseline counterfactual later in the notebook is therefore required."
)

## 8. TVB model and stimulus functions

The public 379-region package contains the averaged structural weight
matrix but no compatible averaged tract-length matrix. Following the
educational model, interregional delays remain zero. Consequently, the
pulse metric below is **relative model-response timing**, not anatomical
conduction latency.

Periodic simulations run through 14.5 seconds. The first 2.0 seconds after
stimulus onset are settling time, followed by a 10.0-second analysis
window containing 20 cycles at 2 Hz or 50 cycles at 5 Hz. Pulse simulations
stop at 6.0 seconds. A single long matched control is safely truncated for
the pulse because every run uses identical deterministic initial
conditions for a given seed and condition. The fixed periodic window is an
early evoked-response window, not an assumed steady state; five 2-second
bins preserve its temporal trajectory.


In [ ]:
def fresh_connectivity(weights, labels):
    weights = np.asarray(weights, dtype=float)
    labels = np.asarray(labels)
    if weights.shape != (N_REGIONS, N_REGIONS):
        raise ValueError("weights must be a 379-by-379 matrix.")
    if labels.shape != (N_REGIONS,):
        raise ValueError("labels must contain exactly 379 entries.")
    return connectivity.Connectivity(
        weights=weights.copy(),
        tract_lengths=np.zeros_like(weights),
        centres=np.zeros((N_REGIONS, 3), dtype=float),
        region_labels=labels.copy(),
        speed=np.array([100.0]),
    )


def build_model(b_values):
    background = np.array([0.1085])
    model = models.JansenRit(
        v0=np.array([6.0]),
        mu=background,
        p_min=background,
        p_max=background,
        b=np.asarray(b_values, dtype=float),
        variables_of_interest=("y1", "y2"),
    )
    # TVB's external p(t) input enters the y4 derivative.
    model.stvar = np.array([4], dtype=np.int32)
    return model


def make_temporal_equation(
    probe,
    model,
    input_peak_per_ms,
    onset_ms=STIMULUS_ONSET_MS,
    offset_ms=None,
):
    if offset_ms is None:
        offset_ms = PROBE_SIMULATION_MS[probe]
    derivative_peak = float(
        model.A[0] * model.a[0] * float(input_peak_per_ms)
    )
    if probe == "pulse":
        return equations.TemporalApplicableEquation(
            equation=(
                "where((var >= onset) & (var < onset + width), "
                "amp, 0.0)"
            ),
            parameters={
                "onset": float(onset_ms),
                "width": float(PULSE_WIDTH_MS),
                "amp": derivative_peak,
            },
        )

    frequency_per_ms = {"2Hz": 0.002, "5Hz": 0.005}[probe]
    return equations.TemporalApplicableEquation(
        equation=(
            "where((var >= onset) & (var < offset), "
            "0.5 * amp * (1.0 + sin(6.283185307179586 * "
            "frequency * (var - onset))), 0.0)"
        ),
        parameters={
            "onset": float(onset_ms),
            "offset": float(offset_ms),
            "amp": derivative_peak,
            "frequency": frequency_per_ms,
        },
    )


def sample_stimulus_temporal_waveform(
    time_ms,
    probe,
    model,
    input_peak_per_ms,
    onset_ms=STIMULUS_ONSET_MS,
    offset_ms=None,
):
    """Sample the same temporal equation supplied to TVB."""
    time_ms = np.asarray(time_ms, dtype=float)
    if offset_ms is None:
        offset_ms = PROBE_SIMULATION_MS[probe]
    derivative_peak = float(
        model.A[0] * model.a[0] * float(input_peak_per_ms)
    )
    if probe == "pulse":
        active = (time_ms >= float(onset_ms)) & (
            time_ms < float(onset_ms) + PULSE_WIDTH_MS
        )
        return np.where(active, derivative_peak, 0.0)
    frequency_per_ms = {"2Hz": 0.002, "5Hz": 0.005}[probe]
    active = (time_ms >= float(onset_ms)) & (
        time_ms < float(offset_ms)
    )
    phase = (
        2.0 * np.pi * frequency_per_ms
        * (time_ms - float(onset_ms))
    )
    return np.where(
        active,
        0.5 * derivative_peak * (1.0 + np.sin(phase)),
        0.0,
    )


def make_initial_conditions(seed):
    rng = np.random.default_rng(int(seed))
    # With zero delays, one state-history sample is sufficient.
    return rng.random((1, 6, N_REGIONS, 1))



class _ExactTVBMessageFilter(logging.Filter):
    def __init__(self, blocked_message):
        super().__init__()
        self.blocked_message = str(blocked_message)

    def filter(self, record):
        return record.getMessage() != self.blocked_message


_TVB_DETERMINISTIC_RANDOM_STATE_MESSAGE = (
    "random_state supplied for non-stochastic integration"
)
_TVB_DETERMINISTIC_RANDOM_STATE_FILTER = _ExactTVBMessageFilter(
    _TVB_DETERMINISTIC_RANDOM_STATE_MESSAGE
)


def suppress_known_tvb_deterministic_random_state_warning():
    integrator_logger = logging.getLogger("tvb.simulator.integrators")
    if _TVB_DETERMINISTIC_RANDOM_STATE_FILTER not in integrator_logger.filters:
        integrator_logger.addFilter(
            _TVB_DETERMINISTIC_RANDOM_STATE_FILTER
        )


def run_tvb(
    b_values,
    probe,
    global_coupling,
    input_peak_per_ms,
    seed,
    weights,
    labels,
    a1_indices,
    dt_ms=MAIN_DT_MS,
    simulation_ms=CONTROL_SIMULATION_MS,
):
    b_values = np.asarray(b_values, dtype=float)
    if b_values.shape != (N_REGIONS,) or not np.isfinite(b_values).all():
        raise ValueError("b_values must be a finite 379-element vector.")
    if probe not in {None, "pulse", "2Hz", "5Hz"}:
        raise ValueError(f"Unknown probe: {probe}")
    if not np.isclose(
        MONITOR_PERIOD_MS / float(dt_ms),
        round(MONITOR_PERIOD_MS / float(dt_ms)),
    ):
        raise ValueError("Monitor period must be an integer multiple of dt.")
    a1_indices = np.asarray(a1_indices, dtype=int)
    if a1_indices.ndim != 1 or len(a1_indices) == 0:
        raise ValueError(
            "a1_indices must be a nonempty one-dimensional array."
        )

    white_matter = fresh_connectivity(weights, labels)
    model = build_model(b_values)
    stimulus = None
    if probe is not None:
        regional_weights = np.zeros(N_REGIONS, dtype=float)
        regional_weights[a1_indices] = 1.0 / np.sqrt(len(a1_indices))
        stimulus = patterns.StimuliRegion(
            temporal=make_temporal_equation(
                probe,
                model,
                input_peak_per_ms,
                offset_ms=simulation_ms,
            ),
            connectivity=white_matter,
            weight=regional_weights,
        )

    suppress_known_tvb_deterministic_random_state_warning()
    experiment = simulator.Simulator(
        connectivity=white_matter,
        model=model,
        coupling=coupling.SigmoidalJansenRit(
            a=np.array([float(global_coupling)])
        ),
        integrator=integrators.HeunDeterministic(dt=float(dt_ms)),
        monitors=(monitors.SubSample(period=MONITOR_PERIOD_MS),),
        stimulus=stimulus,
        initial_conditions=make_initial_conditions(seed),
    )
    experiment.configure()

    started = time.perf_counter()
    (time_ms, raw), = experiment.run(
        simulation_length=float(simulation_ms)
    )
    wall_seconds = time.perf_counter() - started
    psp = raw[:, 0, :, 0] - raw[:, 1, :, 0]
    time_ms = np.asarray(time_ms, dtype=float)
    psp = np.asarray(psp, dtype=float)

    if psp.shape[1] != N_REGIONS:
        raise RuntimeError(f"Unexpected TVB output shape: {psp.shape}")
    if not np.isfinite(psp).all():
        raise RuntimeError("TVB produced NaN or infinite values.")
    if float(np.max(np.abs(psp))) > 100.0:
        raise RuntimeError(
            "TVB activity exceeded the prespecified safety bound of 100."
        )
    return time_ms, psp, wall_seconds


## 9. Locked response metrics and experiment runner

For every parcel, the evoked trace is

\[
e_j(t)=PSP_{\mathrm{stimulated},j}(t)
       -PSP_{\mathrm{matched\ control},j}(t).
\]

### Evoked transfer gain

The primary periodic response is the root-mean-square of the linearly
detrended evoked PSP in the fixed 10-second analysis window. This measures
the complete control-subtracted response, including broadband or
nonstationary activity, without pretending that every target remains
phase-locked to the drive. Each parcel is normalized by ipsilateral A1.
Parcel values are averaged within each hemisphere, then left and right
hemisphere summaries receive equal weight. Positive transfer values are
baseline-referenced with a log2 ratio.

The same response is also calculated in five predeclared 2-second bins.
Their log2 slope and first-half versus second-half difference quantify
temporal escalation. They are diagnostics of the modeled response
trajectory, not criteria for deleting a nonstationary condition.

### Evoked functional connectivity

Pearson correlation is calculated between each linearly detrended evoked
target trace \(e_j(t)\) and its ipsilateral evoked A1 trace. Correlations
are Fisher-z transformed before network averaging:

\[
FC_j=\operatorname{atanh}\left(
    r(e_{\mathrm{A1},h}(t),e_j(t))
\right).
\]

Severity effects subtract the same network's baseline FC. Because the
matched control is already subtracted in \(e_j(t)\), this avoids
correlating near-zero residuals after arbitrary harmonic regression.
It remains common-input-sensitive PSP-level functional connectivity, not
BOLD FC and not directed effective connectivity.

### Pulse-response latency

For the 3.5-second post-onset pulse window, \(t_{50}\) is the first time at
which 50% of the evoked squared-response energy has accumulated. Node
latency is \(t_{50,j}-t_{50,\mathrm{ipsilateral\ A1}}\); the network value
is the equal-hemisphere mean of hemisphere medians. Severity effects are
baseline-referenced differences in milliseconds. Negative values are
retained.

### Frequency-response QA

### Frequency-specific sensitivity and technical gates

A separate frequency-locked transfer sensitivity uses the median of five
2-second exact-frequency sine/cosine amplitudes. Drive-frequency SNR uses
a five-taper DPSS spectrum with \(NW=3\); cycle phase consistency is also
retained. Harmonic-fit \(R^2\) remains a legacy diagnostic.

Both A1 hemispheres must meet SNR \(\ge 6\) dB and phase consistency
\(\ge 0.80\). Primary target networks must have at least 80% of parcels
above a response/A1 floor of \(10^{-5}\). Evoked FC must agree between the
two five-second halves within 0.10 Fisher-z, and pulse latency requires at
least 80% valid parcels with median final-200-ms energy fraction no larger
than 0.10. Target frequency locking is reported separately and does not
invalidate the primary broadband evoked-transfer outcome.


In [ ]:
def b_signature(values):
    return hashlib.sha256(
        np.asarray(values, dtype=np.float64).tobytes()
    ).hexdigest()[:16]


def _periodic_frequency_hz(probe):
    return {"2Hz": 2.0, "5Hz": 5.0}[probe]


def periodic_window(time_ms):
    return (
        (time_ms >= PERIODIC_ANALYSIS_START_MS)
        & (time_ms < PERIODIC_ANALYSIS_END_MS)
    )


def periodic_half_windows(time_ms):
    midpoint_ms = 0.5 * (
        PERIODIC_ANALYSIS_START_MS + PERIODIC_ANALYSIS_END_MS
    )
    return (
        (
            (time_ms >= PERIODIC_ANALYSIS_START_MS)
            & (time_ms < midpoint_ms)
        ),
        (
            (time_ms >= midpoint_ms)
            & (time_ms < PERIODIC_ANALYSIS_END_MS)
        ),
    )


def periodic_segment_windows(time_ms):
    windows = []
    for segment_index in range(PERIODIC_SEGMENT_COUNT):
        segment_start = (
            PERIODIC_ANALYSIS_START_MS
            + segment_index * PERIODIC_SEGMENT_MS
        )
        segment_end = segment_start + PERIODIC_SEGMENT_MS
        windows.append(
            (time_ms >= segment_start) & (time_ms < segment_end)
        )
    if not all(window.any() for window in windows):
        raise RuntimeError("A periodic segment window is empty.")
    return tuple(windows)


def pulse_window(time_ms):
    return (
        (time_ms >= STIMULUS_ONSET_MS)
        & (time_ms < PULSE_ANALYSIS_END_MS)
    )


def detrended_ac_rms(values, window):
    y = np.asarray(values[window], dtype=float)
    if y.shape[0] < 100:
        raise RuntimeError("An evoked-response window is too short.")
    y = signal.detrend(y, axis=0, type="linear")
    return np.sqrt(np.mean(y ** 2, axis=0))


def exact_frequency_fit(
    time_ms,
    evoked,
    probe,
    analysis_window=None,
):
    frequency_hz = _periodic_frequency_hz(probe)
    window = (
        periodic_window(time_ms)
        if analysis_window is None
        else np.asarray(analysis_window, dtype=bool)
    )
    t_seconds = (
        time_ms[window] - float(time_ms[window][0])
    ) / 1000.0
    y = np.asarray(evoked[window], dtype=float)
    if y.shape[0] < 100:
        raise RuntimeError(
            "Periodic analysis window is unexpectedly short."
        )

    omega_t = 2.0 * np.pi * frequency_hz * t_seconds
    design = np.column_stack(
        [
            np.sin(omega_t),
            np.cos(omega_t),
            np.ones_like(t_seconds),
            t_seconds - np.mean(t_seconds),
        ]
    )
    coefficients, _, _, _ = np.linalg.lstsq(design, y, rcond=None)
    fitted = design @ coefficients
    sin_coefficient = coefficients[0]
    cos_coefficient = coefficients[1]
    amplitude = np.sqrt(sin_coefficient ** 2 + cos_coefficient ** 2)

    # Convention: y_f(t) = A sin(2*pi*f*t + phase), so
    # beta_s = A cos(phase), beta_c = A sin(phase), and
    # phase = atan2(beta_c, beta_s). The time origin is the first sample
    # in the fitted window. Relative A1-target phase is invariant to it.
    phase_rad = np.arctan2(cos_coefficient, sin_coefficient)
    residual_ss = np.sum((y - fitted) ** 2, axis=0)
    total_ss = np.sum(
        (y - np.mean(y, axis=0)) ** 2,
        axis=0,
    )
    r_squared = 1.0 - residual_ss / np.maximum(total_ss, 1e-15)
    return {
        "amplitude": amplitude,
        "r_squared": r_squared,
        "sin_coefficient": sin_coefficient,
        "cos_coefficient": cos_coefficient,
        "phase_rad": phase_rad,
    }


def exact_frequency_amplitude_and_r_squared(
    time_ms,
    evoked,
    probe,
    analysis_window=None,
):
    fit = exact_frequency_fit(
        time_ms,
        evoked,
        probe,
        analysis_window=analysis_window,
    )
    return fit["amplitude"], fit["r_squared"]


def drive_frequency_snr_db(time_ms, evoked, probe):
    frequency_hz = _periodic_frequency_hz(probe)
    window = periodic_window(time_ms)
    y = signal.detrend(
        np.asarray(evoked[window], dtype=float),
        axis=0,
        type="linear",
    )
    sampling_hz = 1000.0 / float(np.median(np.diff(time_ms[window])))
    tapers = signal.windows.dpss(
        y.shape[0],
        NW=MULTITAPER_TIME_BANDWIDTH,
        Kmax=MULTITAPER_TAPERS,
        sym=False,
        norm=2,
    )
    frequencies = np.fft.rfftfreq(y.shape[0], d=1.0 / sampling_hz)
    power = np.zeros(
        (len(frequencies), y.shape[1]),
        dtype=float,
    )
    for taper in tapers:
        spectrum = np.fft.rfft(
            y * np.asarray(taper)[:, None],
            axis=0,
        )
        power += np.abs(spectrum) ** 2
    power /= float(len(tapers))
    signal_mask = (
        np.abs(frequencies - frequency_hz)
        <= SNR_SIGNAL_HALF_WIDTH_HZ
    )
    flank_mask = (
        (
            frequencies
            >= frequency_hz - SNR_FLANK_OUTER_HZ
        )
        & (
            frequencies
            <= frequency_hz - SNR_FLANK_INNER_HZ
        )
    ) | (
        (
            frequencies
            >= frequency_hz + SNR_FLANK_INNER_HZ
        )
        & (
            frequencies
            <= frequency_hz + SNR_FLANK_OUTER_HZ
        )
    )
    if not signal_mask.any() or not flank_mask.any():
        raise RuntimeError("SNR frequency bands are empty.")
    signal_power = np.mean(power[signal_mask], axis=0)
    noise_power = np.mean(power[flank_mask], axis=0)
    return 10.0 * np.log10(
        np.maximum(signal_power, 1e-30)
        / np.maximum(noise_power, 1e-30)
    )


def cycle_phase_consistency(time_ms, evoked, probe):
    frequency_hz = _periodic_frequency_hz(probe)
    window = periodic_window(time_ms)
    y = np.asarray(evoked[window], dtype=float)
    sampling_hz = 1000.0 / float(np.median(np.diff(time_ms[window])))
    samples_per_cycle_float = sampling_hz / frequency_hz
    samples_per_cycle = int(round(samples_per_cycle_float))
    if not np.isclose(
        samples_per_cycle,
        samples_per_cycle_float,
        atol=1e-10,
    ):
        raise RuntimeError(
            "Monitor sampling does not contain an integer number of "
            "samples per drive cycle."
        )
    cycle_count = y.shape[0] // samples_per_cycle
    if cycle_count < 10:
        raise RuntimeError("Too few complete cycles for phase QA.")
    trimmed = y[: cycle_count * samples_per_cycle]
    cycles = trimmed.reshape(
        cycle_count,
        samples_per_cycle,
        trimmed.shape[1],
    )
    cycles = signal.detrend(cycles, axis=1, type="linear")
    cycle_time = np.arange(samples_per_cycle) / sampling_hz
    basis = np.exp(-2j * np.pi * frequency_hz * cycle_time)
    coefficients = np.einsum("csr,s->cr", cycles, basis)
    magnitudes = np.abs(coefficients)
    unit_phase = coefficients / np.maximum(magnitudes, 1e-30)
    phase_consistency = np.abs(np.mean(unit_phase, axis=0))
    phase_consistency[
        np.max(magnitudes, axis=0) <= 1e-20
    ] = np.nan
    return phase_consistency


def columnwise_correlation(reference, values):
    reference = np.asarray(reference, dtype=float)
    values = np.asarray(values, dtype=float)
    reference_centered = reference - np.mean(reference)
    values_centered = values - np.mean(values, axis=0)
    numerator = reference_centered @ values_centered
    denominator = np.sqrt(
        np.sum(reference_centered ** 2)
        * np.sum(values_centered ** 2, axis=0)
    )
    correlations = np.full(values.shape[1], np.nan, dtype=float)
    valid = denominator > 1e-24
    correlations[valid] = numerator[valid] / denominator[valid]
    return np.clip(correlations, -0.999999, 0.999999)


def fisher_z(correlations):
    return np.arctanh(
        np.clip(correlations, -0.999999, 0.999999)
    )


def evoked_a1_target_fc_z(
    time_ms,
    evoked,
    labels,
    analysis_window=None,
):
    window = (
        periodic_window(time_ms)
        if analysis_window is None
        else np.asarray(analysis_window, dtype=bool)
    )
    evoked_detrended = signal.detrend(
        np.asarray(evoked[window], dtype=float),
        axis=0,
        type="linear",
    )
    result = np.full(N_REGIONS, np.nan, dtype=float)

    for hemisphere, a1_index in A1_INDEX_BY_HEMISPHERE.items():
        target_indices = np.array(
            [
                index for index, label in enumerate(labels)
                if label_hemisphere(label) == hemisphere
            ],
            dtype=int,
        )
        result[target_indices] = fisher_z(
            columnwise_correlation(
                evoked_detrended[:, a1_index],
                evoked_detrended[:, target_indices],
            )
        )

    midline_indices = np.array(
        [
            index for index, label in enumerate(labels)
            if label_hemisphere(label) == "M"
        ],
        dtype=int,
    )
    if len(midline_indices):
        bilateral_a1 = np.mean(
            evoked_detrended[:, A1_INDICES],
            axis=1,
        )
        result[midline_indices] = fisher_z(
            columnwise_correlation(
                bilateral_a1,
                evoked_detrended[:, midline_indices],
            )
        )
    return result


def fractional_energy_t50_ms(time_ms, evoked):
    window = pulse_window(time_ms)
    times = np.asarray(time_ms[window], dtype=float)
    y = np.asarray(evoked[window], dtype=float)
    if y.shape[0] < 100:
        raise RuntimeError("Pulse analysis window is unexpectedly short.")
    energy = y ** 2
    cumulative = np.cumsum(energy, axis=0)
    total = cumulative[-1]
    valid = total > 1e-24
    fraction = cumulative / np.maximum(total, 1e-30)
    threshold_crossed = fraction >= 0.5
    first_index = np.argmax(threshold_crossed, axis=0)
    t50 = times[first_index] - STIMULUS_ONSET_MS
    t50[~valid] = np.nan
    return t50


def pulse_tail_energy_fraction(time_ms, evoked):
    window = pulse_window(time_ms)
    times = np.asarray(time_ms[window], dtype=float)
    y = np.asarray(evoked[window], dtype=float)
    energy = y ** 2
    total_energy = np.sum(energy, axis=0)
    tail_start_ms = (
        PULSE_ANALYSIS_END_MS - PULSE_TAIL_WINDOW_MS
    )
    tail_mask = times >= tail_start_ms
    if not tail_mask.any():
        raise RuntimeError("Pulse tail-energy window is empty.")
    tail_energy = np.sum(energy[tail_mask], axis=0)
    result = tail_energy / np.maximum(total_energy, 1e-30)
    result[total_energy <= 1e-24] = np.nan
    return result


def pulse_peak_characteristics(time_ms, evoked, labels):
    window = pulse_window(time_ms)
    times = np.asarray(time_ms[window], dtype=float)
    y = np.asarray(evoked[window], dtype=float)
    absolute_peak_indices = np.argmax(np.abs(y), axis=0)
    region_indices = np.arange(N_REGIONS)
    signed_peak_response = y[absolute_peak_indices, region_indices]
    absolute_peak_response = np.abs(signed_peak_response)
    absolute_peak_time_ms = times[absolute_peak_indices]
    peak_time_after_onset_ms = (
        absolute_peak_time_ms - STIMULUS_ONSET_MS
    )
    total_evoked_energy_psp2_ms = scipy.integrate.trapezoid(
        y ** 2,
        x=times,
        axis=0,
    )
    return {
        "absolute_peak_response": absolute_peak_response,
        "signed_peak_response": signed_peak_response,
        "absolute_peak_time_ms": absolute_peak_time_ms,
        "peak_time_after_stimulus_onset_ms": peak_time_after_onset_ms,
        "peak_time_relative_to_ipsilateral_a1_ms": (
            absolute_peak_time_ms
            - ipsilateral_reference_values(absolute_peak_time_ms, labels)
        ),
        "total_evoked_energy_psp2_ms": total_evoked_energy_psp2_ms,
    }


def ipsilateral_reference_values(node_values, labels):
    node_values = np.asarray(node_values, dtype=float)
    result = np.empty(N_REGIONS, dtype=float)
    bilateral_reference = float(np.nanmean(node_values[A1_INDICES]))
    for index, label in enumerate(labels):
        hemisphere = label_hemisphere(label)
        if hemisphere in A1_INDEX_BY_HEMISPHERE:
            result[index] = node_values[
                A1_INDEX_BY_HEMISPHERE[hemisphere]
            ]
        else:
            result[index] = bilateral_reference
    return result


def wrap_phase_rad(values):
    values = np.asarray(values, dtype=float)
    return np.angle(np.exp(1j * values))


def phase_lag_to_ipsilateral_a1(phase_rad, labels):
    phase_rad = np.asarray(phase_rad, dtype=float)
    reference = np.empty(N_REGIONS, dtype=float)
    bilateral_reference = float(
        np.angle(np.mean(np.exp(1j * phase_rad[A1_INDICES])))
    )
    for index, label in enumerate(labels):
        hemisphere = label_hemisphere(label)
        if hemisphere in A1_INDEX_BY_HEMISPHERE:
            reference[index] = phase_rad[
                A1_INDEX_BY_HEMISPHERE[hemisphere]
            ]
        else:
            reference[index] = bilateral_reference
    return wrap_phase_rad(phase_rad - reference)


def _trace_archive_filename(severity, seed, probe):
    severity_token = f"{float(severity):.1f}".replace(".", "p")
    return (
        f"severity_{severity_token}_seed_{int(seed):04d}_"
        f"{str(probe)}.npz"
    )


def build_parcel_trace_archive(
    *,
    scope,
    variant,
    condition_name,
    severity,
    seed,
    probe,
    dt_ms,
    global_coupling,
    input_peak_per_ms,
    b_values,
    time_ms,
    stimulated_psp,
    control_psp,
    labels,
    a1_indices,
):
    """Serialize untouched selected PSP samples with lossless compression."""
    time_ms = np.asarray(time_ms, dtype=float)
    stimulated_psp = np.asarray(stimulated_psp)
    control_psp = np.asarray(control_psp)
    if stimulated_psp.shape != control_psp.shape:
        raise RuntimeError("Raw stimulated and control PSP shapes differ.")
    if stimulated_psp.shape != (len(time_ms), N_REGIONS):
        raise RuntimeError("Raw PSP shape does not match time and parcels.")

    selected_stimulated = np.ascontiguousarray(
        stimulated_psp[:, TRACE_REGION_INDICES]
    )
    selected_control = np.ascontiguousarray(
        control_psp[:, TRACE_REGION_INDICES]
    )
    selected_evoked = selected_stimulated - selected_control
    model = build_model(b_values)
    temporal_waveform = sample_stimulus_temporal_waveform(
        time_ms,
        probe,
        model,
        input_peak_per_ms,
        offset_ms=float(time_ms[-1]),
    )
    a1_indices = np.asarray(a1_indices, dtype=int)
    a1_spatial_weights = np.full(
        len(a1_indices),
        1.0 / np.sqrt(len(a1_indices)),
        dtype=float,
    )
    actual_a1_waveform = (
        temporal_waveform[:, None] * a1_spatial_weights[None, :]
    )

    buffer = io.BytesIO()
    np.savez_compressed(
        buffer,
        stimulated_psp=selected_stimulated,
        control_psp=selected_control,
        evoked_psp=selected_evoked,
        time_ms=time_ms,
        region_labels=np.asarray(TRACE_REGION_LABELS),
        region_indices=TRACE_REGION_INDICES,
        semantic_expanded_membership=np.isin(
            TRACE_REGION_INDICES,
            NETWORK_INDICES[PRIMARY_SEMANTIC_NETWORK],
        ),
        episodic_expanded_membership=np.isin(
            TRACE_REGION_INDICES,
            NETWORK_INDICES[PRIMARY_EPISODIC_NETWORK],
        ),
        a1_membership=np.isin(TRACE_REGION_INDICES, A1_INDICES),
        scope=np.asarray(str(scope)),
        variant=np.asarray(str(variant)),
        condition=np.asarray(str(condition_name)),
        severity=np.asarray(float(severity)),
        seed=np.asarray(int(seed)),
        probe=np.asarray(str(probe)),
        integration_step_ms=np.asarray(float(dt_ms)),
        monitor_period_ms=np.asarray(float(MONITOR_PERIOD_MS)),
        global_coupling=np.asarray(float(global_coupling)),
        input_amplitude_per_ms=np.asarray(float(input_peak_per_ms)),
        stimulus_temporal_waveform=temporal_waveform,
        stimulus_waveform=actual_a1_waveform,
        stimulus_region_labels=np.asarray(labels)[a1_indices],
        stimulus_region_indices=a1_indices,
        stimulus_spatial_weights=a1_spatial_weights,
        stimulus_waveform_units=np.asarray("model_derivative_input"),
        stimulus_onset_ms=np.asarray(float(STIMULUS_ONSET_MS)),
        pulse_width_ms=np.asarray(float(PULSE_WIDTH_MS)),
        trace_format_version=np.asarray(TRACE_FORMAT_VERSION),
    )
    payload = buffer.getvalue()
    filename = _trace_archive_filename(severity, seed, probe)
    return {
        "filename": filename,
        "payload": payload,
        "sha256": hashlib.sha256(payload).hexdigest(),
        "byte_size": len(payload),
        "sample_count": len(time_ms),
        "region_count": len(TRACE_REGION_INDICES),
    }


def write_parcel_trace_archive(archive):
    filename = str(archive["filename"])
    if Path(filename).name != filename or not filename.endswith(".npz"):
        raise RuntimeError("Unsafe raw trace archive filename.")
    output_path = RAW_TRACE_DIR / filename
    expected_sha256 = str(archive["sha256"])
    if output_path.exists() and sha256_file(output_path) == expected_sha256:
        return output_path
    temporary_path = output_path.with_name(
        f".{output_path.name}.{os.getpid()}.partial"
    )
    with temporary_path.open("wb") as handle:
        handle.write(archive["payload"])
        handle.flush()
        os.fsync(handle.fileno())
    if sha256_file(temporary_path) != expected_sha256:
        temporary_path.unlink(missing_ok=True)
        raise RuntimeError("Raw trace archive failed its SHA-256 check.")
    os.replace(temporary_path, output_path)
    return output_path


def classify_integration_step_check(
    *,
    finite,
    direction_consistent,
    inference_consistent,
    precision_target_passed,
):
    fatal_validity_passed = bool(
        finite and direction_consistent and inference_consistent
    )
    return {
        "fatal_validity_passed": fatal_validity_passed,
        "precision_target_passed": bool(precision_target_passed),
        "passed": fatal_validity_passed,
        "gate_action": (
            "pass"
            if fatal_validity_passed and precision_target_passed
            else "warn_exact_magnitude_dt_sensitive"
            if fatal_validity_passed
            else "stop_fatal_validity_failure"
        ),
    }


def compute_node_metrics(
    time_ms,
    stimulated_psp,
    control_psp,
    probe,
    labels,
):
    evoked = stimulated_psp - control_psp
    empty = np.full(N_REGIONS, np.nan, dtype=float)
    if probe == "pulse":
        window = pulse_window(time_ms)
        response = np.sqrt(
            np.mean(evoked[window] ** 2, axis=0)
        )
        t50_ms = fractional_energy_t50_ms(time_ms, evoked)
        relative_latency_raw_ms = (
            t50_ms - ipsilateral_reference_values(t50_ms, labels)
        )
        pulse_tail_fraction = pulse_tail_energy_fraction(
            time_ms,
            evoked,
        )
        fit_r_squared = empty.copy()
        snr_db = empty.copy()
        phase_consistency = empty.copy()
        locked_response = empty.copy()
        sin_coefficient = empty.copy()
        cos_coefficient = empty.copy()
        phase_rad = empty.copy()
        phase_lag_rad = empty.copy()
        phase_lag_degrees = empty.copy()
        pulse_peaks = pulse_peak_characteristics(time_ms, evoked, labels)
        evoked_fc_z = empty.copy()
        evoked_fc_first_half_z = empty.copy()
        evoked_fc_second_half_z = empty.copy()
        first_half_response = empty.copy()
        second_half_response = empty.copy()
        segment_responses = None
        segment_locked_responses = None
    else:
        full_window = periodic_window(time_ms)
        first_half_window, second_half_window = (
            periodic_half_windows(time_ms)
        )
        segment_windows = periodic_segment_windows(time_ms)
        response = detrended_ac_rms(evoked, full_window)
        first_half_response = detrended_ac_rms(
            evoked,
            first_half_window,
        )
        second_half_response = detrended_ac_rms(
            evoked,
            second_half_window,
        )
        segment_responses = np.stack(
            [
                detrended_ac_rms(evoked, segment_window)
                for segment_window in segment_windows
            ],
            axis=0,
        )
        frequency_fit = exact_frequency_fit(time_ms, evoked, probe)
        locked_response = frequency_fit["amplitude"]
        fit_r_squared = frequency_fit["r_squared"]
        sin_coefficient = frequency_fit["sin_coefficient"]
        cos_coefficient = frequency_fit["cos_coefficient"]
        phase_rad = frequency_fit["phase_rad"]
        phase_lag_rad = phase_lag_to_ipsilateral_a1(
            phase_rad,
            labels,
        )
        phase_lag_degrees = np.degrees(phase_lag_rad)
        pulse_peaks = {
            "absolute_peak_response": empty.copy(),
            "signed_peak_response": empty.copy(),
            "absolute_peak_time_ms": empty.copy(),
            "peak_time_after_stimulus_onset_ms": empty.copy(),
            "peak_time_relative_to_ipsilateral_a1_ms": empty.copy(),
            "total_evoked_energy_psp2_ms": empty.copy(),
        }
        segment_locked_responses = np.stack(
            [
                exact_frequency_amplitude_and_r_squared(
                    time_ms,
                    evoked,
                    probe,
                    segment_window,
                )[0]
                for segment_window in segment_windows
            ],
            axis=0,
        )
        snr_db = drive_frequency_snr_db(time_ms, evoked, probe)
        phase_consistency = cycle_phase_consistency(
            time_ms, evoked, probe
        )
        evoked_fc_z = evoked_a1_target_fc_z(
            time_ms,
            evoked,
            labels,
        )
        evoked_fc_first_half_z = evoked_a1_target_fc_z(
            time_ms,
            evoked,
            labels,
            first_half_window,
        )
        evoked_fc_second_half_z = evoked_a1_target_fc_z(
            time_ms,
            evoked,
            labels,
            second_half_window,
        )
        relative_latency_raw_ms = empty.copy()
        pulse_tail_fraction = empty.copy()

    a1_reference = ipsilateral_reference_values(response, labels)
    if np.nanmin(a1_reference[A1_INDICES]) <= 1e-10:
        raise RuntimeError(
            "A1 response is too small for stable normalization."
        )
    node_transfer = response / np.maximum(a1_reference, 1e-30)
    response_valid = (
        np.isfinite(node_transfer)
        & (node_transfer >= TARGET_RESPONSE_RATIO_FLOOR)
    )

    if probe == "pulse":
        first_half_node_transfer = empty.copy()
        second_half_node_transfer = empty.copy()
        segment_node_transfer = None
        locked_node_transfer = empty.copy()
        segment_locked_node_transfer = None
        frequency_qa_valid = np.zeros(N_REGIONS, dtype=bool)
        fc_valid = np.zeros(N_REGIONS, dtype=bool)
        latency_valid = (
            response_valid
            & np.isfinite(relative_latency_raw_ms)
            & np.isfinite(pulse_tail_fraction)
            & (
                pulse_tail_fraction
                <= MAX_NODE_PULSE_TAIL_FRACTION
            )
        )
        relative_latency_ms = np.where(
            latency_valid,
            relative_latency_raw_ms,
            np.nan,
        )
    else:
        first_half_reference = ipsilateral_reference_values(
            first_half_response,
            labels,
        )
        second_half_reference = ipsilateral_reference_values(
            second_half_response,
            labels,
        )
        first_half_node_transfer = (
            first_half_response
            / np.maximum(first_half_reference, 1e-30)
        )
        second_half_node_transfer = (
            second_half_response
            / np.maximum(second_half_reference, 1e-30)
        )
        segment_node_transfer = np.stack(
            [
                segment_response
                / np.maximum(
                    ipsilateral_reference_values(
                        segment_response,
                        labels,
                    ),
                    1e-30,
                )
                for segment_response in segment_responses
            ],
            axis=0,
        )
        segment_locked_node_transfer = np.stack(
            [
                segment_locked_response
                / np.maximum(
                    ipsilateral_reference_values(
                        segment_locked_response,
                        labels,
                    ),
                    1e-30,
                )
                for segment_locked_response
                in segment_locked_responses
            ],
            axis=0,
        )
        locked_node_transfer = np.median(
            segment_locked_node_transfer,
            axis=0,
        )
        frequency_qa_valid = (
            np.isfinite(snr_db)
            & np.isfinite(phase_consistency)
            & (snr_db >= MIN_TARGET_FREQUENCY_SNR_DB)
            & (
                phase_consistency
                >= MIN_TARGET_PHASE_CONSISTENCY
            )
        )
        fc_valid = (
            response_valid
            & np.isfinite(evoked_fc_z)
            & np.isfinite(evoked_fc_first_half_z)
            & np.isfinite(evoked_fc_second_half_z)
        )
        latency_valid = np.zeros(N_REGIONS, dtype=bool)
        relative_latency_ms = empty.copy()

    return {
        "response": response,
        "node_transfer": node_transfer,
        "response_valid": response_valid,
        "first_half_node_transfer": first_half_node_transfer,
        "second_half_node_transfer": second_half_node_transfer,
        "segment_node_transfer": segment_node_transfer,
        "locked_response": locked_response,
        "locked_node_transfer": locked_node_transfer,
        "sin_coefficient": sin_coefficient,
        "cos_coefficient": cos_coefficient,
        "phase_rad": phase_rad,
        "phase_lag_to_ipsilateral_a1_rad": phase_lag_rad,
        "phase_lag_degrees": phase_lag_degrees,
        **pulse_peaks,
        "segment_locked_node_transfer": (
            segment_locked_node_transfer
        ),
        "fit_r_squared": fit_r_squared,
        "snr_db": snr_db,
        "phase_consistency": phase_consistency,
        "frequency_qa_valid": frequency_qa_valid,
        "evoked_fc_z": evoked_fc_z,
        "evoked_fc_first_half_z": evoked_fc_first_half_z,
        "evoked_fc_second_half_z": evoked_fc_second_half_z,
        "fc_valid": fc_valid,
        "relative_latency_raw_ms": relative_latency_raw_ms,
        "relative_latency_ms": relative_latency_ms,
        "latency_valid": latency_valid,
        "pulse_tail_energy_fraction": pulse_tail_fraction,
        "max_abs_evoked": float(np.max(np.abs(evoked))),
    }


def _hemisphere_index_groups(indices, labels):
    groups = {}
    for hemisphere in ("L", "R", "M"):
        group = np.array(
            [
                int(index) for index in indices
                if label_hemisphere(labels[int(index)]) == hemisphere
            ],
            dtype=int,
        )
        if len(group):
            groups[hemisphere] = group
    return groups


def aggregate_node_metric(values, indices, labels, metric_kind):
    values = np.asarray(values, dtype=float)
    groups = _hemisphere_index_groups(indices, labels)
    hemisphere_values = []
    for group_indices in groups.values():
        group_values = values[group_indices]
        finite = group_values[np.isfinite(group_values)]
        if not len(finite):
            continue
        if metric_kind == "latency":
            hemisphere_values.append(float(np.median(finite)))
        else:
            hemisphere_values.append(float(np.mean(finite)))
    if not hemisphere_values:
        return np.nan
    if metric_kind == "transfer":
        if any(value <= 0 for value in hemisphere_values):
            raise RuntimeError(
                "Transfer aggregation received a nonpositive value."
            )
        return float(
            np.exp(np.mean(np.log(hemisphere_values)))
        )
    return float(np.mean(hemisphere_values))


def _execute_condition_seed_block(
    job,
    weights,
    labels,
    a1_indices,
    network_index_items,
):
    '''Run one matched control plus every requested probe.'''
    ordinal = int(job["ordinal"])
    scope = str(job["scope"])
    severity = float(job["severity"])
    condition_name = str(job["condition"])
    variant = str(job["variant"])
    seed = int(job["seed"])
    probes = tuple(job["probes"])
    b_values = np.asarray(job["b_values"], dtype=float)
    global_coupling = float(job["global_coupling"])
    input_peak_per_ms = float(job["input_peak_per_ms"])
    dt_ms = float(job["dt_ms"])
    export_parcel_traces = bool(
        job.get("export_parcel_traces", False)
    )
    worker_pid = int(os.getpid())

    node_rows = []
    network_rows = []
    manifest_rows = []
    trace_archives = []

    longest_probe_ms = max(PROBE_SIMULATION_MS[probe] for probe in probes)
    control_time, control_psp, control_wall = run_tvb(
        b_values=b_values,
        probe=None,
        global_coupling=global_coupling,
        input_peak_per_ms=input_peak_per_ms,
        seed=seed,
        weights=weights,
        labels=labels,
        a1_indices=a1_indices,
        dt_ms=dt_ms,
        simulation_ms=longest_probe_ms,
    )
    manifest_rows.append(
        {
            "scope": scope,
            "variant": variant,
            "condition": condition_name,
            "severity": severity,
            "seed": seed,
            "probe": "none",
            "simulation_type": "matched_control",
            "global_coupling": global_coupling,
            "input_peak_per_ms": input_peak_per_ms,
            "dt_ms": dt_ms,
            "simulation_ms": longest_probe_ms,
            "b_signature": b_signature(b_values),
            "wall_seconds": float(control_wall),
            "max_abs_psp": float(np.max(np.abs(control_psp))),
            "max_abs_evoked": np.nan,
            "job_ordinal": ordinal,
            "worker_pid": worker_pid,
        }
    )

    for probe in probes:
        probe_simulation_ms = PROBE_SIMULATION_MS[probe]
        stimulated_time, stimulated_psp, stimulated_wall = run_tvb(
            b_values=b_values,
            probe=probe,
            global_coupling=global_coupling,
            input_peak_per_ms=input_peak_per_ms,
            seed=seed,
            weights=weights,
            labels=labels,
            a1_indices=a1_indices,
            dt_ms=dt_ms,
            simulation_ms=probe_simulation_ms,
        )
        control_time_for_probe = control_time[: len(stimulated_time)]
        control_psp_for_probe = control_psp[: len(stimulated_time)]
        if not np.allclose(control_time_for_probe, stimulated_time):
            raise RuntimeError("Control and stimulated time axes differ.")

        metrics = compute_node_metrics(
            stimulated_time,
            stimulated_psp,
            control_psp_for_probe,
            probe,
            labels,
        )
        response = metrics["response"]
        if not np.isfinite(response).all():
            raise RuntimeError("A response-amplitude metric is nonfinite.")

        common = {
            "scope": scope,
            "variant": variant,
            "condition": condition_name,
            "severity": severity,
            "seed": seed,
            "probe": probe,
            "global_coupling": global_coupling,
            "input_peak_per_ms": input_peak_per_ms,
            "dt_ms": dt_ms,
            "b_signature": b_signature(b_values),
        }

        for region_index in range(N_REGIONS):
            node_rows.append(
                {
                    **common,
                    "region_index": region_index,
                    "region_label": labels[region_index],
                    "hemisphere": label_hemisphere(
                        labels[region_index]
                    ),
                    "response": float(response[region_index]),
                    **{
                        f"segment_transfer_{segment_index + 1}": (
                            float(
                                metrics["segment_node_transfer"][
                                    segment_index, region_index
                                ]
                            )
                            if probe in PERIODIC_PROBES
                            else np.nan
                        )
                        for segment_index in range(PERIODIC_SEGMENT_COUNT)
                    },
                    **{
                        f"segment_locked_transfer_{segment_index + 1}": (
                            float(
                                metrics["segment_locked_node_transfer"][
                                    segment_index, region_index
                                ]
                            )
                            if probe in PERIODIC_PROBES
                            else np.nan
                        )
                        for segment_index in range(PERIODIC_SEGMENT_COUNT)
                    },
                    "node_transfer": float(
                        metrics["node_transfer"][region_index]
                    ),
                    "response_valid": bool(
                        metrics["response_valid"][region_index]
                    ),
                    "locked_response": float(
                        metrics["locked_response"][region_index]
                    ),
                    "locked_node_transfer": float(
                        metrics["locked_node_transfer"][region_index]
                    ),
                    "fit_r_squared": float(
                        metrics["fit_r_squared"][region_index]
                    ),
                    "snr_db": float(
                        metrics["snr_db"][region_index]
                    ),
                    "phase_consistency": float(
                        metrics["phase_consistency"][region_index]
                    ),
                    "sin_coefficient": float(
                        metrics["sin_coefficient"][region_index]
                    ),
                    "cos_coefficient": float(
                        metrics["cos_coefficient"][region_index]
                    ),
                    "phase_rad": float(
                        metrics["phase_rad"][region_index]
                    ),
                    "phase_lag_to_ipsilateral_a1_rad": float(
                        metrics[
                            "phase_lag_to_ipsilateral_a1_rad"
                        ][region_index]
                    ),
                    "phase_lag_degrees": float(
                        metrics["phase_lag_degrees"][region_index]
                    ),
                    "frequency_qa_valid": bool(
                        metrics["frequency_qa_valid"][region_index]
                    ),
                    "evoked_fc_z": float(
                        metrics["evoked_fc_z"][region_index]
                    ),
                    "fc_split_abs_z": float(
                        abs(
                            metrics[
                                "evoked_fc_first_half_z"
                            ][region_index]
                            - metrics[
                                "evoked_fc_second_half_z"
                            ][region_index]
                        )
                    ),
                    "fc_valid": bool(
                        metrics["fc_valid"][region_index]
                    ),
                    "relative_latency_raw_ms": float(
                        metrics[
                            "relative_latency_raw_ms"
                        ][region_index]
                    ),
                    "relative_latency_ms": float(
                        metrics["relative_latency_ms"][region_index]
                    ),
                    "latency_valid": bool(
                        metrics["latency_valid"][region_index]
                    ),
                    "pulse_tail_energy_fraction": float(
                        metrics[
                            "pulse_tail_energy_fraction"
                        ][region_index]
                    ),
                    "absolute_peak_response": float(
                        metrics["absolute_peak_response"][region_index]
                    ),
                    "signed_peak_response": float(
                        metrics["signed_peak_response"][region_index]
                    ),
                    "absolute_peak_time_ms": float(
                        metrics["absolute_peak_time_ms"][region_index]
                    ),
                    "peak_time_after_stimulus_onset_ms": float(
                        metrics[
                            "peak_time_after_stimulus_onset_ms"
                        ][region_index]
                    ),
                    "peak_time_relative_to_ipsilateral_a1_ms": float(
                        metrics[
                            "peak_time_relative_to_ipsilateral_a1_ms"
                        ][region_index]
                    ),
                    "total_evoked_energy_psp2_ms": float(
                        metrics[
                            "total_evoked_energy_psp2_ms"
                        ][region_index]
                    ),
                    "b_value": float(b_values[region_index]),
                }
            )

        for network_name, indices in network_index_items:
            indices = np.asarray(indices, dtype=int)
            transfer_value = aggregate_node_metric(
                metrics["node_transfer"],
                indices,
                labels,
                "transfer",
            )
            target_response_coverage = float(
                np.mean(metrics["response_valid"][indices])
            )

            if probe in PERIODIC_PROBES:
                transfer_first_half = aggregate_node_metric(
                    metrics["first_half_node_transfer"],
                    indices,
                    labels,
                    "transfer",
                )
                transfer_second_half = aggregate_node_metric(
                    metrics["second_half_node_transfer"],
                    indices,
                    labels,
                    "transfer",
                )
                transfer_split_abs_log2 = float(
                    abs(
                        np.log2(
                            max(transfer_first_half, 1e-30)
                            / max(transfer_second_half, 1e-30)
                        )
                    )
                )
                segment_transfers = np.array(
                    [
                        aggregate_node_metric(
                            segment_values,
                            indices,
                            labels,
                            "transfer",
                        )
                        for segment_values
                        in metrics["segment_node_transfer"]
                    ],
                    dtype=float,
                )
                segment_locked_transfers = np.array(
                    [
                        aggregate_node_metric(
                            segment_values,
                            indices,
                            labels,
                            "transfer",
                        )
                        for segment_values
                        in metrics[
                            "segment_locked_node_transfer"
                        ]
                    ],
                    dtype=float,
                )
                segment_midpoints_s = (
                    np.arange(PERIODIC_SEGMENT_COUNT, dtype=float)
                    + 0.5
                ) * PERIODIC_SEGMENT_MS / 1000.0
                temporal_slope = float(
                    stats.linregress(
                        segment_midpoints_s,
                        np.log2(
                            np.maximum(segment_transfers, 1e-30)
                        ),
                    ).slope
                )
                evoked_fc_value = aggregate_node_metric(
                    metrics["evoked_fc_z"],
                    indices,
                    labels,
                    "fc",
                )
                evoked_fc_first_half = aggregate_node_metric(
                    metrics["evoked_fc_first_half_z"],
                    indices,
                    labels,
                    "fc",
                )
                evoked_fc_second_half = aggregate_node_metric(
                    metrics["evoked_fc_second_half_z"],
                    indices,
                    labels,
                    "fc",
                )
                evoked_fc_split_abs_z = float(
                    abs(
                        evoked_fc_first_half
                        - evoked_fc_second_half
                    )
                )
                fc_valid_fraction = float(
                    np.mean(metrics["fc_valid"][indices])
                )
                frequency_qa_valid_fraction = float(
                    np.mean(
                        metrics["frequency_qa_valid"][indices]
                    )
                )
                locked_transfer_value = float(
                    np.median(segment_locked_transfers)
                )
                relative_latency_raw_value = np.nan
                relative_latency_value = np.nan
                latency_valid_fraction = np.nan
                median_pulse_tail_fraction = np.nan
            else:
                segment_transfers = np.full(
                    PERIODIC_SEGMENT_COUNT,
                    np.nan,
                    dtype=float,
                )
                segment_locked_transfers = np.full(
                    PERIODIC_SEGMENT_COUNT,
                    np.nan,
                    dtype=float,
                )
                transfer_first_half = np.nan
                transfer_second_half = np.nan
                transfer_split_abs_log2 = np.nan
                temporal_slope = np.nan
                evoked_fc_value = np.nan
                evoked_fc_first_half = np.nan
                evoked_fc_second_half = np.nan
                evoked_fc_split_abs_z = np.nan
                fc_valid_fraction = np.nan
                frequency_qa_valid_fraction = np.nan
                locked_transfer_value = np.nan
                relative_latency_raw_value = (
                    aggregate_node_metric(
                        metrics["relative_latency_raw_ms"],
                        indices,
                        labels,
                        "latency",
                    )
                )
                relative_latency_value = aggregate_node_metric(
                    metrics["relative_latency_ms"],
                    indices,
                    labels,
                    "latency",
                )
                latency_valid_fraction = float(
                    np.mean(metrics["latency_valid"][indices])
                )
                median_pulse_tail_fraction = float(
                    np.nanmedian(
                        metrics[
                            "pulse_tail_energy_fraction"
                        ][indices]
                    )
                )

            network_rows.append(
                {
                    **common,
                    "network": network_name,
                    "network_response": float(
                        np.mean(response[indices])
                    ),
                    "transfer": transfer_value,
                    **{
                        f"segment_transfer_{segment_index + 1}": float(
                            segment_transfers[segment_index]
                        )
                        for segment_index in range(PERIODIC_SEGMENT_COUNT)
                    },
                    **{
                        f"segment_locked_transfer_{segment_index + 1}": float(
                            segment_locked_transfers[segment_index]
                        )
                        for segment_index in range(PERIODIC_SEGMENT_COUNT)
                    },
                    "target_response_coverage": (
                        target_response_coverage
                    ),
                    "transfer_first_half": transfer_first_half,
                    "transfer_second_half": transfer_second_half,
                    "transfer_split_abs_log2": (
                        transfer_split_abs_log2
                    ),
                    "transfer_nonstationary_flag": bool(
                        probe in PERIODIC_PROBES
                        and transfer_split_abs_log2
                        > NONSTATIONARY_HALF_LOG2_FLAG
                    ),
                    "transfer_log2_slope_per_s": temporal_slope,
                    "locked_transfer_segment_median": (
                        locked_transfer_value
                    ),
                    "evoked_fc_z": evoked_fc_value,
                    "evoked_fc_first_half_z": (
                        evoked_fc_first_half
                    ),
                    "evoked_fc_second_half_z": (
                        evoked_fc_second_half
                    ),
                    "evoked_fc_split_abs_z": (
                        evoked_fc_split_abs_z
                    ),
                    "fc_valid_fraction": fc_valid_fraction,
                    "relative_latency_raw_ms": (
                        relative_latency_raw_value
                    ),
                    "relative_latency_ms": relative_latency_value,
                    "latency_valid_fraction": (
                        latency_valid_fraction
                    ),
                    "median_pulse_tail_energy_fraction": (
                        median_pulse_tail_fraction
                    ),
                    "median_target_fit_r_squared": (
                        float(
                            np.nanmedian(
                                metrics["fit_r_squared"][indices]
                            )
                        )
                        if probe in PERIODIC_PROBES
                        else np.nan
                    ),
                    "median_target_snr_db": (
                        float(
                            np.nanmedian(metrics["snr_db"][indices])
                        )
                        if probe in PERIODIC_PROBES
                        else np.nan
                    ),
                    "median_target_phase_consistency": (
                        float(
                            np.nanmedian(
                                metrics["phase_consistency"][indices]
                            )
                        )
                        if probe in PERIODIC_PROBES
                        else np.nan
                    ),
                    "target_frequency_qa_valid_fraction": (
                        frequency_qa_valid_fraction
                    ),
                    "a1_snr_db_min": (
                        float(
                            np.nanmin(
                                metrics["snr_db"][A1_INDICES]
                            )
                        )
                        if probe in PERIODIC_PROBES
                        else np.nan
                    ),
                    "a1_phase_consistency_min": (
                        float(
                            np.nanmin(
                                metrics["phase_consistency"][
                                    A1_INDICES
                                ]
                            )
                        )
                        if probe in PERIODIC_PROBES
                        else np.nan
                    ),
                }
            )

        trace_manifest_fields = {
            "trace_format_version": None,
            "trace_file": None,
            "trace_sha256": None,
            "trace_byte_size": np.nan,
            "trace_sample_count": np.nan,
            "trace_region_count": np.nan,
        }
        if export_parcel_traces:
            trace_archive = build_parcel_trace_archive(
                scope=scope,
                variant=variant,
                condition_name=condition_name,
                severity=severity,
                seed=seed,
                probe=probe,
                dt_ms=dt_ms,
                global_coupling=global_coupling,
                input_peak_per_ms=input_peak_per_ms,
                b_values=b_values,
                time_ms=stimulated_time,
                stimulated_psp=stimulated_psp,
                control_psp=control_psp_for_probe,
                labels=labels,
                a1_indices=a1_indices,
            )
            trace_archives.append(trace_archive)
            trace_manifest_fields = {
                "trace_format_version": TRACE_FORMAT_VERSION,
                "trace_file": str(
                    Path(TRACE_ARCHIVE_SUBDIRECTORY)
                    / trace_archive["filename"]
                ),
                "trace_sha256": trace_archive["sha256"],
                "trace_byte_size": int(trace_archive["byte_size"]),
                "trace_sample_count": int(
                    trace_archive["sample_count"]
                ),
                "trace_region_count": int(
                    trace_archive["region_count"]
                ),
            }

        manifest_rows.append(
            {
                **common,
                **trace_manifest_fields,
                "simulation_type": "stimulated",
                "simulation_ms": probe_simulation_ms,
                "wall_seconds": float(stimulated_wall),
                "max_abs_psp": float(
                    np.max(np.abs(stimulated_psp))
                ),
                "max_abs_evoked": metrics["max_abs_evoked"],
                "job_ordinal": ordinal,
                "worker_pid": worker_pid,
            }
        )
        del stimulated_psp, control_psp_for_probe, metrics
        gc.collect()

    del control_psp
    gc.collect()
    return {
        "ordinal": ordinal,
        "node_rows": node_rows,
        "network_rows": network_rows,
        "manifest_rows": manifest_rows,
        "trace_archives": trace_archives,
    }


def execute_grid(
    scope,
    conditions,
    seeds,
    probes,
    global_coupling,
    input_peak_per_ms,
    dt_ms=MAIN_DT_MS,
):
    conditions = list(conditions)
    seeds = [int(seed) for seed in seeds]
    probes = tuple(probes)
    if not conditions or not seeds or not probes:
        raise ValueError(
            "conditions, seeds, and probes must all be nonempty."
        )

    jobs = []
    for condition in conditions:
        severity = float(condition["severity"])
        condition_name = str(condition["condition"])
        b_values = np.asarray(condition["b_values"], dtype=float)
        variant = str(condition.get("variant", condition_name))
        job_scope = str(condition.get("scope", scope))
        job_coupling = float(
            condition.get("global_coupling", global_coupling)
        )
        job_input_peak = float(
            condition.get("input_peak_per_ms", input_peak_per_ms)
        )
        job_export_parcel_traces = bool(
            condition.get(
                "export_parcel_traces",
                job_scope in TRACE_EXPORT_SCOPES,
            )
        )
        for seed in seeds:
            jobs.append(
                {
                    "ordinal": len(jobs),
                    "scope": job_scope,
                    "variant": variant,
                    "condition": condition_name,
                    "severity": severity,
                    "b_values": b_values,
                    "seed": seed,
                    "probes": probes,
                    "global_coupling": job_coupling,
                    "input_peak_per_ms": job_input_peak,
                    "dt_ms": float(dt_ms),
                    "export_parcel_traces": job_export_parcel_traces,
                }
            )

    network_index_items = tuple(
        (
            network_name,
            np.asarray(indices, dtype=int),
        )
        for network_name, indices in NETWORK_INDICES.items()
    )
    trace_stage = any(
        job["export_parcel_traces"] for job in jobs
    )
    stage_description = (
        f"{scope} {TRACE_CHECKPOINT_TAG} condition-seed blocks"
        if trace_stage
        else f"{scope} condition-seed blocks"
    )
    outcomes = run_parallel_jobs(
        _execute_condition_seed_block,
        jobs,
        (
            WEIGHTS,
            LABELS,
            A1_INDICES,
            network_index_items,
        ),
        stage_description,
    )

    node_rows = []
    network_rows = []
    manifest_rows = []
    for outcome in sorted(
        outcomes, key=lambda item: item["ordinal"]
    ):
        for trace_archive in outcome.get("trace_archives", ()):
            write_parcel_trace_archive(trace_archive)
        node_rows.extend(outcome["node_rows"])
        network_rows.extend(outcome["network_rows"])
        manifest_rows.extend(outcome["manifest_rows"])

    expected_node_rows = len(jobs) * len(probes) * N_REGIONS
    expected_network_rows = (
        len(jobs) * len(probes) * len(NETWORK_INDICES)
    )
    expected_manifest_rows = len(jobs) * (1 + len(probes))
    if len(node_rows) != expected_node_rows:
        raise RuntimeError(
            f"Expected {expected_node_rows} node rows, "
            f"received {len(node_rows)}."
        )
    if len(network_rows) != expected_network_rows:
        raise RuntimeError(
            f"Expected {expected_network_rows} network rows, "
            f"received {len(network_rows)}."
        )
    if len(manifest_rows) != expected_manifest_rows:
        raise RuntimeError(
            f"Expected {expected_manifest_rows} manifest rows, "
            f"received {len(manifest_rows)}."
        )
    expected_trace_archives = sum(
        int(job["export_parcel_traces"]) * len(probes)
        for job in jobs
    )
    trace_manifest_rows = [
        row for row in manifest_rows
        if row.get("trace_file") is not None
    ]
    if len(trace_manifest_rows) != expected_trace_archives:
        raise RuntimeError(
            "Raw trace export is incomplete: expected "
            f"{expected_trace_archives}, found {len(trace_manifest_rows)}."
        )
    missing_trace_files = [
        row["trace_file"]
        for row in trace_manifest_rows
        if not (RESULTS_DIR / row["trace_file"]).is_file()
    ]
    if missing_trace_files:
        raise RuntimeError(
            f"Raw trace archives were not written: {missing_trace_files[:5]}"
        )

    return (
        pd.DataFrame(node_rows),
        pd.DataFrame(network_rows),
        pd.DataFrame(manifest_rows),
    )


def normalize_to_baseline(network_df, baseline_df=None):
    target = network_df.copy()
    if baseline_df is None:
        source = target
        merge_keys = ["scope", "seed", "probe", "network"]
    else:
        source = baseline_df.copy()
        merge_keys = ["seed", "probe", "network"]

    baseline_columns = [
        *merge_keys,
        "transfer",
        "evoked_fc_z",
        "relative_latency_ms",
    ]
    baseline = source[source["severity"] == 0.0][
        baseline_columns
    ].rename(
        columns={
            "transfer": "baseline_transfer",
            "evoked_fc_z": "baseline_evoked_fc_z",
            "relative_latency_ms": "baseline_relative_latency_ms",
        }
    )
    target = target.merge(
        baseline,
        on=merge_keys,
        how="left",
        validate="many_to_one",
    )
    if target["baseline_transfer"].isna().any():
        raise RuntimeError("A baseline transfer value is missing.")

    target["log2_transfer_vs_baseline"] = np.log2(
        np.maximum(target["transfer"], 1e-30)
        / np.maximum(target["baseline_transfer"], 1e-30)
    )
    target["evoked_fc_z_vs_baseline"] = (
        target["evoked_fc_z"] - target["baseline_evoked_fc_z"]
    )
    target["latency_ms_vs_baseline"] = (
        target["relative_latency_ms"]
        - target["baseline_relative_latency_ms"]
    )
    return target


METRIC_DEFINITIONS = {
    "transfer_gain": {
        "column": "log2_transfer_vs_baseline",
        "probes": PERIODIC_PROBES,
        "unit": "log2 ratio",
    },
    "functional_connectivity": {
        "column": "evoked_fc_z_vs_baseline",
        "probes": PERIODIC_PROBES,
        "unit": "Fisher-z difference",
    },
    "response_latency": {
        "column": "latency_ms_vs_baseline",
        "probes": ("pulse",),
        "unit": "ms",
    },
}


def make_pair_interactions(normalized_df, pair_definitions):
    rows = []
    index_columns = [
        "scope",
        "variant",
        "condition",
        "severity",
        "seed",
        "probe",
        "global_coupling",
        "input_peak_per_ms",
        "dt_ms",
    ]
    for pair_name, (
        semantic_network,
        episodic_network,
    ) in pair_definitions.items():
        for outcome_name, specification in METRIC_DEFINITIONS.items():
            for probe in specification["probes"]:
                subset = normalized_df[
                    (normalized_df["probe"] == probe)
                    & (
                        normalized_df["network"].isin(
                            [semantic_network, episodic_network]
                        )
                    )
                ]
                if subset.empty:
                    continue
                pivot = subset.pivot(
                    index=index_columns,
                    columns="network",
                    values=specification["column"],
                ).reset_index()
                if (
                    semantic_network not in pivot.columns
                    or episodic_network not in pivot.columns
                ):
                    raise RuntimeError(
                        f"Missing networks for pair {pair_name!r}, "
                        f"outcome {outcome_name!r}, probe {probe!r}."
                    )
                for record in pivot.to_dict("records"):
                    semantic_value = float(record[semantic_network])
                    episodic_value = float(record[episodic_network])
                    rows.append(
                        {
                            **{
                                column: record[column]
                                for column in index_columns
                            },
                            "pair": pair_name,
                            "semantic_network": semantic_network,
                            "episodic_network": episodic_network,
                            "outcome": outcome_name,
                            "unit": specification["unit"],
                            "semantic_value": semantic_value,
                            "episodic_value": episodic_value,
                            "semantic_minus_episodic_interaction": (
                                semantic_value - episodic_value
                            ),
                        }
                    )
    return pd.DataFrame(rows)


def interaction_statistics(
    interaction_df,
    additional_group_columns=(),
):
    rows = []
    group_columns = [
        "pair",
        "outcome",
        "probe",
        "unit",
        *list(additional_group_columns),
    ]
    for group_key, group in interaction_df.groupby(
        group_columns, sort=False
    ):
        endpoint = group[group["severity"] == 1.0].sort_values("seed")
        values = endpoint[
            "semantic_minus_episodic_interaction"
        ].to_numpy(dtype=float)
        if len(values) < 1:
            continue

        curvature_values = []
        for seed, seed_group in group.groupby("seed"):
            severity_values = {
                float(row.severity): float(
                    row.semantic_minus_episodic_interaction
                )
                for row in seed_group.itertuples(index=False)
            }
            if {0.0, 0.5, 1.0}.issubset(severity_values):
                curvature_values.append(
                    severity_values[0.5]
                    - 0.5 * (
                        severity_values[0.0]
                        + severity_values[1.0]
                    )
                )

        n = len(values)
        mean_value = float(np.mean(values))
        sd_value = float(np.std(values, ddof=1)) if n > 1 else np.nan
        if n > 1 and sd_value > 1e-15:
            t_critical = float(stats.t.ppf(0.975, n - 1))
            half_width = t_critical * sd_value / np.sqrt(n)
            hedges_correction = 1.0 - 3.0 / (4.0 * n - 5.0)
            hedges_gz = hedges_correction * mean_value / sd_value
        elif n > 1:
            half_width = 0.0
            hedges_gz = np.nan
        else:
            half_width = np.nan
            hedges_gz = np.nan

        rows.append(
            {
                **dict(zip(group_columns, group_key)),
                "numerical_initializations": n,
                "mean_interaction": mean_value,
                "median_interaction": float(np.median(values)),
                "minimum_interaction": float(np.min(values)),
                "maximum_interaction": float(np.max(values)),
                "ci95_lower_numerical": (
                    mean_value - half_width
                    if np.isfinite(half_width)
                    else np.nan
                ),
                "ci95_upper_numerical": (
                    mean_value + half_width
                    if np.isfinite(half_width)
                    else np.nan
                ),
                "paired_hedges_gz": hedges_gz,
                "positive_fraction": float(np.mean(values > 0)),
                "sign_consistent": bool(
                    np.all(values > 0) or np.all(values < 0)
                ),
                "mean_interaction_curvature": (
                    float(np.mean(curvature_values))
                    if curvature_values
                    else np.nan
                ),
            }
        )
    return pd.DataFrame(rows)


def build_science_validity_table(network_df, stage):
    required = network_df[
        network_df["network"].isin(PRIMARY_INFERENTIAL_NETWORKS)
    ].copy()
    rows = []
    for row in required.itertuples(index=False):
        periodic = row.probe in PERIODIC_PROBES
        pulse = row.probe == "pulse"
        a1_frequency_passed = bool(
            not periodic
            or (
                row.a1_snr_db_min >= A1_SNR_GATE_DB
                and row.a1_phase_consistency_min
                >= A1_PHASE_CONSISTENCY_GATE
            )
        )
        response_coverage_passed = bool(
            row.target_response_coverage
            >= MIN_TARGET_RESPONSE_COVERAGE
        )
        transfer_valid = bool(
            periodic
            and a1_frequency_passed
            and response_coverage_passed
        )
        fc_valid = bool(
            periodic
            and transfer_valid
            and row.fc_valid_fraction
            >= MIN_TARGET_RESPONSE_COVERAGE
            and row.evoked_fc_split_abs_z
            <= MAX_FC_SPLIT_ABS_Z
        )
        latency_valid = bool(
            pulse
            and response_coverage_passed
            and row.latency_valid_fraction
            >= MIN_TARGET_RESPONSE_COVERAGE
            and row.median_pulse_tail_energy_fraction
            <= MAX_PULSE_TAIL_FRACTION
        )
        frequency_locked_valid = bool(
            periodic
            and transfer_valid
            and row.target_frequency_qa_valid_fraction
            >= MIN_TARGET_RESPONSE_COVERAGE
        )
        rows.append(
            {
                "stage": str(stage),
                "seed": int(row.seed),
                "severity": float(row.severity),
                "condition": str(row.condition),
                "probe": str(row.probe),
                "network": str(row.network),
                "a1_frequency_passed": a1_frequency_passed,
                "target_response_coverage": float(
                    row.target_response_coverage
                ),
                "response_coverage_passed": (
                    response_coverage_passed
                ),
                "transfer_valid": transfer_valid,
                "evoked_fc_split_abs_z": float(
                    row.evoked_fc_split_abs_z
                ),
                "fc_valid_fraction": float(
                    row.fc_valid_fraction
                ),
                "functional_connectivity_valid": fc_valid,
                "latency_valid_fraction": float(
                    row.latency_valid_fraction
                ),
                "median_pulse_tail_energy_fraction": float(
                    row.median_pulse_tail_energy_fraction
                ),
                "response_latency_valid": latency_valid,
                "target_frequency_qa_valid_fraction": float(
                    row.target_frequency_qa_valid_fraction
                ),
                "frequency_locked_transfer_valid": (
                    frequency_locked_valid
                ),
                "transfer_split_abs_log2": float(
                    row.transfer_split_abs_log2
                ),
                "transfer_nonstationary_flag": bool(
                    row.transfer_nonstationary_flag
                ),
                "required_metric_gate_passed": bool(
                    (periodic and transfer_valid and fc_valid)
                    or (pulse and latency_valid)
                ),
            }
        )
    return pd.DataFrame(rows)


def summarize_outcome_eligibility(validity_df):
    definitions = [
        (
            "transfer_gain",
            PERIODIC_PROBES,
            "transfer_valid",
            "confirmatory_primary",
        ),
        (
            "functional_connectivity",
            PERIODIC_PROBES,
            "functional_connectivity_valid",
            "confirmatory_secondary",
        ),
        (
            "response_latency",
            ("pulse",),
            "response_latency_valid",
            "confirmatory_secondary",
        ),
        (
            "frequency_locked_transfer",
            PERIODIC_PROBES,
            "frequency_locked_transfer_valid",
            "sensitivity_only",
        ),
    ]
    rows = []
    for outcome, probes, validity_column, hierarchy in definitions:
        for probe in probes:
            subset = validity_df[
                validity_df["probe"] == probe
            ]
            if subset.empty:
                continue
            passed = subset[validity_column].astype(bool)
            all_passed = bool(passed.all())
            rows.append(
                {
                    "outcome": outcome,
                    "probe": probe,
                    "hierarchy": hierarchy,
                    "valid_rows": int(passed.sum()),
                    "required_rows": int(len(passed)),
                    "all_quality_gates_passed": all_passed,
                    "analysis_status": (
                        "eligible_as_prespecified"
                        if all_passed
                        else "descriptive_only_quality_gate_failed"
                    ),
                }
            )
    return pd.DataFrame(rows)


## 10. Machine-readable analysis lock

The complete primary specification and the secondary prolonged-stimulation
follow-up are serialized before simulation. No fabricated trajectories,
dummy observations, or placeholder outputs are used.


In [ ]:
ANALYSIS_SPEC = {
    "specification_version": "semantic_episodic_v6_late_window_followup_2026-08-02",
    "research_question": (
        "How does increasing AD-like amyloid-linked inhibitory "
        "perturbation differentially affect stimulus-evoked transmission "
        "into expanded musical-semantic-associated and "
        "musical-episodic-associated proxy parcel sets?"
    ),
    "primary_networks": {
        "semantic_expanded": list(SEMANTIC_EXPANDED_LABELS),
        "episodic_expanded": list(EPISODIC_EXPANDED_LABELS),
    },
    "definition_sensitivities": {
        pair_name: {
            "semantic_network": pair_networks[0],
            "episodic_network": pair_networks[1],
        }
        for pair_name, pair_networks in NETWORK_PAIRS.items()
        if pair_name != "expanded_bilateral"
    },
    "main_numerical_seeds": list(CFG["seeds"]),
    "counterfactual_numerical_seeds": list(CFG["seeds"]),
    "parameter_sensitivity_seeds": list(
        CFG["sensitivity_seeds"]
    ),
    "integration_step_seeds": list(CFG["dt_check_seeds"]),
    "severities": list(CFG["severities"]),
    "probes": list(PROBES),
    "windows_ms": {
        "stimulus_onset": STIMULUS_ONSET_MS,
        "pulse_width": PULSE_WIDTH_MS,
        "pulse_analysis_end": PULSE_ANALYSIS_END_MS,
        "periodic_settling_end": PERIODIC_SETTLING_END_MS,
        "periodic_analysis_start": PERIODIC_ANALYSIS_START_MS,
        "periodic_analysis_end": PERIODIC_ANALYSIS_END_MS,
    },
    "metrics": {
        "transfer_gain": {
            "node": (
                "linearly detrended evoked PSP AC-RMS / ipsilateral A1 "
                "AC-RMS in the fixed 10-second window"
            ),
            "network": (
                "geometric mean of left and right hemisphere means; "
                "one-sided sets retain their available hemisphere"
            ),
            "baseline_reference": "log2(network / own baseline)",
            "temporal_diagnostic": (
                "five 2-second transfer bins, log2 slope, and "
                "first-half versus second-half absolute log2 difference"
            ),
            "frequency_locked_sensitivity": (
                "median of five 2-second exact-frequency transfer gains"
            ),
        },
        "functional_connectivity": {
            "node": (
                "Fisher-z Pearson correlation between matched-control-"
                "subtracted evoked target and ipsilateral A1 PSP traces"
            ),
            "network": (
                "arithmetic mean of hemisphere means with equal "
                "hemisphere weight"
            ),
            "baseline_reference": (
                "network evoked FC z minus own baseline"
            ),
            "interpretation": (
                "common-input-sensitive model PSP functional "
                "connectivity, not causal effective connectivity"
            ),
        },
        "response_latency": {
            "node": (
                "50% fractional-energy timing minus ipsilateral A1 timing"
            ),
            "saved_pulse_descriptors": (
                "absolute and signed peak, absolute peak time, peak time "
                "relative to ipsilateral A1, and integrated evoked energy"
            ),
            "network": (
                "arithmetic mean of left/right hemisphere medians"
            ),
            "baseline_reference": (
                "network relative latency minus own baseline, ms"
            ),
            "interpretation": (
                "relative model-response timing; zero tract delays mean "
                "this is not anatomical conduction latency"
            ),
        },
    },
    "frequency_qa": {
        "method": "DPSS multitaper spectrum",
        "harmonic_phase_convention": (
            "y_f=A*sin(2*pi*f*t+phase); phase=atan2(beta_cos,beta_sin); "
            "target lag is wrapped target phase minus ipsilateral A1 phase"
        ),
        "time_bandwidth": MULTITAPER_TIME_BANDWIDTH,
        "tapers": MULTITAPER_TAPERS,
        "snr_signal_half_width_hz": SNR_SIGNAL_HALF_WIDTH_HZ,
        "snr_flank_inner_hz": SNR_FLANK_INNER_HZ,
        "snr_flank_outer_hz": SNR_FLANK_OUTER_HZ,
        "a1_snr_gate_db": A1_SNR_GATE_DB,
        "a1_phase_consistency_gate": (
            A1_PHASE_CONSISTENCY_GATE
        ),
        "target_frequency_snr_gate_db": (
            MIN_TARGET_FREQUENCY_SNR_DB
        ),
        "target_phase_consistency_gate": (
            MIN_TARGET_PHASE_CONSISTENCY
        ),
    },
    "science_validity_gates": {
        "preflight_seed_count": PREFLIGHT_SEED_COUNT,
        "target_response_ratio_floor": (
            TARGET_RESPONSE_RATIO_FLOOR
        ),
        "minimum_target_response_coverage": (
            MIN_TARGET_RESPONSE_COVERAGE
        ),
        "maximum_fc_split_abs_z": MAX_FC_SPLIT_ABS_Z,
        "maximum_median_pulse_tail_fraction": (
            MAX_PULSE_TAIL_FRACTION
        ),
        "pulse_tail_window_ms": PULSE_TAIL_WINDOW_MS,
        "nonstationary_transfer_flag_abs_log2": (
            NONSTATIONARY_HALF_LOG2_FLAG
        ),
    },
    "statistics": {
        "interaction": (
            "semantic severity slope minus episodic severity slope; "
            "with severity coded 0, 0.5, 1 this equals the endpoint "
            "difference in baseline-referenced values"
        ),
        "interval": (
            "two-sided t interval across paired numerical "
            "initializations"
        ),
        "effect_size": "paired Hedges g_z",
        "curvature": (
            "midpoint interaction minus mean of endpoint interactions"
        ),
    },
    "robustness": {
        "spatial_shuffles": int(CFG["spatial_shuffles"]),
        "spatial_shuffle_rng_seed": 3792026,
        "matched_null_sets": int(CFG["matched_null_sets"]),
        "matched_null_rng_seed": 20260729,
        "matched_features": [
            "log weighted strength",
            "log direct A1 affinity",
            "local b reduction",
            "hemisphere",
        ],
    },
    "integration_step_gates": {
        "comparison_unit": (
            "absolute bias in the finite 20-seed mean baseline-to-high "
            "semantic-minus-episodic interaction at 0.5 versus 0.25 ms"
        ),
        "reference_seeds": list(CFG["dt_check_seeds"]),
        "transfer_interaction_log2_absolute": (
            DT_TRANSFER_INTERACTION_LOG2_TOLERANCE
        ),
        "fc_interaction_absolute_z": DT_FC_ABSOLUTE_TOLERANCE,
        "latency_interaction_ms": DT_LATENCY_TOLERANCE_MS,
        "a1_snr_max_seed_difference_db": (
            DT_A1_SNR_TOLERANCE_DB
        ),
        "direction_rule": (
            "main and reference interaction means must share a sign, "
            "unless both are within the outcome tolerance of zero"
        ),
        "inference_rule": (
            "the 95% t-intervals must give the same two-sided "
            "zero-exclusion conclusion at both integration steps"
        ),
        "pointwise_seed_differences": (
            "retained as diagnostics; not a hard gate for sensitive "
            "nonlinear trajectories"
        ),
        "fatal_validity_rule": (
            "nonfinite estimates, interaction-direction reversal, changed "
            "95% interval conclusion, or failed A1 stimulation quality stop "
            "the notebook"
        ),
        "precision_rule": (
            "the unchanged transfer, FC, latency, and A1-SNR tolerances "
            "produce warnings and exact-magnitude eligibility labels, not "
            "a fatal stop"
        ),
        "paired_step_bias_interval": (
            "two-sided t interval across the 20 paired main-minus-reference "
            "seed-level interaction differences"
        ),
        "amendment_basis": (
            "the completed 20-seed comparison preserved interaction "
            "direction and the 95% interval conclusion while exceeding "
            "three exact-magnitude precision targets"
        ),
    },
    "raw_trace_export": {
        "scope": list(TRACE_EXPORT_SCOPES),
        "format_version": TRACE_FORMAT_VERSION,
        "checkpoint_tag": TRACE_CHECKPOINT_TAG,
        "regions": list(TRACE_REGION_LABELS),
        "arrays": [
            "stimulated_psp",
            "control_psp",
            "evoked_psp",
            "time_ms",
            "stimulus_temporal_waveform",
            "stimulus_waveform",
        ],
        "trace_transform": (
            "none; float PSP monitor samples are saved directly"
        ),
        "compression": "lossless NumPy compressed archive",
    },
    "prolonged_stimulation_followup": {
        "hierarchy": "secondary_stability_and_entrainment_followup",
        "changes_to_primary_method": "none",
        "simulation_end_ms": FOLLOWUP_SIMULATION_END_MS,
        "windows_ms": FOLLOWUP_WINDOWS_MS,
        "probes": list(FOLLOWUP_PERIODIC_PROBES),
        "seeds": list(CFG["seeds"]),
        "severities": list(CFG["severities"]),
        "dt_values_ms": list(FOLLOWUP_DT_VALUES_MS),
        "zero_input_control_retained": True,
        "dc_matched_control": (
            "constant A/2 input over the same onset and offset; the "
            "periodic-minus-DC result is a paired nonlinear contrast, "
            "not an additive decomposition"
        ),
        "saved_metrics": [
            "broadband evoked-response ratio",
            "five 2-second broadband segments per window",
            "full-window and segment-median exact-frequency transfer",
            "harmonic R-squared and fitted phase",
            "cycle phase consistency",
            "multitaper dominant frequency and applied-frequency power ratio",
            "multitaper magnitude-squared A1-target coherence",
            "raw and baseline-referenced evoked FC",
            "first-half versus second-half transfer and FC",
            "ten-segment slopes",
        ],
        "stability_assessment": {
            "method": "paired two-one-sided equivalence tests across numerical seeds",
            "transfer_margin_log2": FOLLOWUP_TRANSFER_EQUIVALENCE_MARGIN_LOG2,
            "transfer_margin_interpretation": "operational plus/minus 20 percent",
            "fc_margin_fisher_z": FOLLOWUP_FC_EQUIVALENCE_MARGIN_Z,
            "alpha_each_one_sided": FOLLOWUP_EQUIVALENCE_ALPHA,
            "margin_status": "operational_not_biologically_validated",
        },
        "trace_format_version": FOLLOWUP_TRACE_FORMAT_VERSION,
    },
    "sources": {
        "Platel_2003": "doi:10.1016/S1053-8119(03)00287-8",
        "Slattery_2019": "doi:10.1016/j.cortex.2019.02.003",
        "Rolls_2023": "doi:10.1002/hbm.26089",
        "Friston_1993": "pmid:8417010",
        "Glasser_2016": "doi:10.1038/nature18933",
        "Stefanovski_2019": "doi:10.3389/fncom.2019.00054",
        "Carter_1987": "doi:10.1109/PROC.1987.13723",
        "Schuirmann_1987": "doi:10.1007/BF01068419",
    },
}
analysis_spec_json = json.dumps(
    ANALYSIS_SPEC,
    sort_keys=True,
    separators=(",", ":"),
)
ANALYSIS_SPEC_SHA256 = hashlib.sha256(
    analysis_spec_json.encode("utf-8")
).hexdigest()
(RESULTS_DIR / "analysis_spec.json").write_text(
    json.dumps(ANALYSIS_SPEC, indent=2) + "\n"
)
print("Analysis specification locked without fabricated data.")
print("Analysis specification SHA-256:", ANALYSIS_SPEC_SHA256)


## 11. Baseline-only coupling diagnostic

Global coupling remains fixed at 60 from the prior design. This scan is a
safety and scale diagnostic, not a procedure for selecting the coupling
that produces a preferred semantic-versus-episodic result.


In [ ]:
def _run_calibration_block(
    job,
    weights,
    labels,
    a1_indices,
    semantic_indices,
    episodic_indices,
):
    candidate_g = float(job["global_coupling"])
    calibration_seed = int(job["seed"])
    ordinal = int(job["ordinal"])
    simulation_ms = float(job["simulation_ms"])

    control_time, control_psp, control_wall = run_tvb(
        b_values=BASELINE_B,
        probe=None,
        global_coupling=candidate_g,
        input_peak_per_ms=MAIN_INPUT_PEAK_PER_MS,
        seed=calibration_seed,
        weights=weights,
        labels=labels,
        a1_indices=a1_indices,
        simulation_ms=simulation_ms,
    )
    pulse_time, pulse_psp, pulse_wall = run_tvb(
        b_values=BASELINE_B,
        probe="pulse",
        global_coupling=candidate_g,
        input_peak_per_ms=MAIN_INPUT_PEAK_PER_MS,
        seed=calibration_seed,
        weights=weights,
        labels=labels,
        a1_indices=a1_indices,
        simulation_ms=simulation_ms,
    )
    if not np.allclose(control_time, pulse_time):
        raise RuntimeError("Calibration time axes differ.")
    metrics = compute_node_metrics(
        pulse_time,
        pulse_psp,
        control_psp,
        "pulse",
        labels,
    )
    semantic_transfer = aggregate_node_metric(
        metrics["node_transfer"],
        semantic_indices,
        labels,
        "transfer",
    )
    episodic_transfer = aggregate_node_metric(
        metrics["node_transfer"],
        episodic_indices,
        labels,
        "transfer",
    )
    return {
        "ordinal": ordinal,
        "row": {
            "global_coupling": candidate_g,
            "semantic_transfer": semantic_transfer,
            "episodic_transfer": episodic_transfer,
            "balanced_target_score": float(
                np.sqrt(semantic_transfer * episodic_transfer)
            ),
            "max_abs_evoked": metrics["max_abs_evoked"],
            "wall_seconds": float(control_wall + pulse_wall),
            "worker_pid": int(os.getpid()),
        },
    }


calibration_jobs = [
    {
        "ordinal": ordinal,
        "global_coupling": candidate_g,
        "seed": CFG["seeds"][0],
        "simulation_ms": PULSE_ANALYSIS_END_MS,
    }
    for ordinal, candidate_g in enumerate(
        CFG["calibration_couplings"]
    )
]
calibration_outcomes = run_parallel_jobs(
    _run_calibration_block,
    calibration_jobs,
    (
        WEIGHTS,
        LABELS,
        A1_INDICES,
        NETWORK_INDICES["semantic_expanded"],
        NETWORK_INDICES["episodic_expanded"],
    ),
    "baseline coupling candidates",
)
calibration_df = pd.DataFrame(
    [
        outcome["row"]
        for outcome in sorted(
            calibration_outcomes,
            key=lambda item: item["ordinal"],
        )
    ]
)
if not np.isfinite(
    calibration_df.select_dtypes("number")
).all().all():
    raise RuntimeError("Calibration produced a nonfinite value.")
selected_calibration = calibration_df[
    calibration_df["global_coupling"] == MAIN_GLOBAL_COUPLING
]
if selected_calibration.empty:
    warnings.warn(
        "The selected coupling is absent from this shortened scan."
    )
elif float(
    selected_calibration["max_abs_evoked"].iloc[0]
) >= 50.0:
    raise RuntimeError(
        "The selected coupling produced a saturated response."
    )
display(calibration_df)

fig, ax = plt.subplots(figsize=(7.2, 4.2))
ax.plot(
    calibration_df["global_coupling"],
    calibration_df["semantic_transfer"],
    marker="o",
    label="Semantic expanded proxy",
)
ax.plot(
    calibration_df["global_coupling"],
    calibration_df["episodic_transfer"],
    marker="o",
    label="Episodic expanded proxy",
)
ax.axvline(
    MAIN_GLOBAL_COUPLING,
    color="black",
    linestyle="--",
    label="Locked main G",
)
ax.set(
    xlabel="Global coupling G",
    ylabel="Pulse target / ipsilateral A1",
    title="Baseline-only coupling diagnostic",
)
ax.legend()
fig.tight_layout()
fig.savefig(
    FIGURE_DIR / "01_baseline_coupling_diagnostic.png",
    dpi=180,
)
plt.show()


## 12. Technical preflight, main paired experiment, and interaction

Each numerical seed is reused across all severities and probes. The
inferential unit is the paired seed-level semantic-minus-episodic
interaction. The 95% interval describes variation across numerical
initializations only.

Before the remaining jobs start, the first two numerical seeds run at the
baseline and high endpoints. Only measurement validity is inspected:
response coverage, A1 drive quality, FC split-window agreement, and pulse
latency coverage. The semantic-minus-episodic direction is not used as a
gate. These preflight simulations are reused in the final main dataset.


In [ ]:
main_condition_by_severity = {
    severity: {
        "condition": SEVERITY_LABELS[severity],
        "severity": severity,
        "b_values": B_BY_SEVERITY[severity],
        "variant": "full_field",
        "export_parcel_traces": True,
    }
    for severity in CFG["severities"]
}
preflight_seeds = list(
    CFG["seeds"][: min(PREFLIGHT_SEED_COUNT, len(CFG["seeds"]))]
)
preflight_severities = [
    severity
    for severity in (0.0, 1.0)
    if severity in main_condition_by_severity
]
(
    preflight_node_df,
    preflight_network_df,
    preflight_manifest_df,
) = execute_grid(
    scope="main_full_field",
    conditions=[
        main_condition_by_severity[severity]
        for severity in preflight_severities
    ],
    seeds=preflight_seeds,
    probes=PROBES,
    global_coupling=MAIN_GLOBAL_COUPLING,
    input_peak_per_ms=MAIN_INPUT_PEAK_PER_MS,
)

preflight_science_validity_df = build_science_validity_table(
    preflight_network_df,
    stage="technical_preflight",
)
display(preflight_science_validity_df)
if not preflight_science_validity_df[
    "required_metric_gate_passed"
].all():
    failed_preflight = preflight_science_validity_df[
        ~preflight_science_validity_df[
            "required_metric_gate_passed"
        ]
    ]
    display(failed_preflight)
    raise RuntimeError(
        "The prespecified scientific measurement preflight failed. "
        "The remaining final jobs were not started."
    )

main_grid_parts = [
    (
        preflight_node_df,
        preflight_network_df,
        preflight_manifest_df,
    )
]
for severity in CFG["severities"]:
    remaining_seeds = [
        seed
        for seed in CFG["seeds"]
        if not (
            severity in preflight_severities
            and seed in preflight_seeds
        )
    ]
    if not remaining_seeds:
        continue
    main_grid_parts.append(
        execute_grid(
            scope="main_full_field",
            conditions=[main_condition_by_severity[severity]],
            seeds=remaining_seeds,
            probes=PROBES,
            global_coupling=MAIN_GLOBAL_COUPLING,
            input_peak_per_ms=MAIN_INPUT_PEAK_PER_MS,
        )
    )

main_node_df = pd.concat(
    [part[0] for part in main_grid_parts],
    ignore_index=True,
)
main_network_df = pd.concat(
    [part[1] for part in main_grid_parts],
    ignore_index=True,
)
main_manifest_df = pd.concat(
    [part[2] for part in main_grid_parts],
    ignore_index=True,
)
main_manifest_df["job_ordinal"] = (
    main_manifest_df.groupby(
        ["severity", "seed"],
        sort=True,
    ).ngroup()
)

expected_main_blocks = (
    len(CFG["severities"]) * len(CFG["seeds"])
)
observed_main_blocks = main_network_df[
    ["severity", "seed"]
].drop_duplicates()
if len(observed_main_blocks) != expected_main_blocks:
    raise RuntimeError(
        "The assembled main grid is incomplete or duplicated."
    )
if main_node_df.duplicated(
    ["severity", "seed", "probe", "region_index"]
).any():
    raise RuntimeError("The assembled main node table has duplicates.")
if main_network_df.duplicated(
    ["severity", "seed", "probe", "network"]
).any():
    raise RuntimeError(
        "The assembled main network table has duplicates."
    )

required_trace_columns = {
    "trace_file",
    "trace_sha256",
    "trace_sample_count",
    "trace_region_count",
}
missing_trace_columns = required_trace_columns - set(main_manifest_df.columns)
if missing_trace_columns:
    raise RuntimeError(
        "Main checkpoints predate raw-trace export. Rerun the "
        f"{TRACE_CHECKPOINT_TAG!r} main stage; missing columns: "
        f"{sorted(missing_trace_columns)}"
    )
main_trace_manifest_df = main_manifest_df[
    main_manifest_df["trace_file"].notna()
].copy()
expected_main_trace_archives = (
    len(CFG["severities"]) * len(CFG["seeds"]) * len(PROBES)
)
if len(main_trace_manifest_df) != expected_main_trace_archives:
    raise RuntimeError(
        "The assembled main raw-trace grid is incomplete: expected "
        f"{expected_main_trace_archives}, found "
        f"{len(main_trace_manifest_df)}."
    )
if not (
    main_trace_manifest_df["trace_region_count"].astype(int) == 34
).all():
    raise RuntimeError("A main raw-trace shard has the wrong parcel count.")
if main_trace_manifest_df.duplicated(
    ["severity", "seed", "probe"]
).any():
    raise RuntimeError("The main raw-trace manifest has duplicates.")

main_science_validity_df = build_science_validity_table(
    main_network_df,
    stage="complete_main",
)
outcome_eligibility_df = summarize_outcome_eligibility(
    main_science_validity_df
)
display(outcome_eligibility_df)

main_normalized_df = normalize_to_baseline(main_network_df)
main_pair_interaction_df = make_pair_interactions(
    main_normalized_df,
    NETWORK_PAIRS,
)
main_interaction_statistics_df = interaction_statistics(
    main_pair_interaction_df
)

primary_main_interaction_df = main_pair_interaction_df[
    main_pair_interaction_df["pair"] == "expanded_bilateral"
].copy()
primary_main_statistics_df = main_interaction_statistics_df[
    main_interaction_statistics_df["pair"]
    == "expanded_bilateral"
].copy()
primary_main_statistics_df = primary_main_statistics_df.merge(
    outcome_eligibility_df[
        outcome_eligibility_df["outcome"].isin(
            METRIC_DEFINITIONS
        )
    ][
        [
            "outcome",
            "probe",
            "all_quality_gates_passed",
            "analysis_status",
        ]
    ],
    on=["outcome", "probe"],
    how="left",
    validate="one_to_one",
)
definition_sensitivity_statistics_df = (
    main_interaction_statistics_df[
        main_interaction_statistics_df["pair"]
        != "expanded_bilateral"
    ].copy()
)

a1_frequency_qa_df = (
    main_network_df[
        (main_network_df["network"] == "semantic_expanded")
        & (main_network_df["probe"].isin(PERIODIC_PROBES))
        & (main_network_df["severity"].isin([0.0, 1.0]))
    ][
        [
            "severity",
            "condition",
            "seed",
            "probe",
            "a1_snr_db_min",
            "a1_phase_consistency_min",
        ]
    ]
    .drop_duplicates()
    .sort_values(["probe", "severity", "seed"])
    .reset_index(drop=True)
)
a1_frequency_qa_df["snr_gate_passed"] = (
    a1_frequency_qa_df["a1_snr_db_min"] >= A1_SNR_GATE_DB
)
a1_frequency_qa_df["phase_gate_passed"] = (
    a1_frequency_qa_df["a1_phase_consistency_min"]
    >= A1_PHASE_CONSISTENCY_GATE
)
a1_frequency_qa_df["frequency_qa_passed"] = (
    a1_frequency_qa_df["snr_gate_passed"]
    & a1_frequency_qa_df["phase_gate_passed"]
)
display(a1_frequency_qa_df)

periodic_temporal_qa_df = main_network_df[
    (main_network_df["network"].isin(PRIMARY_INFERENTIAL_NETWORKS))
    & (main_network_df["probe"].isin(PERIODIC_PROBES))
][
    [
        "condition",
        "severity",
        "seed",
        "probe",
        "network",
        "target_response_coverage",
        "transfer_split_abs_log2",
        "transfer_nonstationary_flag",
        "transfer_log2_slope_per_s",
        "locked_transfer_segment_median",
        "median_target_snr_db",
        "median_target_phase_consistency",
        "target_frequency_qa_valid_fraction",
        "evoked_fc_split_abs_z",
    ]
].sort_values(["probe", "severity", "seed", "network"])

display(
    primary_main_interaction_df.sort_values(
        ["outcome", "probe", "seed", "severity"]
    )
)
display(primary_main_statistics_df)
display(definition_sensitivity_statistics_df)
print(
    "The confidence intervals above quantify numerical-initialization "
    "variation, not patient or population uncertainty."
)


## 13. Required integration-step validity and precision assessment

Before the longer robustness workload runs, baseline and high endpoints are
repeated at 0.25 ms for every main numerical initialization in final mode.
The comparison is still performed at the exact grain of the confirmatory
question: the paired 20-seed baseline-to-high
semantic-minus-episodic interaction.

Fatal validity checks stop the notebook only for nonfinite estimates, failed
A1 stimulation quality, interaction-direction reversal, or a changed
two-sided 95% t-interval conclusion about zero. The original precision
targets remain unchanged: 5% transfer ratio, 0.05 Fisher-z FC, 5 ms latency,
and 1 dB A1 SNR. Exceeding one of these precision targets produces an explicit
warning and an exact-magnitude eligibility label rather than discarding a
directionally and inferentially consistent experiment.

The paired 0.5-minus-0.25 ms seed-level bias and its 95% t interval are saved
for every primary outcome. Individual raw metric differences are also retained
as numerical-sensitivity diagnostics.


In [ ]:
dt_check_conditions = [
    {
        "condition": SEVERITY_LABELS[severity],
        "severity": severity,
        "b_values": B_BY_SEVERITY[severity],
        "variant": f"dt_0.25ms_severity_{severity:.1f}",
    }
    for severity in (0.0, 1.0)
]
(
    dt_reference_node_df,
    dt_reference_network_df,
    dt_reference_manifest_df,
) = execute_grid(
    scope="dt_reference_0.25ms",
    conditions=dt_check_conditions,
    seeds=CFG["dt_check_seeds"],
    probes=PROBES,
    global_coupling=MAIN_GLOBAL_COUPLING,
    input_peak_per_ms=MAIN_INPUT_PEAK_PER_MS,
    dt_ms=REFERENCE_DT_MS,
)

dt_primary_pair = {
    "expanded_bilateral": (
        "semantic_expanded",
        "episodic_expanded",
    )
}
dt_main_network_df = main_network_df[
    (main_network_df["seed"].isin(CFG["dt_check_seeds"]))
    & (main_network_df["severity"].isin([0.0, 1.0]))
    & (
        main_network_df["network"].isin(
            PRIMARY_INFERENTIAL_NETWORKS
        )
    )
].copy()
dt_reference_primary_network_df = dt_reference_network_df[
    dt_reference_network_df["network"].isin(
        PRIMARY_INFERENTIAL_NETWORKS
    )
].copy()

dt_main_normalized_df = normalize_to_baseline(
    dt_main_network_df
)
dt_reference_normalized_df = normalize_to_baseline(
    dt_reference_primary_network_df
)
dt_main_interaction_df = make_pair_interactions(
    dt_main_normalized_df,
    dt_primary_pair,
)
dt_reference_interaction_df = make_pair_interactions(
    dt_reference_normalized_df,
    dt_primary_pair,
)

dt_interaction_keys = ["seed", "probe", "outcome", "unit"]
dt_main_endpoint_df = dt_main_interaction_df[
    dt_main_interaction_df["severity"] == 1.0
][
    [
        *dt_interaction_keys,
        "semantic_minus_episodic_interaction",
    ]
].rename(
    columns={
        "semantic_minus_episodic_interaction": (
            "main_interaction"
        )
    }
)
dt_reference_endpoint_df = dt_reference_interaction_df[
    dt_reference_interaction_df["severity"] == 1.0
][
    [
        *dt_interaction_keys,
        "semantic_minus_episodic_interaction",
    ]
].rename(
    columns={
        "semantic_minus_episodic_interaction": (
            "reference_interaction"
        )
    }
)
dt_interaction_seed_diagnostics_df = (
    dt_main_endpoint_df.merge(
        dt_reference_endpoint_df,
        on=dt_interaction_keys,
        how="inner",
        validate="one_to_one",
    )
)
expected_interaction_rows = 5 * len(CFG["dt_check_seeds"])
if (
    len(dt_interaction_seed_diagnostics_df)
    != expected_interaction_rows
):
    raise RuntimeError(
        "Integration-step interaction pairing is incomplete: "
        f"expected {expected_interaction_rows}, found "
        f"{len(dt_interaction_seed_diagnostics_df)}."
    )
dt_interaction_seed_diagnostics_df[
    "signed_main_minus_reference"
] = (
    dt_interaction_seed_diagnostics_df["main_interaction"]
    - dt_interaction_seed_diagnostics_df[
        "reference_interaction"
    ]
)
dt_interaction_seed_diagnostics_df[
    "absolute_main_minus_reference"
] = np.abs(
    dt_interaction_seed_diagnostics_df[
        "signed_main_minus_reference"
    ]
)

dt_tolerance_by_outcome = {
    "transfer_gain": DT_TRANSFER_INTERACTION_LOG2_TOLERANCE,
    "functional_connectivity": DT_FC_ABSOLUTE_TOLERANCE,
    "response_latency": DT_LATENCY_TOLERANCE_MS,
}

def dt_interval_summary(values):
    values = np.asarray(values, dtype=float)
    count = len(values)
    mean_value = float(np.mean(values))
    if count > 1:
        sd_value = float(np.std(values, ddof=1))
        if sd_value > 1e-15:
            half_width = float(
                stats.t.ppf(0.975, count - 1)
                * sd_value
                / np.sqrt(count)
            )
            hedges_correction = (
                1.0 - 3.0 / (4.0 * count - 5.0)
            )
            hedges_gz = float(
                hedges_correction
                * mean_value
                / sd_value
            )
        else:
            half_width = 0.0
            hedges_gz = np.nan
        lower = mean_value - half_width
        upper = mean_value + half_width
        if lower > 0.0:
            inference_class = "positive"
        elif upper < 0.0:
            inference_class = "negative"
        else:
            inference_class = "includes_zero"
    else:
        sd_value = np.nan
        lower = np.nan
        upper = np.nan
        hedges_gz = np.nan
        inference_class = "not_estimable"
    return {
        "mean": mean_value,
        "sd": sd_value,
        "ci95_lower": lower,
        "ci95_upper": upper,
        "hedges_gz": hedges_gz,
        "inference_class": inference_class,
    }

dt_gate_rows = []
for (
    probe,
    outcome,
    unit,
), group in dt_interaction_seed_diagnostics_df.groupby(
    ["probe", "outcome", "unit"],
    sort=True,
):
    seed_count = int(group["seed"].nunique())
    if seed_count != len(CFG["dt_check_seeds"]):
        raise RuntimeError(
            "An integration-step interaction group is missing "
            f"seeds for {probe}/{outcome}: {seed_count}."
        )
    main_summary = dt_interval_summary(
        group["main_interaction"].to_numpy(dtype=float)
    )
    reference_summary = dt_interval_summary(
        group["reference_interaction"].to_numpy(dtype=float)
    )
    step_bias_summary = dt_interval_summary(
        group["signed_main_minus_reference"].to_numpy(dtype=float)
    )
    main_mean = main_summary["mean"]
    reference_mean = reference_summary["mean"]
    signed_difference = main_mean - reference_mean
    absolute_difference = abs(signed_difference)
    tolerance = float(dt_tolerance_by_outcome[outcome])
    direction_consistent = bool(
        np.sign(main_mean) == np.sign(reference_mean)
        or (
            abs(main_mean) <= tolerance
            and abs(reference_mean) <= tolerance
        )
    )
    inference_consistent = bool(
        seed_count < 2
        or (
            main_summary["inference_class"]
            == reference_summary["inference_class"]
        )
    )
    finite = bool(
        np.isfinite(
            [
                main_mean,
                reference_mean,
                signed_difference,
            ]
        ).all()
    )
    classification = classify_integration_step_check(
        finite=finite,
        direction_consistent=direction_consistent,
        inference_consistent=inference_consistent,
        precision_target_passed=(
            finite and absolute_difference <= tolerance
        ),
    )
    dt_gate_rows.append(
        {
            "gate_level": (
                "finite_seed_interaction_mean_and_ci"
            ),
            "severity": 1.0,
            "probe": probe,
            "outcome": outcome,
            "unit": unit,
            "seed_count": seed_count,
            "main_mean": main_mean,
            "reference_mean": reference_mean,
            "signed_difference": signed_difference,
            "absolute_difference": absolute_difference,
            "tolerance": tolerance,
            "direction_consistent": direction_consistent,
            "main_sd": main_summary["sd"],
            "reference_sd": reference_summary["sd"],
            "main_ci95_lower": main_summary["ci95_lower"],
            "main_ci95_upper": main_summary["ci95_upper"],
            "reference_ci95_lower": (
                reference_summary["ci95_lower"]
            ),
            "reference_ci95_upper": (
                reference_summary["ci95_upper"]
            ),
            "main_hedges_gz": main_summary["hedges_gz"],
            "reference_hedges_gz": (
                reference_summary["hedges_gz"]
            ),
            "main_inference_class": (
                main_summary["inference_class"]
            ),
            "reference_inference_class": (
                reference_summary["inference_class"]
            ),
            "inference_consistent": inference_consistent,
            "step_bias_sd": step_bias_summary["sd"],
            "step_bias_ci95_lower": step_bias_summary["ci95_lower"],
            "step_bias_ci95_upper": step_bias_summary["ci95_upper"],
            **classification,
        }
    )

dt_snr_keys = ["severity", "seed", "probe"]
dt_main_snr_df = dt_main_network_df[
    dt_main_network_df["probe"].isin(PERIODIC_PROBES)
][
    [*dt_snr_keys, "a1_snr_db_min"]
].drop_duplicates().rename(
    columns={"a1_snr_db_min": "main_a1_snr_db"}
)
dt_reference_snr_df = dt_reference_primary_network_df[
    dt_reference_primary_network_df["probe"].isin(
        PERIODIC_PROBES
    )
][
    [*dt_snr_keys, "a1_snr_db_min"]
].drop_duplicates().rename(
    columns={
        "a1_snr_db_min": "reference_a1_snr_db"
    }
)
dt_snr_seed_diagnostics_df = dt_main_snr_df.merge(
    dt_reference_snr_df,
    on=dt_snr_keys,
    how="inner",
    validate="one_to_one",
)
expected_snr_rows = (
    2 * len(PERIODIC_PROBES) * len(CFG["dt_check_seeds"])
)
if len(dt_snr_seed_diagnostics_df) != expected_snr_rows:
    raise RuntimeError(
        "Integration-step A1 SNR pairing is incomplete: "
        f"expected {expected_snr_rows}, found "
        f"{len(dt_snr_seed_diagnostics_df)}."
    )
dt_snr_seed_diagnostics_df["absolute_difference_db"] = np.abs(
    dt_snr_seed_diagnostics_df["main_a1_snr_db"]
    - dt_snr_seed_diagnostics_df["reference_a1_snr_db"]
)
for (
    severity,
    probe,
), group in dt_snr_seed_diagnostics_df.groupby(
    ["severity", "probe"],
    sort=True,
):
    maximum_difference = float(
        group["absolute_difference_db"].max()
    )
    snr_step_bias = (
        group["main_a1_snr_db"].to_numpy(dtype=float)
        - group["reference_a1_snr_db"].to_numpy(dtype=float)
    )
    snr_step_bias_summary = dt_interval_summary(snr_step_bias)
    snr_values = group[
        ["main_a1_snr_db", "reference_a1_snr_db"]
    ].to_numpy(dtype=float)
    a1_quality_passed = bool(
        np.isfinite(snr_values).all()
        and float(np.min(snr_values)) >= A1_SNR_GATE_DB
    )
    snr_classification = classify_integration_step_check(
        finite=a1_quality_passed,
        direction_consistent=True,
        inference_consistent=True,
        precision_target_passed=(
            np.isfinite(maximum_difference)
            and maximum_difference <= DT_A1_SNR_TOLERANCE_DB
        ),
    )
    dt_gate_rows.append(
        {
            "gate_level": "a1_seed_maximum",
            "severity": float(severity),
            "probe": probe,
            "outcome": "a1_snr_db",
            "unit": "dB",
            "seed_count": int(group["seed"].nunique()),
            "main_mean": float(
                group["main_a1_snr_db"].mean()
            ),
            "reference_mean": float(
                group["reference_a1_snr_db"].mean()
            ),
            "signed_difference": float(
                group["main_a1_snr_db"].mean()
                - group["reference_a1_snr_db"].mean()
            ),
            "absolute_difference": maximum_difference,
            "tolerance": DT_A1_SNR_TOLERANCE_DB,
            "direction_consistent": True,
            "main_sd": np.nan,
            "reference_sd": np.nan,
            "main_ci95_lower": np.nan,
            "main_ci95_upper": np.nan,
            "reference_ci95_lower": np.nan,
            "reference_ci95_upper": np.nan,
            "main_hedges_gz": np.nan,
            "reference_hedges_gz": np.nan,
            "main_inference_class": "not_applicable",
            "reference_inference_class": "not_applicable",
            "inference_consistent": True,
            "step_bias_sd": snr_step_bias_summary["sd"],
            "step_bias_ci95_lower": (
                snr_step_bias_summary["ci95_lower"]
            ),
            "step_bias_ci95_upper": (
                snr_step_bias_summary["ci95_upper"]
            ),
            **snr_classification,
        }
    )

raw_comparison_keys = [
    "severity", "seed", "probe", "network"
]
raw_comparison_columns = [
    *raw_comparison_keys,
    "transfer",
    "evoked_fc_z",
    "relative_latency_ms",
    "a1_snr_db_min",
]
dt_raw_main_df = dt_main_network_df[
    raw_comparison_columns
].rename(
    columns={
        "transfer": "transfer_main",
        "evoked_fc_z": "fc_main",
        "relative_latency_ms": "latency_main",
        "a1_snr_db_min": "a1_snr_main",
    }
)
dt_raw_reference_df = dt_reference_primary_network_df[
    raw_comparison_columns
].rename(
    columns={
        "transfer": "transfer_reference",
        "evoked_fc_z": "fc_reference",
        "relative_latency_ms": "latency_reference",
        "a1_snr_db_min": "a1_snr_reference",
    }
)
dt_raw_merged_df = dt_raw_main_df.merge(
    dt_raw_reference_df,
    on=raw_comparison_keys,
    how="inner",
    validate="one_to_one",
)
dt_raw_rows = []
for row in dt_raw_merged_df.itertuples(index=False):
    if row.probe in PERIODIC_PROBES:
        raw_metrics = (
            (
                "transfer",
                abs(
                    row.transfer_main
                    - row.transfer_reference
                )
                / max(abs(row.transfer_reference), 1e-15),
                DT_TRANSFER_RELATIVE_TOLERANCE,
            ),
            (
                "functional_connectivity",
                abs(row.fc_main - row.fc_reference),
                DT_FC_ABSOLUTE_TOLERANCE,
            ),
            (
                "a1_snr_db",
                abs(
                    row.a1_snr_main
                    - row.a1_snr_reference
                ),
                DT_A1_SNR_TOLERANCE_DB,
            ),
        )
    else:
        raw_metrics = (
            (
                "response_latency",
                abs(
                    row.latency_main
                    - row.latency_reference
                ),
                DT_LATENCY_TOLERANCE_MS,
            ),
        )
    for metric, difference, tolerance in raw_metrics:
        dt_raw_rows.append(
            {
                **{
                    key: getattr(row, key)
                    for key in raw_comparison_keys
                },
                "metric": metric,
                "difference": float(difference),
                "old_pointwise_tolerance": float(tolerance),
                "within_old_pointwise_tolerance": bool(
                    np.isfinite(difference)
                    and difference <= tolerance
                ),
            }
        )
dt_raw_seed_diagnostics_df = pd.DataFrame(dt_raw_rows).sort_values(
    ["metric", "probe", "severity", "seed", "network"]
)

dt_convergence_df = pd.DataFrame(dt_gate_rows).sort_values(
    ["gate_level", "outcome", "probe", "severity"]
).reset_index(drop=True)
display(dt_convergence_df)
pointwise_departure_count = int(
    (~dt_raw_seed_diagnostics_df[
        "within_old_pointwise_tolerance"
    ]).sum()
)
print(
    "Pointwise seed-level differences outside the old raw gate "
    f"(diagnostic only): {pointwise_departure_count}/"
    f"{len(dt_raw_seed_diagnostics_df)}"
)
fatal_dt_failures = dt_convergence_df[
    ~dt_convergence_df["fatal_validity_passed"]
]
if not fatal_dt_failures.empty:
    display(fatal_dt_failures)
    raise RuntimeError(
        "At least one fatal integration-step validity check failed."
    )

precision_dt_warnings = dt_convergence_df[
    ~dt_convergence_df["precision_target_passed"]
]
if not precision_dt_warnings.empty:
    display(precision_dt_warnings)
    warnings.warn(
        "Integration-step validity passed, but one or more unchanged "
        "exact-magnitude precision targets were exceeded. Directional "
        "inference remains eligible; affected exact magnitudes are "
        "labeled integration-step sensitive.",
        RuntimeWarning,
    )
else:
    print("All integration-step precision targets passed.")
print("All fatal integration-step validity checks passed.")

dt_outcome_eligibility_df = dt_convergence_df[
    dt_convergence_df["gate_level"]
    == "finite_seed_interaction_mean_and_ci"
][
    [
        "outcome",
        "probe",
        "fatal_validity_passed",
        "precision_target_passed",
        "absolute_difference",
        "tolerance",
        "step_bias_ci95_lower",
        "step_bias_ci95_upper",
        "gate_action",
    ]
].copy()
dt_outcome_eligibility_df["integration_step_status"] = np.where(
    dt_outcome_eligibility_df["precision_target_passed"],
    "exact_magnitude_precision_target_passed",
    "direction_robust_exact_magnitude_dt_sensitive",
)

outcome_eligibility_df = outcome_eligibility_df.merge(
    dt_outcome_eligibility_df[
        [
            "outcome",
            "probe",
            "fatal_validity_passed",
            "precision_target_passed",
            "integration_step_status",
        ]
    ],
    on=["outcome", "probe"],
    how="left",
    validate="one_to_one",
)
dt_sensitive_mask = (
    outcome_eligibility_df["analysis_status"].eq(
        "eligible_as_prespecified"
    )
    & outcome_eligibility_df[
        "precision_target_passed"
    ].eq(False)
)
outcome_eligibility_df.loc[
    dt_sensitive_mask,
    "analysis_status",
] = "direction_robust_exact_magnitude_descriptive_only"


## 14. Local-dynamics-held-baseline counterfactual

At the high endpoint, \(b\) is restored to baseline in bilateral A1 and
every parcel in both expanded target sets while the rest of the perturbed
field remains unchanged. The same 20 main seeds are reused. This separates
dependence on target-local parameter changes from effects that remain when
the rest of the modeled network is perturbed. It does not establish
biological causality.


In [ ]:
MEMORY_LOCAL_FIXED_HIGH_B = HIGH_B.copy()
MEMORY_LOCAL_FIXED_HIGH_B[
    MEMORY_COUNTERFACTUAL_FIXED_INDICES
] = BASELINE_B[MEMORY_COUNTERFACTUAL_FIXED_INDICES]

local_fixed_conditions = [
    {
        "condition": (
            "High AD-like perturbation with A1 and expanded target "
            "local dynamics held at baseline"
        ),
        "severity": 1.0,
        "b_values": MEMORY_LOCAL_FIXED_HIGH_B,
        "variant": "expanded_targets_local_fixed",
    }
]
(
    local_fixed_node_df,
    local_fixed_network_df,
    local_fixed_manifest_df,
) = execute_grid(
    scope="local_dynamics_counterfactual",
    conditions=local_fixed_conditions,
    seeds=CFG["seeds"],
    probes=PROBES,
    global_coupling=MAIN_GLOBAL_COUPLING,
    input_peak_per_ms=MAIN_INPUT_PEAK_PER_MS,
)
local_fixed_normalized_df = normalize_to_baseline(
    local_fixed_network_df,
    baseline_df=main_network_df,
)
local_fixed_pair_interaction_df = make_pair_interactions(
    local_fixed_normalized_df,
    NETWORK_PAIRS,
)
primary_local_fixed_interaction_df = (
    local_fixed_pair_interaction_df[
        local_fixed_pair_interaction_df["pair"]
        == "expanded_bilateral"
    ].copy()
)
local_fixed_statistics_df = interaction_statistics(
    primary_local_fixed_interaction_df
)

full_endpoint_for_counterfactual = primary_main_interaction_df[
    primary_main_interaction_df["severity"] == 1.0
][
    [
        "seed",
        "outcome",
        "probe",
        "semantic_minus_episodic_interaction",
    ]
].rename(
    columns={
        "semantic_minus_episodic_interaction":
            "full_field_interaction"
    }
)
local_endpoint_for_counterfactual = (
    primary_local_fixed_interaction_df[
        primary_local_fixed_interaction_df["severity"] == 1.0
    ][
        [
            "seed",
            "outcome",
            "probe",
            "semantic_minus_episodic_interaction",
        ]
    ].rename(
        columns={
            "semantic_minus_episodic_interaction":
                "local_fixed_interaction"
        }
    )
)
counterfactual_comparison_df = (
    full_endpoint_for_counterfactual.merge(
        local_endpoint_for_counterfactual,
        on=["seed", "outcome", "probe"],
        validate="one_to_one",
    )
)
counterfactual_comparison_df[
    "absolute_contrast_attenuation_percent"
] = 100.0 * (
    1.0
    - np.abs(
        counterfactual_comparison_df["local_fixed_interaction"]
    )
    / np.maximum(
        np.abs(
            counterfactual_comparison_df["full_field_interaction"]
        ),
        1e-15,
    )
)
counterfactual_summary_df = (
    counterfactual_comparison_df.groupby(
        ["outcome", "probe"], as_index=False
    )
    .agg(
        median_full_interaction=(
            "full_field_interaction", "median"
        ),
        median_local_fixed_interaction=(
            "local_fixed_interaction", "median"
        ),
        median_attenuation_percent=(
            "absolute_contrast_attenuation_percent", "median"
        ),
        minimum_attenuation_percent=(
            "absolute_contrast_attenuation_percent", "min"
        ),
        maximum_attenuation_percent=(
            "absolute_contrast_attenuation_percent", "max"
        ),
    )
)
display(counterfactual_comparison_df)
display(counterfactual_summary_df)
display(local_fixed_statistics_df)


## 15. Topology-, pathology-, size-, and hemisphere-matched controls

Five hundred paired control sets preserve each target parcel's hemisphere
and approximately match log weighted strength, log direct A1 affinity, and
local high-endpoint \(b\) reduction. The semantic and episodic control sets
preserve the primary group sizes of 13 and 19 and are nonoverlapping within
each draw. These controls reuse saved whole-brain node metrics and require
no additional TVB simulations.

Empirical percentiles and central 90% ranges are descriptive simulation
ranks, not clinical p-values.


In [ ]:
MATCH_WEIGHTS = 0.5 * (WEIGHTS + WEIGHTS.T)
CORTICAL_INDICES = np.arange(360)
weighted_strength = MATCH_WEIGHTS.sum(axis=1)
direct_a1_affinity = MATCH_WEIGHTS[:, A1_INDICES].sum(axis=1)
local_b_reduction = BASELINE_B - HIGH_B

raw_matching_features = np.column_stack(
    [
        np.log10(weighted_strength + 1e-15),
        np.log10(direct_a1_affinity + 1e-15),
        local_b_reduction,
    ]
)
cortical_feature_mean = raw_matching_features[
    CORTICAL_INDICES
].mean(axis=0)
cortical_feature_std = raw_matching_features[
    CORTICAL_INDICES
].std(axis=0)
if np.any(cortical_feature_std <= 0):
    raise RuntimeError("A matching feature has zero variance.")
matching_z = (
    raw_matching_features - cortical_feature_mean
) / cortical_feature_std


def same_hemisphere_candidates(target_index):
    target_index = int(target_index)
    if target_index < 180:
        return np.arange(0, 180)
    if target_index < 360:
        return np.arange(180, 360)
    raise ValueError("Matched targets must be cortical.")


def draw_matched_set(
    target_indices,
    rng,
    reserved,
    top_k=30,
):
    selected = []
    distances_selected = []
    for target_index in target_indices:
        candidates = same_hemisphere_candidates(target_index)
        candidates = np.array(
            [
                index
                for index in candidates
                if index not in reserved
                and index not in selected
            ],
            dtype=int,
        )
        if not len(candidates):
            raise RuntimeError(
                "No eligible hemisphere-matched control parcels remain."
            )
        distances = np.linalg.norm(
            matching_z[candidates] - matching_z[int(target_index)],
            axis=1,
        )
        order = np.argsort(distances)
        pool_order = order[: min(top_k, len(order))]
        pool = candidates[pool_order]
        pool_distances = distances[pool_order]
        scale = max(float(np.median(pool_distances)), 1e-6)
        probabilities = np.exp(
            -(pool_distances - pool_distances.min()) / scale
        )
        probabilities /= probabilities.sum()
        chosen = int(rng.choice(pool, p=probabilities))
        selected.append(chosen)
        distances_selected.append(
            float(
                np.linalg.norm(
                    matching_z[chosen]
                    - matching_z[int(target_index)]
                )
            )
        )
    return np.array(selected, dtype=int), distances_selected


def build_matched_pair_sets():
    rng = np.random.default_rng(20260729)
    excluded = set(ALL_DECLARED_INDICES.tolist())
    semantic_targets = NETWORK_INDICES["semantic_expanded"]
    episodic_targets = NETWORK_INDICES["episodic_expanded"]
    rows = []
    for set_id in range(CFG["matched_null_sets"]):
        if set_id % 2 == 0:
            semantic_control, semantic_distances = draw_matched_set(
                semantic_targets,
                rng,
                reserved=excluded,
            )
            episodic_control, episodic_distances = draw_matched_set(
                episodic_targets,
                rng,
                reserved=excluded.union(
                    semantic_control.tolist()
                ),
            )
            draw_order = "semantic_first"
        else:
            episodic_control, episodic_distances = draw_matched_set(
                episodic_targets,
                rng,
                reserved=excluded,
            )
            semantic_control, semantic_distances = draw_matched_set(
                semantic_targets,
                rng,
                reserved=excluded.union(
                    episodic_control.tolist()
                ),
            )
            draw_order = "episodic_first"

        rows.append(
            {
                "set_id": set_id,
                "draw_order": draw_order,
                "semantic_control_indices": ";".join(
                    map(str, semantic_control)
                ),
                "episodic_control_indices": ";".join(
                    map(str, episodic_control)
                ),
                "semantic_control_labels": ";".join(
                    LABELS[semantic_control]
                ),
                "episodic_control_labels": ";".join(
                    LABELS[episodic_control]
                ),
                "mean_standardized_match_distance": float(
                    np.mean(
                        semantic_distances + episodic_distances
                    )
                ),
            }
        )
    return pd.DataFrame(rows)


matched_sets_df = build_matched_pair_sets()
display(matched_sets_df.head())
display(
    matched_sets_df["mean_standardized_match_distance"].describe()
)


NODE_METRIC_COLUMNS = {
    "transfer_gain": ("node_transfer", "transfer"),
    "functional_connectivity": ("evoked_fc_z", "fc"),
    "response_latency": ("relative_latency_ms", "latency"),
}


def node_metric_array(node_df, seed, probe, severity, column):
    subset = node_df[
        (node_df["seed"] == int(seed))
        & (node_df["probe"] == probe)
        & (node_df["severity"] == float(severity))
    ].sort_values("region_index")
    if len(subset) != N_REGIONS:
        raise RuntimeError(
            "Expected exactly one saved node metric per region."
        )
    values = subset[column].to_numpy(dtype=float)
    return values


def baseline_referenced_network_change(
    baseline_values,
    high_values,
    indices,
    metric_kind,
):
    baseline_network = aggregate_node_metric(
        baseline_values, indices, LABELS, metric_kind
    )
    high_network = aggregate_node_metric(
        high_values, indices, LABELS, metric_kind
    )
    if metric_kind == "transfer":
        return float(
            np.log2(
                max(high_network, 1e-30)
                / max(baseline_network, 1e-30)
            )
        )
    return float(high_network - baseline_network)


matched_metric_combinations = [
    ("transfer_gain", "2Hz"),
    ("transfer_gain", "5Hz"),
    ("functional_connectivity", "2Hz"),
    ("functional_connectivity", "5Hz"),
    ("response_latency", "pulse"),
]
matched_null_rows = []
parsed_matched_sets = [
    (
        int(row.set_id),
        np.fromstring(
            row.semantic_control_indices,
            sep=";",
            dtype=int,
        ),
        np.fromstring(
            row.episodic_control_indices,
            sep=";",
            dtype=int,
        ),
        float(row.mean_standardized_match_distance),
    )
    for row in matched_sets_df.itertuples(index=False)
]

for seed in CFG["seeds"]:
    for outcome, probe in matched_metric_combinations:
        node_column, metric_kind = NODE_METRIC_COLUMNS[outcome]
        baseline_values = node_metric_array(
            main_node_df, seed, probe, 0.0, node_column
        )
        high_values = node_metric_array(
            main_node_df, seed, probe, 1.0, node_column
        )
        for (
            set_id,
            semantic_control,
            episodic_control,
            mean_distance,
        ) in parsed_matched_sets:
            semantic_change = baseline_referenced_network_change(
                baseline_values,
                high_values,
                semantic_control,
                metric_kind,
            )
            episodic_change = baseline_referenced_network_change(
                baseline_values,
                high_values,
                episodic_control,
                metric_kind,
            )
            matched_null_rows.append(
                {
                    "set_id": set_id,
                    "seed": int(seed),
                    "outcome": outcome,
                    "probe": probe,
                    "semantic_control_change": semantic_change,
                    "episodic_control_change": episodic_change,
                    "null_semantic_minus_episodic": (
                        semantic_change - episodic_change
                    ),
                    "mean_standardized_match_distance": (
                        mean_distance
                    ),
                }
            )
matched_null_df = pd.DataFrame(matched_null_rows)

observed_endpoint_df = primary_main_interaction_df[
    primary_main_interaction_df["severity"] == 1.0
][
    [
        "seed",
        "outcome",
        "probe",
        "semantic_minus_episodic_interaction",
    ]
].rename(
    columns={
        "semantic_minus_episodic_interaction":
            "observed_interaction"
    }
)

matched_null_summary_rows = []
for observed in observed_endpoint_df.itertuples(index=False):
    null_values = matched_null_df[
        (matched_null_df["seed"] == observed.seed)
        & (matched_null_df["outcome"] == observed.outcome)
        & (matched_null_df["probe"] == observed.probe)
    ]["null_semantic_minus_episodic"].to_numpy(dtype=float)
    lower = float(np.quantile(null_values, 0.05))
    upper = float(np.quantile(null_values, 0.95))
    percentile = float(
        100.0
        * (
            1
            + np.count_nonzero(
                null_values <= observed.observed_interaction
            )
        )
        / (len(null_values) + 1)
    )
    matched_null_summary_rows.append(
        {
            "seed": int(observed.seed),
            "outcome": observed.outcome,
            "probe": observed.probe,
            "observed_interaction": float(
                observed.observed_interaction
            ),
            "null_median": float(np.median(null_values)),
            "null_5th_percentile": lower,
            "null_95th_percentile": upper,
            "observed_empirical_percentile": percentile,
            "outside_central_90_percent": bool(
                observed.observed_interaction < lower
                or observed.observed_interaction > upper
            ),
            "matched_sets": len(null_values),
        }
    )
matched_null_summary_df = pd.DataFrame(
    matched_null_summary_rows
)
display(matched_null_summary_df)


## 16. Prespecified definition, parameter, laterality, and spatial checks

Definition and laterality checks reuse the main whole-brain outputs.
Parameter checks use the original five-seed subset in final mode. Fifty
spatial shuffles permute high-endpoint \(b\) values separately within left
cortex, right cortex, and subcortex while preserving each block's
distribution. Spatial ranks remain descriptive.


In [ ]:
# Direct right-minus-left interaction sensitivity.
laterality_subset = main_pair_interaction_df[
    main_pair_interaction_df["pair"].isin(
        ["expanded_left_only", "expanded_right_only"]
    )
]
laterality_pivot = laterality_subset.pivot(
    index=[
        "scope",
        "variant",
        "condition",
        "severity",
        "seed",
        "outcome",
        "probe",
        "unit",
    ],
    columns="pair",
    values="semantic_minus_episodic_interaction",
).reset_index()
laterality_pivot["right_minus_left_interaction"] = (
    laterality_pivot["expanded_right_only"]
    - laterality_pivot["expanded_left_only"]
)
laterality_difference_rows = []
for group_key, group in laterality_pivot[
    laterality_pivot["severity"] == 1.0
].groupby(["outcome", "probe", "unit"], sort=False):
    values = group["right_minus_left_interaction"].to_numpy(
        dtype=float
    )
    n = len(values)
    mean_value = float(np.mean(values))
    sd_value = float(np.std(values, ddof=1)) if n > 1 else np.nan
    if n > 1 and sd_value > 1e-15:
        half_width = float(
            stats.t.ppf(0.975, n - 1)
            * sd_value
            / np.sqrt(n)
        )
        hedges_gz = float(
            (1.0 - 3.0 / (4.0 * n - 5.0))
            * mean_value
            / sd_value
        )
    elif n > 1:
        half_width = 0.0
        hedges_gz = np.nan
    else:
        half_width = np.nan
        hedges_gz = np.nan
    laterality_difference_rows.append(
        {
            "outcome": group_key[0],
            "probe": group_key[1],
            "unit": group_key[2],
            "numerical_initializations": n,
            "mean_right_minus_left_interaction": mean_value,
            "ci95_lower_numerical": mean_value - half_width,
            "ci95_upper_numerical": mean_value + half_width,
            "paired_hedges_gz": hedges_gz,
        }
    )
laterality_difference_statistics_df = pd.DataFrame(
    laterality_difference_rows
)

# Parameter sensitivity: each scenario has its own matched baseline.
parameter_conditions = []
for scenario in CFG["sensitivity_scenarios"]:
    for severity in (0.0, 1.0):
        parameter_conditions.append(
            {
                "scope": f"parameter_{scenario['scenario']}",
                "condition": (
                    f"{SEVERITY_LABELS[severity]}, "
                    f"{scenario['scenario']}"
                ),
                "severity": severity,
                "b_values": B_BY_SEVERITY[severity],
                "variant": scenario["scenario"],
                "global_coupling": scenario["global_coupling"],
                "input_peak_per_ms": scenario["input_peak"],
            }
        )
(
    parameter_node_df,
    parameter_network_df,
    parameter_manifest_df,
) = execute_grid(
    scope="parameter_sensitivity",
    conditions=parameter_conditions,
    seeds=CFG["sensitivity_seeds"],
    probes=PERIODIC_PROBES,
    global_coupling=MAIN_GLOBAL_COUPLING,
    input_peak_per_ms=MAIN_INPUT_PEAK_PER_MS,
)
parameter_normalized_df = normalize_to_baseline(
    parameter_network_df
)
parameter_pair_interaction_df = make_pair_interactions(
    parameter_normalized_df,
    {"expanded_bilateral": NETWORK_PAIRS["expanded_bilateral"]},
)
parameter_interaction_statistics_df = interaction_statistics(
    parameter_pair_interaction_df,
    additional_group_columns=(
        "variant",
        "global_coupling",
        "input_peak_per_ms",
    ),
)

# Spatial-placement sensitivity.
shuffle_rng = np.random.default_rng(3792026)
shuffle_blocks = [
    np.arange(0, 180),
    np.arange(180, 360),
    np.arange(360, 379),
]
shuffle_conditions = []
for shuffle_id in range(CFG["spatial_shuffles"]):
    shuffled_b = HIGH_B.copy()
    for block in shuffle_blocks:
        shuffled_b[block] = shuffle_rng.permutation(HIGH_B[block])
    shuffle_label = f"shuffle_{shuffle_id + 1:02d}"
    shuffle_conditions.append(
        {
            "scope": f"spatial_{shuffle_label}",
            "condition": (
                "High AD-like perturbation, spatially shuffled"
            ),
            "severity": 1.0,
            "b_values": shuffled_b,
            "variant": shuffle_label,
        }
    )
(
    shuffle_node_df,
    shuffle_network_df,
    shuffle_manifest_df,
) = execute_grid(
    scope="spatial_shuffles",
    conditions=shuffle_conditions,
    seeds=[CFG["seeds"][0]],
    probes=PERIODIC_PROBES,
    global_coupling=MAIN_GLOBAL_COUPLING,
    input_peak_per_ms=MAIN_INPUT_PEAK_PER_MS,
)
shuffle_normalized_df = normalize_to_baseline(
    shuffle_network_df,
    baseline_df=main_network_df,
)
shuffle_pair_interaction_df = make_pair_interactions(
    shuffle_normalized_df,
    {"expanded_bilateral": NETWORK_PAIRS["expanded_bilateral"]},
)

observed_first_seed = primary_main_interaction_df[
    (primary_main_interaction_df["severity"] == 1.0)
    & (primary_main_interaction_df["seed"] == CFG["seeds"][0])
    & (
        primary_main_interaction_df["outcome"].isin(
            ["transfer_gain", "functional_connectivity"]
        )
    )
][
    [
        "outcome",
        "probe",
        "semantic_minus_episodic_interaction",
    ]
].rename(
    columns={
        "semantic_minus_episodic_interaction":
            "observed_interaction"
    }
)
spatial_shuffle_summary_rows = []
for observed in observed_first_seed.itertuples(index=False):
    shuffle_values = shuffle_pair_interaction_df[
        (shuffle_pair_interaction_df["outcome"] == observed.outcome)
        & (shuffle_pair_interaction_df["probe"] == observed.probe)
    ]["semantic_minus_episodic_interaction"].to_numpy(dtype=float)
    lower = float(np.quantile(shuffle_values, 0.05))
    upper = float(np.quantile(shuffle_values, 0.95))
    percentile = float(
        100.0
        * (
            1
            + np.count_nonzero(
                shuffle_values <= observed.observed_interaction
            )
        )
        / (len(shuffle_values) + 1)
    )
    spatial_shuffle_summary_rows.append(
        {
            "outcome": observed.outcome,
            "probe": observed.probe,
            "observed_interaction": float(
                observed.observed_interaction
            ),
            "shuffle_median": float(np.median(shuffle_values)),
            "shuffle_5th_percentile": lower,
            "shuffle_95th_percentile": upper,
            "observed_empirical_percentile": percentile,
            "outside_central_90_percent": bool(
                observed.observed_interaction < lower
                or observed.observed_interaction > upper
            ),
            "spatial_shuffles": len(shuffle_values),
        }
    )
spatial_shuffle_summary_df = pd.DataFrame(
    spatial_shuffle_summary_rows
)

display(definition_sensitivity_statistics_df)
display(laterality_difference_statistics_df)
display(parameter_interaction_statistics_df)
display(spatial_shuffle_summary_df)


## 17. Secondary prolonged-stimulation stability and entrainment follow-up

This follow-up does not change the primary experiment. The original 10-second
endpoint remains fixed at `[4500, 14500)` ms. The periodic simulations are
repeated to 24.5 seconds and analyzed in two equal, phase-aligned windows:

- original: `[4500, 14500)` ms;
- late: `[14500, 24500)` ms.

The stimulus remains on throughout both windows. This is prolonged stimulation,
not a post-stimulation recovery or waiting-period experiment. Each window has
20 cycles at 2 Hz or 50 cycles at 5 Hz and five complete 2-second segments.

The unchanged zero-input control remains the basis for comparison with the
primary result. A secondary DC-matched control applies the periodic waveform's
exact mean input, `A/2`, with identical onset and offset. In this nonlinear
model, periodic minus DC is a paired counterfactual contrast for the modulation;
it is not an additive decomposition of tonic and oscillatory effects.

For both reference controls and both windows, the notebook saves broadband
evoked-response ratios, exact-frequency amplitude and transfer, harmonic
fit quality, cycle phase consistency, dominant frequency, applied-frequency
power fraction, magnitude-squared A1-target coherence, raw evoked FC,
split-window FC, and all five 2-second segments. Fixed parcel membership is
used for network summaries. Phase summaries based on QA-valid parcels are
stored separately with the valid fraction, never silently substituted.

Stability is assessed continuously and with paired two-one-sided equivalence
tests across numerical initializations. The transfer margin of plus/minus 20%
and the existing plus/minus 0.10 Fisher-z FC margin are operational analysis
thresholds, not biologically validated constants. Failures are nonfatal and
downgrade only the follow-up stability claim. The complete prolonged experiment
is repeated at 0.25 ms; exact-magnitude precision failures remain warnings.


In [ ]:
def followup_window_mask(time_ms, bounds_ms):
    start_ms, end_ms = map(float, bounds_ms)
    time_ms = np.asarray(time_ms, dtype=float)
    mask = (time_ms >= start_ms) & (time_ms < end_ms)
    expected_samples = int(round((end_ms - start_ms) / MONITOR_PERIOD_MS))
    if int(mask.sum()) != expected_samples:
        raise RuntimeError(
            f"Window {bounds_ms} has {int(mask.sum())} samples; "
            f"expected {expected_samples}."
        )
    return mask


def followup_half_windows(time_ms, bounds_ms):
    start_ms, end_ms = map(float, bounds_ms)
    midpoint_ms = 0.5 * (start_ms + end_ms)
    return (
        followup_window_mask(time_ms, (start_ms, midpoint_ms)),
        followup_window_mask(time_ms, (midpoint_ms, end_ms)),
    )


def followup_segment_windows(time_ms, bounds_ms):
    start_ms, end_ms = map(float, bounds_ms)
    expected_duration = PERIODIC_SEGMENT_MS * PERIODIC_SEGMENT_COUNT
    if not np.isclose(end_ms - start_ms, expected_duration):
        raise RuntimeError("A follow-up window does not contain five segments.")
    return tuple(
        followup_window_mask(
            time_ms,
            (
                start_ms + index * PERIODIC_SEGMENT_MS,
                start_ms + (index + 1) * PERIODIC_SEGMENT_MS,
            ),
        )
        for index in range(PERIODIC_SEGMENT_COUNT)
    )


def sample_dc_temporal_waveform(
    time_ms,
    model,
    input_peak_per_ms,
    onset_ms=STIMULUS_ONSET_MS,
    offset_ms=FOLLOWUP_SIMULATION_END_MS,
):
    time_ms = np.asarray(time_ms, dtype=float)
    derivative_peak = float(
        model.A[0] * model.a[0] * float(input_peak_per_ms)
    )
    active = (time_ms >= float(onset_ms)) & (time_ms < float(offset_ms))
    return np.where(active, 0.5 * derivative_peak, 0.0)


def run_tvb_dc_only(
    b_values,
    global_coupling,
    input_peak_per_ms,
    seed,
    weights,
    labels,
    a1_indices,
    dt_ms,
    simulation_ms,
):
    b_values = np.asarray(b_values, dtype=float)
    a1_indices = np.asarray(a1_indices, dtype=int)
    white_matter = fresh_connectivity(weights, labels)
    model = build_model(b_values)
    derivative_peak = float(
        model.A[0] * model.a[0] * float(input_peak_per_ms)
    )
    dc_temporal = equations.TemporalApplicableEquation(
        equation=(
            "where((var >= onset) & (var < offset), 0.5 * amp, 0.0)"
        ),
        parameters={
            "onset": float(STIMULUS_ONSET_MS),
            "offset": float(simulation_ms),
            "amp": derivative_peak,
        },
    )
    regional_weights = np.zeros(N_REGIONS, dtype=float)
    regional_weights[a1_indices] = 1.0 / np.sqrt(len(a1_indices))
    stimulus = patterns.StimuliRegion(
        temporal=dc_temporal,
        connectivity=white_matter,
        weight=regional_weights,
    )
    suppress_known_tvb_deterministic_random_state_warning()
    experiment = simulator.Simulator(
        connectivity=white_matter,
        model=model,
        coupling=coupling.SigmoidalJansenRit(
            a=np.array([float(global_coupling)])
        ),
        integrator=integrators.HeunDeterministic(dt=float(dt_ms)),
        monitors=(monitors.SubSample(period=MONITOR_PERIOD_MS),),
        stimulus=stimulus,
        initial_conditions=make_initial_conditions(seed),
    )
    experiment.configure()
    started = time.perf_counter()
    (time_ms, raw), = experiment.run(simulation_length=float(simulation_ms))
    wall_seconds = time.perf_counter() - started
    psp = np.asarray(raw[:, 0, :, 0] - raw[:, 1, :, 0], dtype=float)
    time_ms = np.asarray(time_ms, dtype=float)
    if psp.shape != (len(time_ms), N_REGIONS):
        raise RuntimeError(f"Unexpected DC-control TVB output shape: {psp.shape}")
    if not np.isfinite(psp).all():
        raise RuntimeError("DC-control TVB output contains nonfinite values.")
    if float(np.max(np.abs(psp))) > 100.0:
        raise RuntimeError("DC-control activity exceeded the safety bound.")
    return time_ms, psp, wall_seconds


def multitaper_transform_in_window(time_ms, values, analysis_window):
    time_ms = np.asarray(time_ms, dtype=float)
    window = np.asarray(analysis_window, dtype=bool)
    y = signal.detrend(
        np.asarray(values[window], dtype=float),
        axis=0,
        type="linear",
    )
    if y.ndim == 1:
        y = y[:, None]
    if y.shape[0] < 100:
        raise RuntimeError("A spectral window is too short.")
    sampling_hz = 1000.0 / float(np.median(np.diff(time_ms[window])))
    tapers = signal.windows.dpss(
        y.shape[0],
        NW=MULTITAPER_TIME_BANDWIDTH,
        Kmax=MULTITAPER_TAPERS,
        sym=False,
        norm=2,
    )
    tapered_spectra = np.stack(
        [
            np.fft.rfft(y * np.asarray(taper)[:, None], axis=0)
            for taper in tapers
        ],
        axis=0,
    )
    frequencies_hz = np.fft.rfftfreq(y.shape[0], d=1.0 / sampling_hz)
    power = np.mean(np.abs(tapered_spectra) ** 2, axis=0)
    return frequencies_hz, power, tapered_spectra


def drive_snr_from_multitaper(frequencies_hz, power, probe):
    frequency_hz = _periodic_frequency_hz(probe)
    signal_mask = (
        np.abs(frequencies_hz - frequency_hz)
        <= SNR_SIGNAL_HALF_WIDTH_HZ
    )
    flank_mask = (
        (
            frequencies_hz >= frequency_hz - SNR_FLANK_OUTER_HZ
        )
        & (
            frequencies_hz <= frequency_hz - SNR_FLANK_INNER_HZ
        )
    ) | (
        (
            frequencies_hz >= frequency_hz + SNR_FLANK_INNER_HZ
        )
        & (
            frequencies_hz <= frequency_hz + SNR_FLANK_OUTER_HZ
        )
    )
    if not signal_mask.any() or not flank_mask.any():
        raise RuntimeError("Follow-up SNR bands are empty.")
    signal_power = np.mean(power[signal_mask], axis=0)
    noise_power = np.mean(power[flank_mask], axis=0)
    return 10.0 * np.log10(
        np.maximum(signal_power, 1e-30)
        / np.maximum(noise_power, 1e-30)
    )


def cycle_phase_consistency_in_window(
    time_ms,
    evoked,
    probe,
    analysis_window,
):
    frequency_hz = _periodic_frequency_hz(probe)
    window = np.asarray(analysis_window, dtype=bool)
    y = np.asarray(evoked[window], dtype=float)
    sampling_hz = 1000.0 / float(np.median(np.diff(time_ms[window])))
    samples_per_cycle_float = sampling_hz / frequency_hz
    samples_per_cycle = int(round(samples_per_cycle_float))
    if not np.isclose(samples_per_cycle, samples_per_cycle_float, atol=1e-10):
        raise RuntimeError("Drive cycles do not align with monitor samples.")
    cycle_count = y.shape[0] // samples_per_cycle
    if cycle_count < 10:
        raise RuntimeError("Too few complete cycles for phase consistency.")
    trimmed = y[: cycle_count * samples_per_cycle]
    cycles = trimmed.reshape(
        cycle_count,
        samples_per_cycle,
        trimmed.shape[1],
    )
    cycles = signal.detrend(cycles, axis=1, type="linear")
    cycle_time = np.arange(samples_per_cycle) / sampling_hz
    basis = np.exp(-2j * np.pi * frequency_hz * cycle_time)
    coefficients = np.einsum("csr,s->cr", cycles, basis)
    magnitudes = np.abs(coefficients)
    unit_phase = coefficients / np.maximum(magnitudes, 1e-30)
    consistency = np.abs(np.mean(unit_phase, axis=0))
    consistency[np.max(magnitudes, axis=0) <= 1e-20] = np.nan
    return consistency


def ipsilateral_multitaper_coherence(
    tapered_spectra,
    frequencies_hz,
    probe,
    labels,
):
    frequency_hz = _periodic_frequency_hz(probe)
    drive_mask = (
        np.abs(frequencies_hz - frequency_hz)
        <= SNR_SIGNAL_HALF_WIDTH_HZ
    )
    if not drive_mask.any():
        raise RuntimeError("No multitaper bins cover the drive frequency.")
    result = np.full(N_REGIONS, np.nan, dtype=float)
    for hemisphere, a1_index in A1_INDEX_BY_HEMISPHERE.items():
        target_indices = np.array(
            [
                index
                for index, label in enumerate(labels)
                if label_hemisphere(label) == hemisphere
            ],
            dtype=int,
        )
        reference = tapered_spectra[:, :, a1_index]
        targets = tapered_spectra[:, :, target_indices]
        pxx = np.mean(np.abs(reference) ** 2, axis=0)
        pyy = np.mean(np.abs(targets) ** 2, axis=0)
        pxy = np.mean(np.conj(reference)[:, :, None] * targets, axis=0)
        coherence = (
            np.abs(pxy) ** 2
            / np.maximum(pxx[:, None] * pyy, 1e-30)
        )
        result[target_indices] = np.mean(coherence[drive_mask], axis=0)
    midline_indices = np.array(
        [
            index
            for index, label in enumerate(labels)
            if label_hemisphere(label) == "M"
        ],
        dtype=int,
    )
    if len(midline_indices):
        reference = np.mean(tapered_spectra[:, :, A1_INDICES], axis=2)
        targets = tapered_spectra[:, :, midline_indices]
        pxx = np.mean(np.abs(reference) ** 2, axis=0)
        pyy = np.mean(np.abs(targets) ** 2, axis=0)
        pxy = np.mean(np.conj(reference)[:, :, None] * targets, axis=0)
        coherence = (
            np.abs(pxy) ** 2
            / np.maximum(pxx[:, None] * pyy, 1e-30)
        )
        result[midline_indices] = np.mean(coherence[drive_mask], axis=0)
    return np.clip(result, 0.0, 1.0)


def spectral_followup_features(
    time_ms,
    evoked,
    probe,
    analysis_window,
    labels,
):
    frequencies_hz, power, tapered_spectra = multitaper_transform_in_window(
        time_ms,
        evoked,
        analysis_window,
    )
    frequency_hz = _periodic_frequency_hz(probe)
    drive_mask = (
        np.abs(frequencies_hz - frequency_hz)
        <= SNR_SIGNAL_HALF_WIDTH_HZ
    )
    positive_mask = frequencies_hz > 0.0
    bounded_mask = (
        (frequencies_hz >= FOLLOWUP_DOMINANT_RANGE_HZ[0])
        & (frequencies_hz <= FOLLOWUP_DOMINANT_RANGE_HZ[1])
    )
    if not positive_mask.any() or not bounded_mask.any():
        raise RuntimeError("Dominant-frequency search band is empty.")
    full_indices = np.flatnonzero(positive_mask)[
        np.argmax(power[positive_mask], axis=0)
    ]
    bounded_indices = np.flatnonzero(bounded_mask)[
        np.argmax(power[bounded_mask], axis=0)
    ]
    drive_power = np.mean(power[drive_mask], axis=0)
    bounded_peak_power = power[
        bounded_indices,
        np.arange(power.shape[1]),
    ]
    return {
        "frequencies_hz": frequencies_hz,
        "power": power,
        "tapered_spectra": tapered_spectra,
        "snr_db": drive_snr_from_multitaper(
            frequencies_hz,
            power,
            probe,
        ),
        "dominant_frequency_nonzero_full_hz": frequencies_hz[full_indices],
        "dominant_frequency_0p5_40_hz": frequencies_hz[bounded_indices],
        "applied_frequency_power_to_dominant_ratio": (
            drive_power / np.maximum(bounded_peak_power, 1e-30)
        ),
        "a1_target_magnitude_squared_coherence": (
            ipsilateral_multitaper_coherence(
                tapered_spectra,
                frequencies_hz,
                probe,
                labels,
            )
        ),
    }


def circular_mean_rad(values):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if not len(values):
        return np.nan
    return float(np.angle(np.mean(np.exp(1j * values))))


def compute_followup_periodic_metrics(
    time_ms,
    stimulated_psp,
    reference_psp,
    probe,
    labels,
    bounds_ms,
):
    evoked = np.asarray(stimulated_psp, dtype=float) - np.asarray(
        reference_psp,
        dtype=float,
    )
    full_window = followup_window_mask(time_ms, bounds_ms)
    first_half_window, second_half_window = followup_half_windows(
        time_ms,
        bounds_ms,
    )
    segment_windows = followup_segment_windows(time_ms, bounds_ms)

    response = detrended_ac_rms(evoked, full_window)
    first_half_response = detrended_ac_rms(evoked, first_half_window)
    second_half_response = detrended_ac_rms(evoked, second_half_window)
    segment_responses = np.stack(
        [detrended_ac_rms(evoked, mask) for mask in segment_windows],
        axis=0,
    )
    full_fit = exact_frequency_fit(
        time_ms,
        evoked,
        probe,
        analysis_window=full_window,
    )
    segment_fits = [
        exact_frequency_fit(
            time_ms,
            evoked,
            probe,
            analysis_window=mask,
        )
        for mask in segment_windows
    ]
    locked_response = full_fit["amplitude"]
    segment_locked_responses = np.stack(
        [fit["amplitude"] for fit in segment_fits],
        axis=0,
    )
    phase_lag_rad = phase_lag_to_ipsilateral_a1(
        full_fit["phase_rad"],
        labels,
    )
    phase_consistency = cycle_phase_consistency_in_window(
        time_ms,
        evoked,
        probe,
        full_window,
    )
    spectral = spectral_followup_features(
        time_ms,
        evoked,
        probe,
        full_window,
        labels,
    )
    evoked_fc_z = evoked_a1_target_fc_z(
        time_ms,
        evoked,
        labels,
        analysis_window=full_window,
    )
    evoked_fc_first_half_z = evoked_a1_target_fc_z(
        time_ms,
        evoked,
        labels,
        analysis_window=first_half_window,
    )
    evoked_fc_second_half_z = evoked_a1_target_fc_z(
        time_ms,
        evoked,
        labels,
        analysis_window=second_half_window,
    )
    segment_evoked_fc_z = np.stack(
        [
            evoked_a1_target_fc_z(
                time_ms,
                evoked,
                labels,
                analysis_window=mask,
            )
            for mask in segment_windows
        ],
        axis=0,
    )

    response_reference = ipsilateral_reference_values(response, labels)
    first_reference = ipsilateral_reference_values(first_half_response, labels)
    second_reference = ipsilateral_reference_values(second_half_response, labels)
    locked_reference = ipsilateral_reference_values(locked_response, labels)
    if np.nanmin(response_reference[A1_INDICES]) <= 1e-10:
        raise RuntimeError("Follow-up A1 response is too small to normalize.")
    node_transfer = response / np.maximum(response_reference, 1e-30)
    first_half_node_transfer = (
        first_half_response / np.maximum(first_reference, 1e-30)
    )
    second_half_node_transfer = (
        second_half_response / np.maximum(second_reference, 1e-30)
    )
    full_window_locked_node_transfer = (
        locked_response / np.maximum(locked_reference, 1e-30)
    )
    segment_node_transfer = np.stack(
        [
            values
            / np.maximum(
                ipsilateral_reference_values(values, labels),
                1e-30,
            )
            for values in segment_responses
        ],
        axis=0,
    )
    segment_locked_node_transfer = np.stack(
        [
            values
            / np.maximum(
                ipsilateral_reference_values(values, labels),
                1e-30,
            )
            for values in segment_locked_responses
        ],
        axis=0,
    )
    locked_node_transfer = np.median(
        segment_locked_node_transfer,
        axis=0,
    )
    response_valid = (
        np.isfinite(node_transfer)
        & (node_transfer >= TARGET_RESPONSE_RATIO_FLOOR)
    )
    frequency_qa_valid = (
        np.isfinite(spectral["snr_db"])
        & np.isfinite(phase_consistency)
        & (spectral["snr_db"] >= MIN_TARGET_FREQUENCY_SNR_DB)
        & (phase_consistency >= MIN_TARGET_PHASE_CONSISTENCY)
    )
    fc_valid = (
        response_valid
        & np.isfinite(evoked_fc_z)
        & np.isfinite(evoked_fc_first_half_z)
        & np.isfinite(evoked_fc_second_half_z)
    )
    return {
        "response": response,
        "node_transfer": node_transfer,
        "first_half_node_transfer": first_half_node_transfer,
        "second_half_node_transfer": second_half_node_transfer,
        "segment_node_transfer": segment_node_transfer,
        "locked_response": locked_response,
        "ipsilateral_a1_locked_response": locked_reference,
        "full_window_locked_node_transfer": full_window_locked_node_transfer,
        "locked_node_transfer": locked_node_transfer,
        "segment_locked_node_transfer": segment_locked_node_transfer,
        "fit_r_squared": full_fit["r_squared"],
        "sin_coefficient": full_fit["sin_coefficient"],
        "cos_coefficient": full_fit["cos_coefficient"],
        "phase_rad": full_fit["phase_rad"],
        "phase_lag_to_ipsilateral_a1_rad": phase_lag_rad,
        "phase_lag_degrees": np.degrees(phase_lag_rad),
        "phase_consistency": phase_consistency,
        "snr_db": spectral["snr_db"],
        "dominant_frequency_nonzero_full_hz": spectral[
            "dominant_frequency_nonzero_full_hz"
        ],
        "dominant_frequency_0p5_40_hz": spectral[
            "dominant_frequency_0p5_40_hz"
        ],
        "applied_frequency_power_to_dominant_ratio": spectral[
            "applied_frequency_power_to_dominant_ratio"
        ],
        "a1_target_magnitude_squared_coherence": spectral[
            "a1_target_magnitude_squared_coherence"
        ],
        "evoked_fc_z": evoked_fc_z,
        "evoked_fc_first_half_z": evoked_fc_first_half_z,
        "evoked_fc_second_half_z": evoked_fc_second_half_z,
        "segment_evoked_fc_z": segment_evoked_fc_z,
        "response_valid": response_valid,
        "frequency_qa_valid": frequency_qa_valid,
        "fc_valid": fc_valid,
        "max_abs_evoked": float(np.max(np.abs(evoked))),
    }


def _followup_trace_filename(severity, seed, dt_ms):
    severity_token = f"{float(severity):.1f}".replace(".", "p")
    dt_token = f"{float(dt_ms):.2f}".rstrip("0").rstrip(".").replace(".", "p")
    return (
        f"severity_{severity_token}_seed_{int(seed):04d}_"
        f"dt_{dt_token}ms.npz"
    )


def write_followup_trace_archive(
    *,
    scope,
    condition_name,
    severity,
    seed,
    dt_ms,
    global_coupling,
    input_peak_per_ms,
    b_values,
    time_ms,
    zero_control_psp,
    dc_control_psp,
    stimulated_by_probe,
    labels,
    a1_indices,
):
    time_ms = np.asarray(time_ms, dtype=float)
    selected = {
        "zero_input_control_psp": np.ascontiguousarray(
            zero_control_psp[:, TRACE_REGION_INDICES]
        ),
        "dc_matched_control_psp": np.ascontiguousarray(
            dc_control_psp[:, TRACE_REGION_INDICES]
        ),
    }
    for probe in FOLLOWUP_PERIODIC_PROBES:
        token = probe.lower()
        stimulated = np.ascontiguousarray(
            stimulated_by_probe[probe][:, TRACE_REGION_INDICES]
        )
        selected[f"stimulated_{token}_psp"] = stimulated
        selected[f"total_evoked_{token}_psp"] = (
            stimulated - selected["zero_input_control_psp"]
        )
        selected[f"modulation_evoked_{token}_psp"] = (
            stimulated - selected["dc_matched_control_psp"]
        )

    model = build_model(b_values)
    a1_indices = np.asarray(a1_indices, dtype=int)
    a1_spatial_weights = np.full(
        len(a1_indices),
        1.0 / np.sqrt(len(a1_indices)),
        dtype=float,
    )
    dc_temporal = sample_dc_temporal_waveform(
        time_ms,
        model,
        input_peak_per_ms,
    )
    archive_arrays = {
        **selected,
        "time_ms": time_ms,
        "region_labels": np.asarray(TRACE_REGION_LABELS),
        "region_indices": TRACE_REGION_INDICES,
        "semantic_expanded_membership": np.isin(
            TRACE_REGION_INDICES,
            NETWORK_INDICES[PRIMARY_SEMANTIC_NETWORK],
        ),
        "episodic_expanded_membership": np.isin(
            TRACE_REGION_INDICES,
            NETWORK_INDICES[PRIMARY_EPISODIC_NETWORK],
        ),
        "a1_membership": np.isin(TRACE_REGION_INDICES, A1_INDICES),
        "scope": np.asarray(str(scope)),
        "condition": np.asarray(str(condition_name)),
        "severity": np.asarray(float(severity)),
        "seed": np.asarray(int(seed)),
        "integration_step_ms": np.asarray(float(dt_ms)),
        "monitor_period_ms": np.asarray(float(MONITOR_PERIOD_MS)),
        "global_coupling": np.asarray(float(global_coupling)),
        "input_amplitude_per_ms": np.asarray(float(input_peak_per_ms)),
        "stimulus_region_labels": np.asarray(labels)[a1_indices],
        "stimulus_region_indices": a1_indices,
        "stimulus_spatial_weights": a1_spatial_weights,
        "stimulus_waveform_units": np.asarray("model_derivative_input"),
        "stimulus_onset_ms": np.asarray(float(STIMULUS_ONSET_MS)),
        "followup_windows_ms": np.asarray(
            [FOLLOWUP_WINDOWS_MS[name] for name in FOLLOWUP_WINDOWS_MS],
            dtype=float,
        ),
        "followup_window_labels": np.asarray(list(FOLLOWUP_WINDOWS_MS)),
        "trace_format_version": np.asarray(FOLLOWUP_TRACE_FORMAT_VERSION),
        "dc_stimulus_temporal_waveform": dc_temporal,
        "dc_stimulus_waveform": (
            dc_temporal[:, None] * a1_spatial_weights[None, :]
        ),
    }
    for probe in FOLLOWUP_PERIODIC_PROBES:
        token = probe.lower()
        temporal = sample_stimulus_temporal_waveform(
            time_ms,
            probe,
            model,
            input_peak_per_ms,
            offset_ms=FOLLOWUP_SIMULATION_END_MS,
        )
        archive_arrays[f"stimulus_{token}_temporal_waveform"] = temporal
        archive_arrays[f"stimulus_{token}_waveform"] = (
            temporal[:, None] * a1_spatial_weights[None, :]
        )

    spectral_signals = dict(selected)
    for window_label, bounds_ms in FOLLOWUP_WINDOWS_MS.items():
        window = followup_window_mask(time_ms, bounds_ms)
        frequency_reference = None
        for signal_name, values in spectral_signals.items():
            frequencies_hz, power, _ = multitaper_transform_in_window(
                time_ms,
                values,
                window,
            )
            if frequency_reference is None:
                frequency_reference = frequencies_hz
            elif not np.array_equal(frequency_reference, frequencies_hz):
                raise RuntimeError("Follow-up spectral axes differ.")
            archive_arrays[
                f"{window_label}_{signal_name}_multitaper_power"
            ] = power
        archive_arrays[
            f"{window_label}_spectrum_frequency_hz"
        ] = frequency_reference

    filename = _followup_trace_filename(severity, seed, dt_ms)
    output_path = FOLLOWUP_TRACE_DIR / filename
    temporary_path = output_path.with_name(
        f".{output_path.name}.{os.getpid()}.partial"
    )
    with temporary_path.open("wb") as handle:
        np.savez_compressed(handle, **archive_arrays)
        handle.flush()
        os.fsync(handle.fileno())
    sha256 = sha256_file(temporary_path)
    os.replace(temporary_path, output_path)
    return {
        "scope": str(scope),
        "condition": str(condition_name),
        "severity": float(severity),
        "seed": int(seed),
        "dt_ms": float(dt_ms),
        "trace_format_version": FOLLOWUP_TRACE_FORMAT_VERSION,
        "trace_file": str(
            Path(FOLLOWUP_TRACE_ARCHIVE_SUBDIRECTORY) / filename
        ),
        "trace_sha256": sha256,
        "trace_byte_size": int(output_path.stat().st_size),
        "trace_sample_count": int(len(time_ms)),
        "trace_region_count": int(len(TRACE_REGION_INDICES)),
        "contains_complete_positive_frequency_spectra": True,
    }


def append_followup_metric_rows(
    *,
    node_rows,
    network_rows,
    segment_rows,
    common,
    window_label,
    bounds_ms,
    reference_type,
    metrics,
    labels,
    network_index_items,
):
    start_ms, end_ms = map(float, bounds_ms)
    for region_index in range(N_REGIONS):
        node_rows.append(
            {
                **common,
                "analysis_window": window_label,
                "window_start_ms": start_ms,
                "window_end_ms": end_ms,
                "reference_type": reference_type,
                "region_index": region_index,
                "region_label": labels[region_index],
                "hemisphere": label_hemisphere(labels[region_index]),
                "response": float(metrics["response"][region_index]),
                "node_transfer": float(metrics["node_transfer"][region_index]),
                "transfer_first_half": float(
                    metrics["first_half_node_transfer"][region_index]
                ),
                "transfer_second_half": float(
                    metrics["second_half_node_transfer"][region_index]
                ),
                "locked_response": float(
                    metrics["locked_response"][region_index]
                ),
                "ipsilateral_a1_locked_response": float(
                    metrics["ipsilateral_a1_locked_response"][region_index]
                ),
                "full_window_locked_node_transfer": float(
                    metrics["full_window_locked_node_transfer"][region_index]
                ),
                "locked_node_transfer_segment_median": float(
                    metrics["locked_node_transfer"][region_index]
                ),
                **{
                    f"segment_transfer_{index + 1}": float(
                        metrics["segment_node_transfer"][index, region_index]
                    )
                    for index in range(PERIODIC_SEGMENT_COUNT)
                },
                **{
                    f"segment_locked_transfer_{index + 1}": float(
                        metrics["segment_locked_node_transfer"][
                            index,
                            region_index,
                        ]
                    )
                    for index in range(PERIODIC_SEGMENT_COUNT)
                },
                **{
                    f"segment_evoked_fc_z_{index + 1}": float(
                        metrics["segment_evoked_fc_z"][index, region_index]
                    )
                    for index in range(PERIODIC_SEGMENT_COUNT)
                },
                "fit_r_squared": float(
                    metrics["fit_r_squared"][region_index]
                ),
                "sin_coefficient": float(
                    metrics["sin_coefficient"][region_index]
                ),
                "cos_coefficient": float(
                    metrics["cos_coefficient"][region_index]
                ),
                "phase_rad": float(metrics["phase_rad"][region_index]),
                "phase_lag_to_ipsilateral_a1_rad": float(
                    metrics["phase_lag_to_ipsilateral_a1_rad"][region_index]
                ),
                "phase_lag_degrees": float(
                    metrics["phase_lag_degrees"][region_index]
                ),
                "phase_consistency": float(
                    metrics["phase_consistency"][region_index]
                ),
                "snr_db": float(metrics["snr_db"][region_index]),
                "dominant_frequency_nonzero_full_hz": float(
                    metrics["dominant_frequency_nonzero_full_hz"][region_index]
                ),
                "dominant_frequency_0p5_40_hz": float(
                    metrics["dominant_frequency_0p5_40_hz"][region_index]
                ),
                "applied_frequency_power_to_dominant_ratio": float(
                    metrics[
                        "applied_frequency_power_to_dominant_ratio"
                    ][region_index]
                ),
                "a1_target_magnitude_squared_coherence": float(
                    metrics[
                        "a1_target_magnitude_squared_coherence"
                    ][region_index]
                ),
                "evoked_fc_z": float(metrics["evoked_fc_z"][region_index]),
                "evoked_fc_first_half_z": float(
                    metrics["evoked_fc_first_half_z"][region_index]
                ),
                "evoked_fc_second_half_z": float(
                    metrics["evoked_fc_second_half_z"][region_index]
                ),
                "evoked_fc_second_minus_first_z": float(
                    metrics["evoked_fc_second_half_z"][region_index]
                    - metrics["evoked_fc_first_half_z"][region_index]
                ),
                "response_valid": bool(
                    metrics["response_valid"][region_index]
                ),
                "frequency_qa_valid": bool(
                    metrics["frequency_qa_valid"][region_index]
                ),
                "fc_valid": bool(metrics["fc_valid"][region_index]),
            }
        )

    segment_offset = 0 if window_label == "original" else PERIODIC_SEGMENT_COUNT
    for network_name, raw_indices in network_index_items:
        indices = np.asarray(raw_indices, dtype=int)
        transfer = aggregate_node_metric(
            metrics["node_transfer"], indices, labels, "transfer"
        )
        transfer_first = aggregate_node_metric(
            metrics["first_half_node_transfer"], indices, labels, "transfer"
        )
        transfer_second = aggregate_node_metric(
            metrics["second_half_node_transfer"], indices, labels, "transfer"
        )
        transfer_second_minus_first_log2 = float(
            np.log2(max(transfer_second, 1e-30) / max(transfer_first, 1e-30))
        )
        segment_transfer = np.array(
            [
                aggregate_node_metric(values, indices, labels, "transfer")
                for values in metrics["segment_node_transfer"]
            ],
            dtype=float,
        )
        segment_locked = np.array(
            [
                aggregate_node_metric(values, indices, labels, "transfer")
                for values in metrics["segment_locked_node_transfer"]
            ],
            dtype=float,
        )
        segment_fc = np.array(
            [
                aggregate_node_metric(values, indices, labels, "fc")
                for values in metrics["segment_evoked_fc_z"]
            ],
            dtype=float,
        )
        full_locked_transfer = aggregate_node_metric(
            metrics["full_window_locked_node_transfer"],
            indices,
            labels,
            "transfer",
        )
        evoked_fc = aggregate_node_metric(
            metrics["evoked_fc_z"], indices, labels, "fc"
        )
        evoked_fc_first = aggregate_node_metric(
            metrics["evoked_fc_first_half_z"], indices, labels, "fc"
        )
        evoked_fc_second = aggregate_node_metric(
            metrics["evoked_fc_second_half_z"], indices, labels, "fc"
        )
        qa_mask = metrics["frequency_qa_valid"][indices]
        phase_values = metrics["phase_lag_to_ipsilateral_a1_rad"][indices]
        midpoint_seconds = (
            np.arange(PERIODIC_SEGMENT_COUNT, dtype=float) + 0.5
        ) * PERIODIC_SEGMENT_MS / 1000.0
        network_rows.append(
            {
                **common,
                "analysis_window": window_label,
                "window_start_ms": start_ms,
                "window_end_ms": end_ms,
                "reference_type": reference_type,
                "network": network_name,
                "network_response": float(
                    aggregate_node_metric(
                        metrics["response"], indices, labels, "response"
                    )
                ),
                "transfer": float(transfer),
                "transfer_first_half": float(transfer_first),
                "transfer_second_half": float(transfer_second),
                "transfer_second_minus_first_log2": (
                    transfer_second_minus_first_log2
                ),
                "transfer_split_abs_log2": abs(
                    transfer_second_minus_first_log2
                ),
                "transfer_log2_slope_per_s": float(
                    stats.linregress(
                        midpoint_seconds,
                        np.log2(np.maximum(segment_transfer, 1e-30)),
                    ).slope
                ),
                "full_window_locked_transfer": float(full_locked_transfer),
                "locked_transfer_segment_median": float(
                    np.median(segment_locked)
                ),
                **{
                    f"segment_transfer_{index + 1}": float(
                        segment_transfer[index]
                    )
                    for index in range(PERIODIC_SEGMENT_COUNT)
                },
                **{
                    f"segment_locked_transfer_{index + 1}": float(
                        segment_locked[index]
                    )
                    for index in range(PERIODIC_SEGMENT_COUNT)
                },
                **{
                    f"segment_evoked_fc_z_{index + 1}": float(segment_fc[index])
                    for index in range(PERIODIC_SEGMENT_COUNT)
                },
                "evoked_fc_z": float(evoked_fc),
                "evoked_fc_first_half_z": float(evoked_fc_first),
                "evoked_fc_second_half_z": float(evoked_fc_second),
                "relative_latency_ms": np.nan,
                "evoked_fc_second_minus_first_z": float(
                    evoked_fc_second - evoked_fc_first
                ),
                "evoked_fc_split_abs_z": float(
                    abs(evoked_fc_second - evoked_fc_first)
                ),
                "target_response_coverage": float(
                    np.mean(metrics["response_valid"][indices])
                ),
                "fc_valid_fraction": float(
                    np.mean(metrics["fc_valid"][indices])
                ),
                "target_frequency_qa_valid_fraction": float(np.mean(qa_mask)),
                "median_target_fit_r_squared": float(
                    np.nanmedian(metrics["fit_r_squared"][indices])
                ),
                "median_target_snr_db": float(
                    np.nanmedian(metrics["snr_db"][indices])
                ),
                "median_target_phase_consistency": float(
                    np.nanmedian(metrics["phase_consistency"][indices])
                ),
                "mean_target_harmonic_amplitude": float(
                    aggregate_node_metric(
                        metrics["locked_response"],
                        indices,
                        labels,
                        "response",
                    )
                ),
                "mean_ipsilateral_a1_harmonic_amplitude": float(
                    aggregate_node_metric(
                        metrics["ipsilateral_a1_locked_response"],
                        indices,
                        labels,
                        "response",
                    )
                ),
                "median_dominant_frequency_nonzero_full_hz": float(
                    np.nanmedian(
                        metrics["dominant_frequency_nonzero_full_hz"][indices]
                    )
                ),
                "median_dominant_frequency_0p5_40_hz": float(
                    np.nanmedian(
                        metrics["dominant_frequency_0p5_40_hz"][indices]
                    )
                ),
                "median_applied_frequency_power_to_dominant_ratio": float(
                    np.nanmedian(
                        metrics[
                            "applied_frequency_power_to_dominant_ratio"
                        ][indices]
                    )
                ),
                "mean_a1_target_magnitude_squared_coherence": float(
                    np.nanmean(
                        metrics[
                            "a1_target_magnitude_squared_coherence"
                        ][indices]
                    )
                ),
                "circular_mean_phase_lag_all_fixed_parcels_rad": (
                    circular_mean_rad(phase_values)
                ),
                "circular_mean_phase_lag_qa_valid_parcels_rad": (
                    circular_mean_rad(phase_values[qa_mask])
                ),
                "a1_snr_db_min": float(
                    np.nanmin(metrics["snr_db"][A1_INDICES])
                ),
                "a1_phase_consistency_min": float(
                    np.nanmin(metrics["phase_consistency"][A1_INDICES])
                ),
            }
        )
        for segment_index in range(PERIODIC_SEGMENT_COUNT):
            segment_start_ms = start_ms + segment_index * PERIODIC_SEGMENT_MS
            segment_rows.append(
                {
                    **common,
                    "analysis_window": window_label,
                    "reference_type": reference_type,
                    "network": network_name,
                    "segment_in_window": segment_index + 1,
                    "segment_overall": segment_offset + segment_index + 1,
                    "segment_start_ms": segment_start_ms,
                    "segment_end_ms": segment_start_ms + PERIODIC_SEGMENT_MS,
                    "segment_midpoint_after_analysis_start_s": (
                        (segment_start_ms + 0.5 * PERIODIC_SEGMENT_MS
                         - ORIGINAL_WINDOW_MS[0]) / 1000.0
                    ),
                    "segment_transfer": float(segment_transfer[segment_index]),
                    "segment_locked_transfer": float(
                        segment_locked[segment_index]
                    ),
                    "segment_evoked_fc_z": float(segment_fc[segment_index]),
                }
            )


def execute_late_followup_block(
    job,
    weights,
    labels,
    a1_indices,
    network_index_items,
):
    ordinal = int(job["ordinal"])
    scope = str(job["scope"])
    condition_name = str(job["condition"])
    severity = float(job["severity"])
    seed = int(job["seed"])
    b_values = np.asarray(job["b_values"], dtype=float)
    dt_ms = float(job["dt_ms"])
    global_coupling = float(job["global_coupling"])
    input_peak_per_ms = float(job["input_peak_per_ms"])
    worker_pid = int(os.getpid())

    time_ms, zero_psp, zero_wall = run_tvb(
        b_values=b_values,
        probe=None,
        global_coupling=global_coupling,
        input_peak_per_ms=input_peak_per_ms,
        seed=seed,
        weights=weights,
        labels=labels,
        a1_indices=a1_indices,
        dt_ms=dt_ms,
        simulation_ms=FOLLOWUP_SIMULATION_END_MS,
    )
    dc_time, dc_psp, dc_wall = run_tvb_dc_only(
        b_values=b_values,
        global_coupling=global_coupling,
        input_peak_per_ms=input_peak_per_ms,
        seed=seed,
        weights=weights,
        labels=labels,
        a1_indices=a1_indices,
        dt_ms=dt_ms,
        simulation_ms=FOLLOWUP_SIMULATION_END_MS,
    )
    if not np.array_equal(time_ms, dc_time):
        raise RuntimeError("Zero-input and DC-control time axes differ.")

    stimulated_by_probe = {}
    wall_by_probe = {}
    for probe in FOLLOWUP_PERIODIC_PROBES:
        probe_time, probe_psp, wall_seconds = run_tvb(
            b_values=b_values,
            probe=probe,
            global_coupling=global_coupling,
            input_peak_per_ms=input_peak_per_ms,
            seed=seed,
            weights=weights,
            labels=labels,
            a1_indices=a1_indices,
            dt_ms=dt_ms,
            simulation_ms=FOLLOWUP_SIMULATION_END_MS,
        )
        if not np.array_equal(time_ms, probe_time):
            raise RuntimeError(f"{probe} and control time axes differ.")
        stimulated_by_probe[probe] = probe_psp
        wall_by_probe[probe] = float(wall_seconds)

    trace_manifest = write_followup_trace_archive(
        scope=scope,
        condition_name=condition_name,
        severity=severity,
        seed=seed,
        dt_ms=dt_ms,
        global_coupling=global_coupling,
        input_peak_per_ms=input_peak_per_ms,
        b_values=b_values,
        time_ms=time_ms,
        zero_control_psp=zero_psp,
        dc_control_psp=dc_psp,
        stimulated_by_probe=stimulated_by_probe,
        labels=labels,
        a1_indices=a1_indices,
    )

    node_rows = []
    network_rows = []
    segment_rows = []
    common_base = {
        "scope": scope,
        "variant": "prolonged_periodic_followup",
        "condition": condition_name,
        "severity": severity,
        "seed": seed,
        "global_coupling": global_coupling,
        "input_peak_per_ms": input_peak_per_ms,
        "dt_ms": dt_ms,
        "b_signature": b_signature(b_values),
    }
    reference_psp = {
        "zero_input": zero_psp,
        "dc_matched": dc_psp,
    }
    for probe in FOLLOWUP_PERIODIC_PROBES:
        for reference_type, control_psp in reference_psp.items():
            for window_label, bounds_ms in FOLLOWUP_WINDOWS_MS.items():
                metrics = compute_followup_periodic_metrics(
                    time_ms,
                    stimulated_by_probe[probe],
                    control_psp,
                    probe,
                    labels,
                    bounds_ms,
                )
                append_followup_metric_rows(
                    node_rows=node_rows,
                    network_rows=network_rows,
                    segment_rows=segment_rows,
                    common={**common_base, "probe": probe},
                    window_label=window_label,
                    bounds_ms=bounds_ms,
                    reference_type=reference_type,
                    metrics=metrics,
                    labels=labels,
                    network_index_items=network_index_items,
                )

    manifest_rows = [
        {
            **common_base,
            "probe": "none",
            "simulation_type": "matched_zero_input_control",
            "simulation_ms": FOLLOWUP_SIMULATION_END_MS,
            "wall_seconds": float(zero_wall),
            "max_abs_psp": float(np.max(np.abs(zero_psp))),
            "max_abs_evoked": np.nan,
            "job_ordinal": ordinal,
            "worker_pid": worker_pid,
        },
        {
            **common_base,
            "probe": "DC",
            "simulation_type": "matched_dc_control",
            "simulation_ms": FOLLOWUP_SIMULATION_END_MS,
            "wall_seconds": float(dc_wall),
            "max_abs_psp": float(np.max(np.abs(dc_psp))),
            "max_abs_evoked": float(np.max(np.abs(dc_psp - zero_psp))),
            "job_ordinal": ordinal,
            "worker_pid": worker_pid,
        },
    ]
    for probe in FOLLOWUP_PERIODIC_PROBES:
        manifest_rows.append(
            {
                **common_base,
                "probe": probe,
                "simulation_type": "prolonged_periodic_stimulated",
                "simulation_ms": FOLLOWUP_SIMULATION_END_MS,
                "wall_seconds": wall_by_probe[probe],
                "max_abs_psp": float(
                    np.max(np.abs(stimulated_by_probe[probe]))
                ),
                "max_abs_evoked": float(
                    np.max(np.abs(stimulated_by_probe[probe] - zero_psp))
                ),
                "job_ordinal": ordinal,
                "worker_pid": worker_pid,
            }
        )
    del zero_psp, dc_psp, stimulated_by_probe
    gc.collect()
    return {
        "ordinal": ordinal,
        "node_rows": node_rows,
        "network_rows": network_rows,
        "segment_rows": segment_rows,
        "manifest_rows": manifest_rows,
        "trace_manifest": trace_manifest,
    }


def execute_late_followup_grid(scope, dt_ms):
    jobs = []
    for severity in CFG["severities"]:
        for seed in CFG["seeds"]:
            jobs.append(
                {
                    "ordinal": len(jobs),
                    "scope": scope,
                    "condition": SEVERITY_LABELS[severity],
                    "severity": severity,
                    "seed": int(seed),
                    "b_values": B_BY_SEVERITY[severity],
                    "dt_ms": float(dt_ms),
                    "global_coupling": MAIN_GLOBAL_COUPLING,
                    "input_peak_per_ms": MAIN_INPUT_PEAK_PER_MS,
                }
            )
    network_index_items = tuple(
        (name, np.asarray(indices, dtype=int))
        for name, indices in NETWORK_INDICES.items()
    )
    outcomes = run_parallel_jobs(
        execute_late_followup_block,
        jobs,
        (WEIGHTS, LABELS, A1_INDICES, network_index_items),
        f"{scope} {FOLLOWUP_ANALYSIS_VERSION} condition-seed blocks",
    )
    node_rows = []
    network_rows = []
    segment_rows = []
    manifest_rows = []
    trace_rows = []
    for outcome in sorted(outcomes, key=lambda item: item["ordinal"]):
        node_rows.extend(outcome["node_rows"])
        network_rows.extend(outcome["network_rows"])
        segment_rows.extend(outcome["segment_rows"])
        manifest_rows.extend(outcome["manifest_rows"])
        trace_rows.append(outcome["trace_manifest"])

    # A restricted process host can terminate and replace a worker after the
    # atomic archive write begins. Incomplete dot-prefixed files are never
    # scientific outputs and must not enter the result archive.
    for partial_path in FOLLOWUP_TRACE_DIR.glob(".*.partial"):
        partial_path.unlink(missing_ok=True)
    if list(FOLLOWUP_TRACE_DIR.glob(".*.partial")):
        raise RuntimeError("A partial follow-up trace archive remains.")

    job_count = len(jobs)
    factor = (
        len(FOLLOWUP_PERIODIC_PROBES)
        * len(FOLLOWUP_REFERENCE_TYPES)
        * len(FOLLOWUP_WINDOWS_MS)
    )
    expected_node_rows = job_count * factor * N_REGIONS
    expected_network_rows = job_count * factor * len(NETWORK_INDICES)
    expected_segment_rows = (
        expected_network_rows * PERIODIC_SEGMENT_COUNT
    )
    if len(node_rows) != expected_node_rows:
        raise RuntimeError("Follow-up node grid is incomplete.")
    if len(network_rows) != expected_network_rows:
        raise RuntimeError("Follow-up network grid is incomplete.")
    if len(segment_rows) != expected_segment_rows:
        raise RuntimeError("Follow-up segment grid is incomplete.")
    if len(manifest_rows) != job_count * 4:
        raise RuntimeError("Follow-up call manifest is incomplete.")
    if len(trace_rows) != job_count:
        raise RuntimeError("Follow-up trace manifest is incomplete.")
    return (
        pd.DataFrame(node_rows),
        pd.DataFrame(network_rows),
        pd.DataFrame(segment_rows),
        pd.DataFrame(manifest_rows),
        pd.DataFrame(trace_rows),
    )


def normalize_followup_network_metrics(network_df):
    parts = []
    for (reference_type, window_label), group in network_df.groupby(
        ["reference_type", "analysis_window"],
        sort=False,
    ):
        normalized = normalize_to_baseline(group)
        normalized["reference_type"] = reference_type
        normalized["analysis_window"] = window_label
        parts.append(normalized)
    return pd.concat(parts, ignore_index=True)


def make_followup_pair_interactions(normalized_df):
    parts = []
    for (reference_type, window_label), group in normalized_df.groupby(
        ["reference_type", "analysis_window"],
        sort=False,
    ):
        interactions = make_pair_interactions(group, NETWORK_PAIRS)
        interactions["reference_type"] = reference_type
        interactions["analysis_window"] = window_label
        parts.append(interactions)
    return pd.concat(parts, ignore_index=True)


def build_followup_science_validity(network_df, stage):
    rows = []
    required = network_df[
        network_df["network"].isin(PRIMARY_INFERENTIAL_NETWORKS)
    ]
    for row in required.itertuples(index=False):
        a1_passed = bool(
            row.a1_snr_db_min >= A1_SNR_GATE_DB
            and row.a1_phase_consistency_min >= A1_PHASE_CONSISTENCY_GATE
        )
        coverage_passed = bool(
            row.target_response_coverage >= MIN_TARGET_RESPONSE_COVERAGE
        )
        transfer_valid = bool(a1_passed and coverage_passed)
        fc_valid = bool(
            transfer_valid
            and row.fc_valid_fraction >= MIN_TARGET_RESPONSE_COVERAGE
            and row.evoked_fc_split_abs_z <= MAX_FC_SPLIT_ABS_Z
        )
        frequency_valid = bool(
            transfer_valid
            and row.target_frequency_qa_valid_fraction
            >= MIN_TARGET_RESPONSE_COVERAGE
        )
        rows.append(
            {
                "stage": stage,
                "reference_type": row.reference_type,
                "analysis_window": row.analysis_window,
                "severity": float(row.severity),
                "condition": row.condition,
                "seed": int(row.seed),
                "probe": row.probe,
                "network": row.network,
                "a1_frequency_passed": a1_passed,
                "response_coverage_passed": coverage_passed,
                "transfer_valid": transfer_valid,
                "functional_connectivity_valid": fc_valid,
                "frequency_locked_transfer_valid": frequency_valid,
                "target_response_coverage": float(
                    row.target_response_coverage
                ),
                "fc_valid_fraction": float(row.fc_valid_fraction),
                "target_frequency_qa_valid_fraction": float(
                    row.target_frequency_qa_valid_fraction
                ),
                "transfer_split_abs_log2": float(
                    row.transfer_split_abs_log2
                ),
                "evoked_fc_split_abs_z": float(row.evoked_fc_split_abs_z),
                "required_metric_gate_passed": bool(
                    transfer_valid and fc_valid
                ),
            }
        )
    return pd.DataFrame(rows)


def summarize_followup_eligibility(validity_df):
    rows = []
    definitions = [
        ("broadband_transfer", "transfer_valid"),
        ("functional_connectivity", "functional_connectivity_valid"),
        ("frequency_locked_transfer", "frequency_locked_transfer_valid"),
    ]
    for keys, group in validity_df.groupby(
        ["reference_type", "analysis_window", "probe"],
        sort=False,
    ):
        reference_type, window_label, probe = keys
        for outcome, column in definitions:
            passed = group[column].astype(bool)
            all_passed = bool(passed.all())
            rows.append(
                {
                    "reference_type": reference_type,
                    "analysis_window": window_label,
                    "probe": probe,
                    "outcome": outcome,
                    "valid_rows": int(passed.sum()),
                    "required_rows": int(len(passed)),
                    "all_quality_gates_passed": all_passed,
                    "analysis_status": (
                        "eligible_within_secondary_followup"
                        if all_passed
                        else "descriptive_only_quality_gate_failed"
                    ),
                }
            )
    return pd.DataFrame(rows)


def paired_interval(values, confidence=0.95):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    n = len(values)
    if not n:
        return {"n": 0, "mean": np.nan, "sd": np.nan,
                "lower": np.nan, "upper": np.nan}
    mean = float(np.mean(values))
    sd = float(np.std(values, ddof=1)) if n > 1 else np.nan
    if n > 1 and sd > 0.0:
        critical = float(stats.t.ppf(0.5 + confidence / 2.0, n - 1))
        half_width = critical * sd / np.sqrt(n)
    elif n > 1:
        half_width = 0.0
    else:
        half_width = np.nan
    return {
        "n": n,
        "mean": mean,
        "sd": sd,
        "lower": mean - half_width if np.isfinite(half_width) else np.nan,
        "upper": mean + half_width if np.isfinite(half_width) else np.nan,
    }


def paired_tost(values, margin, alpha=FOLLOWUP_EQUIVALENCE_ALPHA):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    summary = paired_interval(values, confidence=1.0 - 2.0 * alpha)
    n = summary["n"]
    mean = summary["mean"]
    sd = summary["sd"]
    if n < 2:
        return {
            **summary,
            "margin": float(margin),
            "tost_p_lower": np.nan,
            "tost_p_upper": np.nan,
            "equivalence_supported": False,
            "status": "not_assessed_fewer_than_two_initializations",
        }
    if sd <= 1e-15:
        p_lower = 0.0 if mean > -margin else 1.0
        p_upper = 0.0 if mean < margin else 1.0
    else:
        se = sd / np.sqrt(n)
        t_lower = (mean + margin) / se
        t_upper = (mean - margin) / se
        p_lower = float(stats.t.sf(t_lower, n - 1))
        p_upper = float(stats.t.cdf(t_upper, n - 1))
    equivalent = bool(p_lower < alpha and p_upper < alpha)
    return {
        **summary,
        "margin": float(margin),
        "tost_p_lower": p_lower,
        "tost_p_upper": p_upper,
        "equivalence_supported": equivalent,
        "status": (
            "operational_equivalence_supported"
            if equivalent
            else "operational_equivalence_not_supported"
        ),
    }


def build_followup_stability_table(network_df):
    rows = []
    for keys, group in network_df.groupby(
        [
            "reference_type",
            "analysis_window",
            "severity",
            "probe",
            "network",
        ],
        sort=False,
    ):
        reference_type, window_label, severity, probe, network = keys
        transfer = paired_tost(
            group["transfer_second_minus_first_log2"],
            FOLLOWUP_TRANSFER_EQUIVALENCE_MARGIN_LOG2,
        )
        fc = paired_tost(
            group["evoked_fc_second_minus_first_z"],
            FOLLOWUP_FC_EQUIVALENCE_MARGIN_Z,
        )
        for outcome, unit, result in [
            ("broadband_transfer", "log2 second/first", transfer),
            ("functional_connectivity", "Fisher-z second-first", fc),
        ]:
            rows.append(
                {
                    "reference_type": reference_type,
                    "analysis_window": window_label,
                    "severity": float(severity),
                    "probe": probe,
                    "network": network,
                    "outcome": outcome,
                    "unit": unit,
                    "numerical_initializations": result["n"],
                    "mean_within_window_drift": result["mean"],
                    "sd_within_window_drift": result["sd"],
                    "ci90_lower_numerical": result["lower"],
                    "ci90_upper_numerical": result["upper"],
                    "equivalence_margin": result["margin"],
                    "tost_p_lower": result["tost_p_lower"],
                    "tost_p_upper": result["tost_p_upper"],
                    "equivalence_supported": result[
                        "equivalence_supported"
                    ],
                    "status": result["status"],
                    "margin_basis": (
                        "operational_followup_not_biologically_validated"
                    ),
                }
            )
    return pd.DataFrame(rows)


def build_ten_segment_slopes(segment_df):
    rows = []
    group_columns = [
        "scope", "condition", "severity", "seed", "probe",
        "reference_type", "network", "dt_ms",
    ]
    for keys, group in segment_df.groupby(group_columns, sort=False):
        group = group.sort_values("segment_overall")
        if list(group["segment_overall"]) != list(range(1, 11)):
            raise RuntimeError("A ten-segment trajectory is incomplete.")
        x = group["segment_midpoint_after_analysis_start_s"].to_numpy(float)
        transfer_values = group["segment_transfer"].to_numpy(float)
        locked_values = group["segment_locked_transfer"].to_numpy(float)
        fc_values = group["segment_evoked_fc_z"].to_numpy(float)
        rows.append(
            {
                **dict(zip(group_columns, keys)),
                "ten_segment_transfer_log2_slope_per_s": float(
                    stats.linregress(
                        x,
                        np.log2(np.maximum(transfer_values, 1e-30)),
                    ).slope
                ),
                "ten_segment_locked_transfer_log2_slope_per_s": float(
                    stats.linregress(
                        x,
                        np.log2(np.maximum(locked_values, 1e-30)),
                    ).slope
                ),
                "ten_segment_fc_z_slope_per_s": float(
                    stats.linregress(x, fc_values).slope
                ),
            }
        )
    return pd.DataFrame(rows)


def compare_original_and_late(network_df):
    key_columns = [
        "scope", "condition", "severity", "seed", "probe",
        "reference_type", "network", "dt_ms",
    ]
    columns = [
        *key_columns,
        "analysis_window",
        "transfer",
        "full_window_locked_transfer",
        "locked_transfer_segment_median",
        "evoked_fc_z",
        "median_target_fit_r_squared",
        "median_target_phase_consistency",
        "median_dominant_frequency_0p5_40_hz",
        "median_applied_frequency_power_to_dominant_ratio",
        "mean_a1_target_magnitude_squared_coherence",
    ]
    original = network_df[
        network_df["analysis_window"] == "original"
    ][columns].drop(columns="analysis_window")
    late = network_df[
        network_df["analysis_window"] == "late"
    ][columns].drop(columns="analysis_window")
    merged = original.merge(
        late,
        on=key_columns,
        suffixes=("_original", "_late"),
        validate="one_to_one",
    )
    merged["transfer_late_vs_original_log2"] = np.log2(
        np.maximum(merged["transfer_late"], 1e-30)
        / np.maximum(merged["transfer_original"], 1e-30)
    )
    merged["locked_transfer_late_vs_original_log2"] = np.log2(
        np.maximum(merged["locked_transfer_segment_median_late"], 1e-30)
        / np.maximum(merged["locked_transfer_segment_median_original"], 1e-30)
    )
    merged["evoked_fc_late_minus_original_z"] = (
        merged["evoked_fc_z_late"] - merged["evoked_fc_z_original"]
    )
    return merged


def reconcile_followup_prefixes(main_trace_manifest, followup_trace_manifest):
    rows = []
    main_periodic = main_trace_manifest[
        main_trace_manifest["probe"].isin(FOLLOWUP_PERIODIC_PROBES)
    ]
    for followup in followup_trace_manifest.itertuples(index=False):
        followup_path = RESULTS_DIR / followup.trace_file
        with np.load(followup_path, allow_pickle=False) as late_archive:
            late_time = late_archive["time_ms"]
            late_prefix = late_time < PERIODIC_ANALYSIS_END_MS
            for probe in FOLLOWUP_PERIODIC_PROBES:
                match = main_periodic[
                    (main_periodic["severity"] == followup.severity)
                    & (main_periodic["seed"] == followup.seed)
                    & (main_periodic["probe"] == probe)
                ]
                if len(match) != 1:
                    raise RuntimeError("Main trace reconciliation key is not unique.")
                main_path = RESULTS_DIR / match.iloc[0]["trace_file"]
                with np.load(main_path, allow_pickle=False) as main_archive:
                    main_time = main_archive["time_ms"]
                    main_prefix = main_time < PERIODIC_ANALYSIS_END_MS
                    time_equal = np.array_equal(
                        main_time[main_prefix],
                        late_time[late_prefix],
                    )
                    token = probe.lower()
                    stimulated_main = main_archive["stimulated_psp"][main_prefix]
                    control_main = main_archive["control_psp"][main_prefix]
                    stimulated_late = late_archive[
                        f"stimulated_{token}_psp"
                    ][late_prefix]
                    control_late = late_archive[
                        "zero_input_control_psp"
                    ][late_prefix]
                    stimulated_equal = np.array_equal(
                        stimulated_main,
                        stimulated_late,
                    )
                    control_equal = np.array_equal(control_main, control_late)
                    rows.append(
                        {
                            "severity": float(followup.severity),
                            "seed": int(followup.seed),
                            "probe": probe,
                            "time_prefix_equal": time_equal,
                            "stimulated_prefix_equal": stimulated_equal,
                            "control_prefix_equal": control_equal,
                            "max_abs_stimulated_prefix_difference": float(
                                np.max(np.abs(stimulated_main - stimulated_late))
                            ),
                            "max_abs_control_prefix_difference": float(
                                np.max(np.abs(control_main - control_late))
                            ),
                            "passed": bool(
                                time_equal and stimulated_equal and control_equal
                            ),
                        }
                    )
    result = pd.DataFrame(rows)
    if not result["passed"].all():
        display(result[~result["passed"]])
        raise RuntimeError(
            "The prolonged run does not reproduce the original pre-14.5 s "
            "TVB trajectories exactly."
        )
    return result


def reconcile_original_window_metrics(main_network, followup_network):
    metric_columns = [
        "transfer", "transfer_first_half", "transfer_second_half",
        "locked_transfer_segment_median", "evoked_fc_z",
        "evoked_fc_first_half_z", "evoked_fc_second_half_z",
        "median_target_fit_r_squared", "median_target_snr_db",
        "median_target_phase_consistency",
        *[f"segment_transfer_{index + 1}" for index in range(5)],
        *[f"segment_locked_transfer_{index + 1}" for index in range(5)],
    ]
    keys = ["severity", "seed", "probe", "network"]
    left = main_network[
        main_network["probe"].isin(FOLLOWUP_PERIODIC_PROBES)
    ][keys + metric_columns]
    right = followup_network[
        (followup_network["reference_type"] == "zero_input")
        & (followup_network["analysis_window"] == "original")
    ][keys + metric_columns]
    merged = left.merge(
        right,
        on=keys,
        suffixes=("_main", "_followup"),
        validate="one_to_one",
    )
    rows = []
    for metric in metric_columns:
        difference = np.abs(
            merged[f"{metric}_main"] - merged[f"{metric}_followup"]
        )
        rows.append(
            {
                "metric": metric,
                "row_count": int(len(difference)),
                "max_absolute_difference": float(np.nanmax(difference)),
                "passed": bool(np.nanmax(difference) <= 1e-10),
            }
        )
    result = pd.DataFrame(rows)
    if not result["passed"].all():
        display(result[~result["passed"]])
        raise RuntimeError(
            "Original-window metrics changed in the prolonged-run prefix."
        )
    return result


def build_followup_dt_check(main_interactions, reference_interactions):
    keys = [
        "reference_type", "analysis_window", "pair", "outcome",
        "probe", "unit", "severity", "seed",
    ]
    main_endpoint = main_interactions[
        main_interactions["severity"] == 1.0
    ][keys + ["semantic_minus_episodic_interaction"]].rename(
        columns={"semantic_minus_episodic_interaction": "main_interaction"}
    )
    reference_endpoint = reference_interactions[
        reference_interactions["severity"] == 1.0
    ][keys + ["semantic_minus_episodic_interaction"]].rename(
        columns={"semantic_minus_episodic_interaction": "reference_interaction"}
    )
    merged = main_endpoint.merge(
        reference_endpoint,
        on=keys,
        validate="one_to_one",
    )
    rows = []
    for group_key, group in merged.groupby(
        ["reference_type", "analysis_window", "pair", "outcome", "probe", "unit"],
        sort=False,
    ):
        reference_type, window_label, pair, outcome, probe, unit = group_key
        main_values = group["main_interaction"].to_numpy(float)
        reference_values = group["reference_interaction"].to_numpy(float)
        differences = main_values - reference_values
        main_summary = dt_interval_summary(main_values)
        reference_summary = dt_interval_summary(reference_values)
        paired_summary = paired_interval(differences, confidence=0.95)
        tolerance = (
            DT_TRANSFER_INTERACTION_LOG2_TOLERANCE
            if outcome == "transfer_gain"
            else DT_FC_ABSOLUTE_TOLERANCE
        )
        main_mean = main_summary["mean"]
        reference_mean = reference_summary["mean"]
        direction_consistent = bool(
            np.sign(main_mean) == np.sign(reference_mean)
            or (
                abs(main_mean) <= tolerance
                and abs(reference_mean) <= tolerance
            )
        )
        inference_consistent = bool(
            main_summary["inference_class"]
            == reference_summary["inference_class"]
        )
        classification = classify_integration_step_check(
            finite=bool(
                np.isfinite(main_values).all()
                and np.isfinite(reference_values).all()
            ),
            direction_consistent=direction_consistent,
            inference_consistent=inference_consistent,
            precision_target_passed=bool(
                abs(main_mean - reference_mean) <= tolerance
            ),
        )
        rows.append(
            {
                "reference_type": reference_type,
                "analysis_window": window_label,
                "pair": pair,
                "outcome": outcome,
                "probe": probe,
                "unit": unit,
                "seed_count": int(len(group)),
                "main_mean": main_mean,
                "reference_mean": reference_mean,
                "signed_bias_main_minus_reference": float(
                    main_mean - reference_mean
                ),
                "paired_bias_ci95_lower": paired_summary["lower"],
                "paired_bias_ci95_upper": paired_summary["upper"],
                "tolerance": tolerance,
                "direction_consistent": direction_consistent,
                "inference_consistent": inference_consistent,
                "main_inference_class": main_summary["inference_class"],
                "reference_inference_class": reference_summary[
                    "inference_class"
                ],
                **classification,
            }
        )
    return pd.DataFrame(rows)


(
    late_followup_node_df,
    late_followup_network_df,
    late_followup_segment_df,
    late_followup_manifest_df,
    late_followup_trace_manifest_df,
) = execute_late_followup_grid(FOLLOWUP_SCOPE, MAIN_DT_MS)

(
    late_followup_dt_reference_node_df,
    late_followup_dt_reference_network_df,
    late_followup_dt_reference_segment_df,
    late_followup_dt_reference_manifest_df,
    late_followup_dt_reference_trace_manifest_df,
) = execute_late_followup_grid(FOLLOWUP_REFERENCE_SCOPE, REFERENCE_DT_MS)

late_followup_normalized_df = normalize_followup_network_metrics(
    late_followup_network_df
)
late_followup_pair_interaction_df = make_followup_pair_interactions(
    late_followup_normalized_df
)
late_followup_interaction_statistics_df = interaction_statistics(
    late_followup_pair_interaction_df,
    additional_group_columns=("reference_type", "analysis_window"),
)
late_followup_science_validity_df = build_followup_science_validity(
    late_followup_network_df,
    stage="prolonged_periodic_followup",
)
late_followup_outcome_eligibility_df = summarize_followup_eligibility(
    late_followup_science_validity_df
)
late_followup_stability_df = build_followup_stability_table(
    late_followup_network_df
)
late_followup_ten_segment_slopes_df = build_ten_segment_slopes(
    late_followup_segment_df
)
late_followup_original_vs_late_df = compare_original_and_late(
    late_followup_network_df
)

late_followup_dt_reference_normalized_df = normalize_followup_network_metrics(
    late_followup_dt_reference_network_df
)
late_followup_dt_reference_pair_interaction_df = (
    make_followup_pair_interactions(
        late_followup_dt_reference_normalized_df
    )
)
late_followup_dt_check_df = build_followup_dt_check(
    late_followup_pair_interaction_df,
    late_followup_dt_reference_pair_interaction_df,
)
late_followup_prefix_reconciliation_df = reconcile_followup_prefixes(
    main_trace_manifest_df,
    late_followup_trace_manifest_df,
)
late_followup_metric_reconciliation_df = reconcile_original_window_metrics(
    main_network_df,
    late_followup_network_df,
)

if not late_followup_dt_check_df["fatal_validity_passed"].all():
    warnings.warn(
        "The secondary prolonged-stimulation follow-up changes direction or "
        "inference classification at 0.25 ms. The primary result remains "
        "unchanged, but the affected follow-up outcome is descriptive only.",
        RuntimeWarning,
    )
if not late_followup_dt_check_df["precision_target_passed"].all():
    warnings.warn(
        "At least one prolonged-stimulation follow-up magnitude exceeds the "
        "unchanged integration-step precision target. Direction and paired "
        "bias intervals are preserved in late_followup_dt_check.csv.",
        RuntimeWarning,
    )

display(late_followup_outcome_eligibility_df)
display(
    late_followup_stability_df[
        (late_followup_stability_df["analysis_window"] == "late")
        & (late_followup_stability_df["severity"] == 1.0)
        & late_followup_stability_df["network"].isin(
            PRIMARY_INFERENTIAL_NETWORKS
        )
    ]
)
display(
    late_followup_dt_check_df[
        late_followup_dt_check_df["pair"] == "expanded_bilateral"
    ]
)
print(
    "Prolonged-run prefix and original-window metric reconciliation passed."
)


In [ ]:
# Prolonged-stimulation trajectory figures use only executed TVB outputs.
FOLLOWUP_NETWORK_COLORS = {
    "semantic_expanded": "#6B46C1",
    "episodic_expanded": "#0F766E",
}
high_segments = late_followup_segment_df[
    (late_followup_segment_df["severity"] == 1.0)
    & (late_followup_segment_df["reference_type"] == "zero_input")
    & late_followup_segment_df["network"].isin(
        PRIMARY_INFERENTIAL_NETWORKS
    )
].copy()

fig, axes = plt.subplots(2, 2, figsize=(13, 8), sharex=True)
for row_index, probe in enumerate(FOLLOWUP_PERIODIC_PROBES):
    for column_index, (metric, ylabel) in enumerate(
        [
            ("segment_transfer", "Broadband A1-normalized ratio"),
            ("segment_locked_transfer", "Exact-frequency A1-normalized ratio"),
        ]
    ):
        axis = axes[row_index, column_index]
        for network in PRIMARY_INFERENTIAL_NETWORKS:
            subset = high_segments[
                (high_segments["probe"] == probe)
                & (high_segments["network"] == network)
            ]
            summary = subset.groupby(
                "segment_midpoint_after_analysis_start_s"
            )[metric].agg(["mean", "std", "count"]).reset_index()
            x = summary["segment_midpoint_after_analysis_start_s"].to_numpy()
            mean = summary["mean"].to_numpy()
            axis.plot(
                x,
                mean,
                marker="o",
                linewidth=2,
                color=FOLLOWUP_NETWORK_COLORS[network],
                label=network.replace("_", " "),
            )
            if (summary["count"] > 1).all():
                critical = stats.t.ppf(0.975, summary["count"] - 1)
                half = critical * summary["std"] / np.sqrt(summary["count"])
                axis.fill_between(
                    x,
                    mean - half,
                    mean + half,
                    color=FOLLOWUP_NETWORK_COLORS[network],
                    alpha=0.16,
                )
        axis.axvline(10.0, color="black", linestyle=":", linewidth=1.2)
        axis.set_title(f"{probe}: {ylabel}")
        axis.set_ylabel(ylabel)
        axis.set_xlabel("Seconds after original analysis start")
        axis.legend(fontsize=8)
fig.suptitle(
    "High-perturbation prolonged response across ten prespecified segments\n"
    "Zero-input reference; shading is 95% numerical-initialization CI",
    y=1.02,
)
fig.tight_layout()
fig.savefig(
    FIGURE_DIR / "late_followup_ten_segment_transfer.png",
    dpi=180,
    bbox_inches="tight",
)
plt.show()

raw_fc_plot = late_followup_network_df[
    (late_followup_network_df["reference_type"] == "zero_input")
    & late_followup_network_df["network"].isin(
        PRIMARY_INFERENTIAL_NETWORKS
    )
]
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)
for axis, probe in zip(axes, FOLLOWUP_PERIODIC_PROBES):
    subset = raw_fc_plot[raw_fc_plot["probe"] == probe]
    for window_label, linestyle in [("original", "-"), ("late", "--")]:
        for network in PRIMARY_INFERENTIAL_NETWORKS:
            group = subset[
                (subset["analysis_window"] == window_label)
                & (subset["network"] == network)
            ].groupby("severity")["evoked_fc_z"].mean()
            axis.plot(
                group.index,
                group.values,
                marker="o",
                linestyle=linestyle,
                color=FOLLOWUP_NETWORK_COLORS[network],
                label=f"{network.replace('_', ' ')}: {window_label}",
            )
    axis.set_title(f"{probe} raw evoked A1-target FC")
    axis.set_xlabel("AD-like perturbation severity")
    axis.set_ylabel("Fisher z")
    axis.legend(fontsize=7)
fig.tight_layout()
fig.savefig(
    FIGURE_DIR / "late_followup_raw_fc_original_vs_late.png",
    dpi=180,
    bbox_inches="tight",
)
plt.show()


## 18. Prespecified figures

Every figure is generated from saved tables. Titles describe the plotted
quantities without converting model-internal increases into claims of
preserved memory.


In [ ]:
NETWORK_COLORS = {
    "semantic_expanded": "#6B46C1",
    "episodic_expanded": "#0F766E",
}
NETWORK_DISPLAY_NAMES = {
    "semantic_expanded": "Semantic expanded proxy",
    "episodic_expanded": "Episodic expanded proxy",
}
primary_metric_panels = [
    (
        "transfer_gain",
        "2Hz",
        "log2_transfer_vs_baseline",
        "Transfer change (log2 ratio)",
    ),
    (
        "transfer_gain",
        "5Hz",
        "log2_transfer_vs_baseline",
        "Transfer change (log2 ratio)",
    ),
    (
        "functional_connectivity",
        "2Hz",
        "evoked_fc_z_vs_baseline",
        "FC change (Fisher-z difference)",
    ),
    (
        "functional_connectivity",
        "5Hz",
        "evoked_fc_z_vs_baseline",
        "FC change (Fisher-z difference)",
    ),
    (
        "response_latency",
        "pulse",
        "latency_ms_vs_baseline",
        "Relative latency change (ms)",
    ),
]

fig, axes = plt.subplots(2, 3, figsize=(15.5, 8.4))
axes = axes.ravel()
for axis, (
    outcome,
    probe,
    metric_column,
    ylabel,
) in zip(axes, primary_metric_panels):
    subset = main_normalized_df[
        (main_normalized_df["probe"] == probe)
        & (
            main_normalized_df["network"].isin(
                PRIMARY_INFERENTIAL_NETWORKS
            )
        )
    ]
    for network in PRIMARY_INFERENTIAL_NETWORKS:
        network_data = subset[subset["network"] == network]
        summary = (
            network_data.groupby("severity", as_index=False)
            .agg(
                mean=(metric_column, "mean"),
                standard_error=(
                    metric_column,
                    lambda values: (
                        float(np.std(values, ddof=1))
                        / np.sqrt(len(values))
                        if len(values) > 1
                        else 0.0
                    ),
                ),
            )
            .sort_values("severity")
        )
        axis.errorbar(
            summary["severity"],
            summary["mean"],
            yerr=summary["standard_error"],
            marker="o",
            capsize=3,
            color=NETWORK_COLORS[network],
            label=NETWORK_DISPLAY_NAMES[network],
        )
    axis.axhline(0.0, color="black", linewidth=0.8)
    axis.set(
        title=f"{probe}: {outcome.replace('_', ' ')}",
        xlabel="AD-like perturbation severity",
        ylabel=ylabel,
        xticks=[0.0, 0.5, 1.0],
    )
axes[0].legend()
axes[-1].axis("off")
fig.suptitle(
    "Baseline-referenced semantic and episodic proxy trajectories",
    y=1.01,
)
fig.tight_layout()
fig.savefig(
    FIGURE_DIR / "02_primary_metric_trajectories.png",
    dpi=180,
    bbox_inches="tight",
)
plt.show()

interaction_plot_df = primary_main_statistics_df.copy()
interaction_plot_df["label"] = (
    interaction_plot_df["outcome"].str.replace("_", " ")
    + "\n"
    + interaction_plot_df["probe"]
)
x_positions = np.arange(len(interaction_plot_df))
lower_error = (
    interaction_plot_df["mean_interaction"]
    - interaction_plot_df["ci95_lower_numerical"]
).fillna(0.0)
upper_error = (
    interaction_plot_df["ci95_upper_numerical"]
    - interaction_plot_df["mean_interaction"]
).fillna(0.0)
fig, ax = plt.subplots(figsize=(10.5, 4.8))
ax.errorbar(
    x_positions,
    interaction_plot_df["mean_interaction"],
    yerr=np.vstack([lower_error, upper_error]),
    fmt="o",
    capsize=4,
    color="#334155",
)
ax.axhline(0.0, color="black", linewidth=0.9)
ax.set(
    xticks=x_positions,
    xticklabels=interaction_plot_df["label"],
    ylabel="Semantic-minus-episodic interaction",
    title=(
        "Primary endpoint interactions with 95% numerical-"
        "initialization intervals"
    ),
)
fig.tight_layout()
fig.savefig(
    FIGURE_DIR / "03_primary_interactions.png",
    dpi=180,
)
plt.show()

counterfactual_plot = counterfactual_summary_df.copy()
counterfactual_plot["label"] = (
    counterfactual_plot["outcome"].str.replace("_", " ")
    + "\n"
    + counterfactual_plot["probe"]
)
x_positions = np.arange(len(counterfactual_plot))
width = 0.36
fig, ax = plt.subplots(figsize=(10.5, 4.8))
ax.bar(
    x_positions - width / 2,
    counterfactual_plot["median_full_interaction"],
    width,
    label="Full field",
    color="#6B46C1",
)
ax.bar(
    x_positions + width / 2,
    counterfactual_plot["median_local_fixed_interaction"],
    width,
    label="Expanded targets locally fixed",
    color="#94A3B8",
)
ax.axhline(0.0, color="black", linewidth=0.9)
ax.set(
    xticks=x_positions,
    xticklabels=counterfactual_plot["label"],
    ylabel="Median semantic-minus-episodic interaction",
    title="Local-dynamics-held-baseline counterfactual",
)
ax.legend()
fig.tight_layout()
fig.savefig(
    FIGURE_DIR / "04_local_dynamics_counterfactual.png",
    dpi=180,
)
plt.show()

definition_plot_df = main_interaction_statistics_df.copy()
definition_plot_df["label"] = (
    definition_plot_df["pair"].str.replace("_", " ")
    + "\n"
    + definition_plot_df["outcome"].str.replace("_", " ")
    + " "
    + definition_plot_df["probe"]
)
fig, ax = plt.subplots(figsize=(13.5, 6.0))
x_positions = np.arange(len(definition_plot_df))
ax.scatter(
    x_positions,
    definition_plot_df["mean_interaction"],
    color=[
        "#111827" if pair == "expanded_bilateral" else "#64748B"
        for pair in definition_plot_df["pair"]
    ],
    zorder=3,
)
for x_value, row in enumerate(
    definition_plot_df.itertuples(index=False)
):
    if (
        np.isfinite(row.ci95_lower_numerical)
        and np.isfinite(row.ci95_upper_numerical)
    ):
        ax.vlines(
            x_value,
            row.ci95_lower_numerical,
            row.ci95_upper_numerical,
            color="#94A3B8",
            linewidth=1.2,
        )
ax.axhline(0.0, color="black", linewidth=0.9)
ax.set(
    xticks=x_positions,
    xticklabels=definition_plot_df["label"],
    ylabel="Mean semantic-minus-episodic interaction",
    title="ROI-definition and laterality sensitivity",
)
ax.tick_params(axis="x", rotation=75)
fig.tight_layout()
fig.savefig(
    FIGURE_DIR / "05_definition_laterality_sensitivity.png",
    dpi=180,
)
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(13.0, 4.8))
percentile_summary = (
    matched_null_summary_df.groupby(
        ["outcome", "probe"], as_index=False
    )["observed_empirical_percentile"]
    .agg(["median", "min", "max"])
    .reset_index()
)
labels = (
    percentile_summary["outcome"].str.replace("_", " ")
    + "\n"
    + percentile_summary["probe"]
)
x_positions = np.arange(len(percentile_summary))
axes[0].errorbar(
    x_positions,
    percentile_summary["median"],
    yerr=np.vstack(
        [
            percentile_summary["median"]
            - percentile_summary["min"],
            percentile_summary["max"]
            - percentile_summary["median"],
        ]
    ),
    fmt="o",
    capsize=4,
    color="#334155",
)
axes[0].axhspan(5, 95, color="#E2E8F0", alpha=0.7)
axes[0].set(
    xticks=x_positions,
    xticklabels=labels,
    ylim=(0, 100),
    ylabel="Observed percentile within matched controls",
    title="Matched parcel-set ranks across seeds",
)

shuffle_labels = (
    spatial_shuffle_summary_df["outcome"].str.replace("_", " ")
    + "\n"
    + spatial_shuffle_summary_df["probe"]
)
shuffle_x = np.arange(len(spatial_shuffle_summary_df))
axes[1].scatter(
    shuffle_x,
    spatial_shuffle_summary_df["observed_empirical_percentile"],
    color="#0F766E",
)
axes[1].axhspan(5, 95, color="#E2E8F0", alpha=0.7)
axes[1].set(
    xticks=shuffle_x,
    xticklabels=shuffle_labels,
    ylim=(0, 100),
    ylabel="Observed percentile within spatial shuffles",
    title="Spatial-placement ranks at seed 11",
)
fig.tight_layout()
fig.savefig(
    FIGURE_DIR / "06_matched_and_spatial_robustness.png",
    dpi=180,
)
plt.show()


## 19. Save complete results and provenance

Every primary, sensitivity, QA, counterfactual, null, and manifest table is
saved. For the main scope, losslessly compressed parcel-level shards retain
the original stimulated and control PSP monitor samples before any detrending,
normalization, z-scoring, network averaging, or harmonic fitting; the direct
evoked difference and exact stimulus waveform are stored alongside them.

The main node and network tables also retain all five segment transfer values.
Parcel rows retain exact-frequency sine/cosine coefficients, phase and
ipsilateral-A1 phase lag, plus pulse peak timing, magnitude, and energy. A
static `regional_features.csv` table records pathology and structural-network
features for all 379 regions. The archive also records the analysis-specification
hash and the exact worker/call and raw-trace manifests.


In [ ]:
regional_features_df = pd.DataFrame(
    {
        "region_index": np.arange(N_REGIONS, dtype=int),
        "region_label": LABELS,
        "hemisphere": [label_hemisphere(label) for label in LABELS],
        "surrogate_amyloid": AD_AMYLOID,
        "baseline_b": B_BY_SEVERITY[0.0],
        "intermediate_b": B_BY_SEVERITY[0.5],
        "high_b": B_BY_SEVERITY[1.0],
        "b_reduction": B_BY_SEVERITY[0.0] - B_BY_SEVERITY[1.0],
        "weighted_structural_strength": weighted_strength,
        "structural_feature_basis": np.repeat(
            "symmetrized_0.5_times_W_plus_W_transpose",
            N_REGIONS,
        ),
        "direct_connectivity_to_left_a1": MATCH_WEIGHTS[
            :, A1_INDEX_BY_HEMISPHERE["L"]
        ],
        "direct_connectivity_to_right_a1": MATCH_WEIGHTS[
            :, A1_INDEX_BY_HEMISPHERE["R"]
        ],
        "bilateral_a1_affinity": direct_a1_affinity,
        "semantic_expanded_membership": np.isin(
            np.arange(N_REGIONS),
            NETWORK_INDICES[PRIMARY_SEMANTIC_NETWORK],
        ),
        "episodic_expanded_membership": np.isin(
            np.arange(N_REGIONS),
            NETWORK_INDICES[PRIMARY_EPISODIC_NETWORK],
        ),
        "semantic_anatomical_core_membership": np.isin(
            LABELS,
            SEMANTIC_ANATOMICAL_CORE_LABELS,
        ),
        "episodic_anatomical_core_membership": np.isin(
            LABELS,
            EPISODIC_ANATOMICAL_CORE_LABELS,
        ),
        "anatomical_core_membership": np.isin(
            LABELS,
            ordered_union(
                SEMANTIC_ANATOMICAL_CORE_LABELS,
                EPISODIC_ANATOMICAL_CORE_LABELS,
            ),
        ),
        "semantic_platel_peak_membership": np.isin(
            LABELS,
            SEMANTIC_PLATEL_LABELS,
        ),
        "episodic_platel_peak_membership": np.isin(
            LABELS,
            EPISODIC_PLATEL_LABELS,
        ),
        "platel_peak_membership": np.isin(
            LABELS,
            ordered_union(
                SEMANTIC_PLATEL_LABELS,
                EPISODIC_PLATEL_LABELS,
            ),
        ),
        "raw_trace_region_membership": np.isin(
            np.arange(N_REGIONS),
            TRACE_REGION_INDICES,
        ),
    }
)

all_manifest_df = pd.concat(
    [
        main_manifest_df,
        dt_reference_manifest_df,
        local_fixed_manifest_df,
        parameter_manifest_df,
        shuffle_manifest_df,
        late_followup_manifest_df,
        late_followup_dt_reference_manifest_df,
    ],
    ignore_index=True,
)
output_tables = {
    "source_manifest.csv": source_manifest_df,
    "data_quality_checks.csv": data_quality_df,
    "roi_definitions.csv": roi_definition_df,
    "network_evidence.csv": network_evidence_df,
    "music_memory_peak_mapping.csv": music_memory_peak_mapping_df,
    "roi_pathology_values.csv": roi_pathology_df,
    "pathology_summary.csv": pathology_summary_df,
    "regional_features.csv": regional_features_df,
    "main_parcel_trace_manifest.csv": main_trace_manifest_df,
    "baseline_coupling_diagnostic.csv": calibration_df,
    "main_node_metrics.csv": main_node_df,
    "main_network_metrics.csv": main_network_df,
    "main_network_metrics_normalized.csv": main_normalized_df,
    "main_pair_interactions.csv": main_pair_interaction_df,
    "main_interaction_statistics.csv": (
        main_interaction_statistics_df
    ),
    "primary_interaction_statistics.csv": (
        primary_main_statistics_df
    ),
    "definition_sensitivity_statistics.csv": (
        definition_sensitivity_statistics_df
    ),
    "laterality_difference_statistics.csv": (
        laterality_difference_statistics_df
    ),
    "technical_preflight_validity.csv": (
        preflight_science_validity_df
    ),
    "main_science_validity.csv": main_science_validity_df,
    "outcome_eligibility.csv": outcome_eligibility_df,
    "a1_frequency_qa.csv": a1_frequency_qa_df,
    "periodic_temporal_qa.csv": periodic_temporal_qa_df,
    "integration_step_check.csv": dt_convergence_df,
    "integration_step_outcome_eligibility.csv": (
        dt_outcome_eligibility_df
    ),
    "integration_step_interaction_seed_diagnostics.csv": (
        dt_interaction_seed_diagnostics_df
    ),
    "integration_step_a1_snr_seed_diagnostics.csv": (
        dt_snr_seed_diagnostics_df
    ),
    "integration_step_raw_metric_seed_diagnostics.csv": (
        dt_raw_seed_diagnostics_df
    ),
    "dt_reference_node_metrics.csv": dt_reference_node_df,
    "dt_reference_network_metrics.csv": dt_reference_network_df,
    "local_fixed_node_metrics.csv": local_fixed_node_df,
    "local_fixed_network_metrics.csv": local_fixed_network_df,
    "local_fixed_pair_interactions.csv": (
        local_fixed_pair_interaction_df
    ),
    "local_fixed_interaction_statistics.csv": (
        local_fixed_statistics_df
    ),
    "counterfactual_comparison.csv": (
        counterfactual_comparison_df
    ),
    "counterfactual_summary.csv": counterfactual_summary_df,
    "matched_control_sets.csv": matched_sets_df,
    "matched_control_null_metrics.csv": matched_null_df,
    "matched_control_null_summary.csv": (
        matched_null_summary_df
    ),
    "parameter_node_metrics.csv": parameter_node_df,
    "parameter_network_metrics.csv": parameter_network_df,
    "parameter_pair_interactions.csv": (
        parameter_pair_interaction_df
    ),
    "parameter_interaction_statistics.csv": (
        parameter_interaction_statistics_df
    ),
    "spatial_shuffle_node_metrics.csv": shuffle_node_df,
    "spatial_shuffle_network_metrics.csv": shuffle_network_df,
    "spatial_shuffle_pair_interactions.csv": (
        shuffle_pair_interaction_df
    ),
    "spatial_shuffle_summary.csv": spatial_shuffle_summary_df,
    "late_followup_node_metrics.csv": late_followup_node_df,
    "late_followup_network_metrics.csv": late_followup_network_df,
    "late_followup_network_metrics_normalized.csv": (
        late_followup_normalized_df
    ),
    "late_followup_segment_metrics.csv": late_followup_segment_df,
    "late_followup_pair_interactions.csv": (
        late_followup_pair_interaction_df
    ),
    "late_followup_interaction_statistics.csv": (
        late_followup_interaction_statistics_df
    ),
    "late_followup_science_validity.csv": (
        late_followup_science_validity_df
    ),
    "late_followup_outcome_eligibility.csv": (
        late_followup_outcome_eligibility_df
    ),
    "late_followup_stability_equivalence.csv": (
        late_followup_stability_df
    ),
    "late_followup_ten_segment_slopes.csv": (
        late_followup_ten_segment_slopes_df
    ),
    "late_followup_original_vs_late.csv": (
        late_followup_original_vs_late_df
    ),
    "late_followup_prefix_reconciliation.csv": (
        late_followup_prefix_reconciliation_df
    ),
    "late_followup_metric_reconciliation.csv": (
        late_followup_metric_reconciliation_df
    ),
    "late_followup_trace_manifest.csv": (
        late_followup_trace_manifest_df
    ),
    "late_followup_dt_reference_node_metrics.csv": (
        late_followup_dt_reference_node_df
    ),
    "late_followup_dt_reference_network_metrics.csv": (
        late_followup_dt_reference_network_df
    ),
    "late_followup_dt_reference_segment_metrics.csv": (
        late_followup_dt_reference_segment_df
    ),
    "late_followup_dt_reference_pair_interactions.csv": (
        late_followup_dt_reference_pair_interaction_df
    ),
    "late_followup_dt_check.csv": late_followup_dt_check_df,
    "late_followup_dt_reference_trace_manifest.csv": (
        late_followup_dt_reference_trace_manifest_df
    ),
    "run_manifest.csv": all_manifest_df,
}
for filename, frame in output_tables.items():
    frame.to_csv(RESULTS_DIR / filename, index=False)

total_tvb_calls = int(
    len(all_manifest_df) + 2 * len(calibration_df)
)
metadata = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "run_mode": RUN_MODE,
    "analysis_spec_sha256": ANALYSIS_SPEC_SHA256,
    "research_question": ANALYSIS_SPEC["research_question"],
    "source_commits": {
        "educase": EDUCASE_COMMIT,
        "adni_tvb_pipeline": PIPELINE_COMMIT,
    },
    "source_hashes": {
        row.source: row.sha256
        for row in source_manifest_df.itertuples(index=False)
    },
    "execution": {
        "parallel_backend": PARALLEL_BACKEND,
        "joblib_version": joblib.__version__,
        "cpu_allocation_sources": CPU_ALLOCATION_SOURCES,
        "available_cpu_count": AVAILABLE_CPU_COUNT,
        "requested_worker_processes": (
            REQUESTED_PARALLEL_WORKERS
        ),
        "configured_worker_processes": PARALLEL_WORKERS,
        "native_threads_per_worker": NATIVE_THREADS_PER_WORKER,
        "observed_worker_pids": sorted(
            int(pid)
            for pid in all_manifest_df[
                "worker_pid"
            ].dropna().unique()
        ),
        "observed_worker_processes": int(
            all_manifest_df["worker_pid"].nunique()
        ),
        "worker_override_environment_variable": (
            "RISE_N_WORKERS"
        ),
        "total_tvb_calls_including_calibration": total_tvb_calls,
    },
    "model": {
        "regions": N_REGIONS,
        "global_coupling": MAIN_GLOBAL_COUPLING,
        "input_peak_per_ms": MAIN_INPUT_PEAK_PER_MS,
        "dt_ms": MAIN_DT_MS,
        "reference_dt_ms": REFERENCE_DT_MS,
        "monitor_period_ms": MONITOR_PERIOD_MS,
        "probe_simulation_ms": PROBE_SIMULATION_MS,
        "stimulus_onset_ms": STIMULUS_ONSET_MS,
        "interregional_delays": "all zero",
        "numerical_seeds": list(CFG["seeds"]),
    },
    "primary_parcels": {
        "semantic_expanded": list(SEMANTIC_EXPANDED_LABELS),
        "episodic_expanded": list(EPISODIC_EXPANDED_LABELS),
    },
    "raw_trace_export": {
        "scope": list(TRACE_EXPORT_SCOPES),
        "format_version": TRACE_FORMAT_VERSION,
        "checkpoint_tag": TRACE_CHECKPOINT_TAG,
        "archive_subdirectory": TRACE_ARCHIVE_SUBDIRECTORY,
        "trace_regions": list(TRACE_REGION_LABELS),
        "trace_region_indices": TRACE_REGION_INDICES.tolist(),
        "archive_count": int(len(main_trace_manifest_df)),
        "lossless_compression": True,
        "transform_before_save": "none",
    },
    "prolonged_stimulation_followup": {
        "hierarchy": "secondary_stability_and_entrainment_followup",
        "analysis_version": FOLLOWUP_ANALYSIS_VERSION,
        "simulation_end_ms": FOLLOWUP_SIMULATION_END_MS,
        "windows_ms": FOLLOWUP_WINDOWS_MS,
        "reference_types": list(FOLLOWUP_REFERENCE_TYPES),
        "dt_values_ms": list(FOLLOWUP_DT_VALUES_MS),
        "main_dt_trace_archives": int(
            len(late_followup_trace_manifest_df)
        ),
        "reference_dt_trace_archives": int(
            len(late_followup_dt_reference_trace_manifest_df)
        ),
        "trace_format_version": FOLLOWUP_TRACE_FORMAT_VERSION,
        "trace_transform_before_save": "none",
        "spectral_method": (
            "DPSS multitaper, NW=3, K=5, complete nonnegative "
            "frequency axis saved for each signal and window"
        ),
        "coherence_method": (
            "magnitude-squared coherence from the same five DPSS "
            "tapered spectra, averaged in the applied-frequency band"
        ),
        "stability_equivalence": (
            late_followup_stability_df.to_dict("records")
        ),
        "integration_step_check": (
            late_followup_dt_check_df.to_dict("records")
        ),
        "main_prefix_reconciliation_passed": bool(
            late_followup_prefix_reconciliation_df["passed"].all()
            and late_followup_metric_reconciliation_df["passed"].all()
        ),
    },
    "component_parcels": {
        "semantic_anatomical_core": list(
            SEMANTIC_ANATOMICAL_CORE_LABELS
        ),
        "episodic_anatomical_core": list(
            EPISODIC_ANATOMICAL_CORE_LABELS
        ),
        "semantic_platel_peaks": list(SEMANTIC_PLATEL_LABELS),
        "episodic_platel_peaks": list(EPISODIC_PLATEL_LABELS),
    },
    "metric_interpretation": {
        "transfer_gain": (
            "A1-normalized control-subtracted AC-RMS response in a "
            "fixed 10-second window, log2 baseline-referenced"
        ),
        "functional_connectivity": (
            "Fisher-z correlation of matched-control-subtracted evoked "
            "target and ipsilateral A1 PSP traces, then baseline-"
            "referenced"
        ),
        "response_latency": (
            "50% evoked-energy timing relative to ipsilateral A1, "
            "then baseline-referenced in ms"
        ),
        "harmonic_phase": (
            "phase=atan2(cosine coefficient, sine coefficient) under "
            "y_f=A*sin(2*pi*f*t+phase); target lag subtracts ipsilateral "
            "A1 and is wrapped to [-pi, pi]"
        ),
        "pulse_peak": (
            "absolute maximum of the untransformed evoked PSP in the "
            "pulse window; signed value and timing at that sample are saved"
        ),
    },
    "science_quality_control": {
        "preflight_seeds": preflight_seeds,
        "preflight_severities": preflight_severities,
        "target_response_ratio_floor": (
            TARGET_RESPONSE_RATIO_FLOOR
        ),
        "minimum_target_response_coverage": (
            MIN_TARGET_RESPONSE_COVERAGE
        ),
        "maximum_fc_split_abs_z": MAX_FC_SPLIT_ABS_Z,
        "maximum_median_pulse_tail_fraction": (
            MAX_PULSE_TAIL_FRACTION
        ),
        "periodic_segment_ms": PERIODIC_SEGMENT_MS,
        "periodic_segment_count": PERIODIC_SEGMENT_COUNT,
        "integration_step_reference_seeds": list(
            CFG["dt_check_seeds"]
        ),
        "integration_step_gate": (
            dt_convergence_df.to_dict("records")
        ),
        "integration_step_outcome_eligibility": (
            dt_outcome_eligibility_df.to_dict("records")
        ),
        "outcome_eligibility": (
            outcome_eligibility_df.to_dict("records")
        ),
    },
    "interpretation_limits": [
        (
            "Numerical initializations are not participants; reported "
            "intervals are not patient or population intervals."
        ),
        (
            "The public amyloid endpoint is an artificial surrogate, "
            "not patient data."
        ),
        (
            "The model has no memory task, encoding, recollection, "
            "familiarity, or behavioral outcome."
        ),
        (
            "Expanded parcel sets are operational proxies and do not "
            "form proven isolated pathways."
        ),
        (
            "Zero tract delays make pulse latency a relative model-"
            "response timing metric, not anatomical conduction latency."
        ),
        (
            "Functional connectivity is common-input-sensitive evoked "
            "PSP correlation, not BOLD FC or directed effective "
            "connectivity."
        ),
        (
            "The 2 Hz and 5 Hz inputs are temporal probes, not literal "
            "music or speech."
        ),
    ],
}
(RESULTS_DIR / "experiment_metadata.json").write_text(
    json.dumps(metadata, indent=2) + "\n"
)

archive_base = WORK_DIR / (
    f"RISE_TVB379_{RESULTS_DIR.name}"
)
archive_path = shutil.make_archive(
    str(archive_base),
    "zip",
    root_dir=RESULTS_DIR,
)
print("Saved result archive:", archive_path)
print("Analysis specification SHA-256:", ANALYSIS_SPEC_SHA256)
print("Total TVB calls:", total_tvb_calls)
print(
    "Aggregate recorded TVB wall time, minutes:",
    round(all_manifest_df["wall_seconds"].sum() / 60.0, 2),
)

if DOWNLOAD_RESULTS_AT_END and IS_COLAB:
    from google.colab import files
    files.download(archive_path)


## 20. How to interpret the final outputs

Read each outcome separately:

1. inspect the semantic and episodic trajectories, not only their
   difference;
2. inspect the paired semantic-minus-episodic interaction and its
   numerical-initialization interval;
3. compare the expanded result with anatomical-core-only,
   Platel-peak-only, left-only, and right-only sensitivities;
4. inspect the local-dynamics-held-baseline counterfactual;
5. inspect matched-control and spatial-shuffle ranks; and
6. inspect parameter and integration-step sensitivity; and
7. use parcel-level raw traces, five-segment values, gain/phase, and
   pulse-peak descriptors to explain the aggregate result without
   replacing the prespecified primary metrics.

Outcome signs are not interchangeable:

- Positive transfer interaction means semantic-associated transfer
  increased more, or decreased less, relative to its own baseline than
  episodic-associated transfer.
- Positive FC interaction means the baseline-referenced evoked A1-target
  PSP correlation change was larger for the semantic proxy.
- Positive latency interaction means the semantic proxy became **more
  delayed** relative to baseline than the episodic proxy. It is not a
  favorable direction.

None of these signs alone means preserved semantic memory, impaired
episodic memory, or superior clinical function. A defensible result
statement must say "modeled transmission into semantic- and
episodic-associated proxy parcel sets."


### Secondary prolonged-stimulation follow-up

Interpret the original and late windows side by side. A stable late broadband
response does not establish frequency entrainment. Frequency-specific wording
requires acceptable target harmonic fit, phase consistency, applied-frequency
power, and coherence. Report the zero-input and DC-matched contrasts separately.
The DC-matched result is modulation-specific only in the paired counterfactual
sense; nonlinear interactions prevent additive decomposition.

Operational equivalence must be read from the 90% numerical-initialization
interval and both one-sided tests. An interval that merely contains zero is not
evidence of stability. Integration-step-sensitive late magnitudes remain
descriptive even when their direction is unchanged.


## 21. Known limitations

1. The downloadable amyloid values are artificial surrogates, so this is
   not a clinical HC/MCI/AD comparison.
2. Only one proposed AD-related mechanism is varied: amyloid-linked change
   in the Jansen-Rit inhibitory rate parameter \(b\).
3. Tau, atrophy, synaptic loss, inflammation, vascular disease, and
   condition-specific structural degeneration are absent.
4. Every condition uses the same averaged healthy structural weight
   matrix.
5. Interregional tract lengths and delays are zero. Relative pulse-response
   timing is measurable, but anatomical conduction latency is not.
6. Evoked PSP correlation is model functional connectivity, not BOLD
   functional connectivity and not causal effective connectivity.
7. Subtracting the matched unstimulated trace isolates the modeled evoked
   response but does not remove the bilateral common-input effect.
8. The SNR, phase, response-coverage, FC split-window, and pulse-tail
   thresholds are prespecified technical gates, not
   biologically validated diagnostic thresholds.
9. The 2 Hz and 5 Hz probes omit melody, pitch, timbre, meaning,
   familiarity, emotion, learning, delay, and recognition.
10. The expanded proxies are comprehensive only relative to the two locked
    evidence components. They are not complete semantic- or
    episodic-memory systems.
11. Platel peak-to-HCP-MMP mapping is approximate because the source PET
    coordinate framework and the volumetric HCP reference are not
    identical.
12. The original Platel episodic peak set is right-sided; its one-sided
    sensitivity is not directly equivalent to the bilateral core
    comparison.
13. Semantic and episodic retrieval share substantial circuitry; parcel-set
    separation is an operational contrast, not proof of isolated pathways.
14. Twenty initializations improve numerical stability, but do not create
    biological replication.
15. Fifty spatial shuffles support only descriptive ranks, not a precise
    formal permutation p-value.
16. High-perturbation periodic responses can escalate across time. The
    primary AC-RMS gain is therefore a fixed-window evoked-response
    summary, not a claim of steady-state transfer; five segment values and
    their log2 slope must be inspected alongside it.
17. Passing fatal integration-step validity checks does not imply that every
    exact effect magnitude is converged. Outcomes exceeding an unchanged
    precision target are explicitly labeled as directionally robust but
    exact-magnitude integration-step sensitive.

18. The late window is measured during continued stimulation. It does not test
    recovery after stimulus offset.
19. Periodic-minus-DC is a paired counterfactual contrast in a nonlinear model,
    not proof that tonic and oscillatory responses add linearly.
20. The plus/minus 20% transfer and plus/minus 0.10 Fisher-z stability margins
    are operational follow-up thresholds, not biologically validated constants.
21. Magnitude-squared coherence is frequency-specific linear association under
    common bilateral input; it is not directed effective connectivity.
22. The prolonged-stimulation analysis is secondary. It cannot retroactively
    change the status of the fixed original primary endpoint.


## 22. Primary references

- Stefanovski, L., et al. (2019). Linking molecular pathways and
  large-scale computational modeling to assess candidate disease
  mechanisms and pharmacodynamics in Alzheimer's disease. *Frontiers in
  Computational Neuroscience, 13*, 54.
  https://doi.org/10.3389/fncom.2019.00054
- BrainModes. *TVB Educase AD molecular pathways*, pinned public surrogate
  model and data.
  https://github.com/BrainModes/TVB_EducaseAD_molecular_pathways_TVB
- BrainModes. *ADNI-TVB pipeline*, pinned 379-region label order.
  https://github.com/BrainModes/ADNI-TVB-pipeline
- Glasser, M. F., et al. (2016). A multi-modal parcellation of human
  cerebral cortex. *Nature, 536*, 171-178.
  https://doi.org/10.1038/nature18933
- Platel, H., Baron, J.-C., Desgranges, B., Bernard, F., & Eustache, F.
  (2003). Semantic and episodic memory of music are subserved by distinct
  neural networks. *NeuroImage, 20*, 244-256.
  https://doi.org/10.1016/S1053-8119(03)00287-8
- Slattery, C. F., et al. (2019). The functional neuroanatomy of musical
  memory in Alzheimer's disease. *Cortex, 115*, 357-370.
  https://doi.org/10.1016/j.cortex.2019.02.003
- Rolls, E. T., Wirth, S., et al. (2023). The human posterior cingulate,
  retrosplenial, and medial parietal cortex effective connectome and
  implications for memory and navigation. *Human Brain Mapping, 44*,
  629-655. https://doi.org/10.1002/hbm.26089
- Friston, K. J., Frith, C. D., Liddle, P. F., & Frackowiak, R. S. J.
  (1993). Functional connectivity: the principal-component analysis of
  large (PET) data sets. *Journal of Cerebral Blood Flow & Metabolism,
  13*, 5-14. PMID: 8417010.
- Tibon, R., et al. (2026). Neural activations and representations during
  episodic versus semantic memory retrieval. *Nature Human Behaviour, 10*,
  803-821. https://doi.org/10.1038/s41562-025-02390-4
- Thomson, D. J. (1982). Spectrum estimation and harmonic analysis.
  *Proceedings of the IEEE, 70*, 1055-1096.
  https://doi.org/10.1109/PROC.1982.12433

- Carter, G. C. (1987). Coherence and time delay estimation. *Proceedings
  of the IEEE, 75*, 236-255. https://doi.org/10.1109/PROC.1987.13723
- Schuirmann, D. J. (1987). A comparison of the two one-sided tests
  procedure and the power approach for assessing equivalence. *Journal of
  Pharmacokinetics and Biopharmaceutics, 15*, 657-680.
  https://doi.org/10.1007/BF01068419
